# 4. Outputs for Transportation Research Part A (TRA)

This notebook copies notebooks 1, 2 and 3 and runs them in one pass, with two changes for the TRA guide for authors:

- **Resolution.** Charts are bitmapped line drawings, so they are saved at **1000 dpi** (TRA minimum). Photographs and the map are combined in Figures 1 and 2 at **600 dpi** (TRA minimum for photographs: 300 dpi).
- **Output folder.** Everything is written to `outputs_TRA/`, so the CSTP outputs in `outputs/` are not touched.

At the end, Part 4 draws Figure 1 (study area and site views) and Figure 2 (vehicle types), and exports every figure as a separate file named for submission in `outputs_TRA/submission_figures/`.

The only figure not drawn here is the research-flow diagram (Figure 3), which is copied from `data/raw/figure_inputs/research_flow.png`.

Inputs outside the data folder: the site views are read from the original videos in `../Traffic Detection (YOLO)/Raw Videos/`.

# Part 1: copy of `1_data_preparation.ipynb`

Code copied unchanged except for the output folder (`outputs_TRA`) and the chart resolution (1000 dpi).

# Notebook 1 — Data preparation: from the raw extraction to data04

Run from the `notebooks/` folder. Inputs come from `../data/raw/`; everything this notebook makes is written to `../outputs/`
(`data/`, `tables/`, `figures/`). Figures are PNG at 600 dpi in Cambria with the house colour scheme of `style.py`; tables are Excel in Cambria.
Run the notebooks in order: **1 → 2 → 3**.

| Part | Content | Writes |
|---|---|---|
| A | Raw manual extraction and YOLO crossing logs → timestamps, headways, speeds, flow; quality checks; data01 (898 events), data02 (894), data03 | `outputs/data/data01–03.xlsx` |
| B | Pooling tests for target and leader classes → data04 (894 events, 14 variables, 5 leader classes) | `outputs/data/data04.xlsx`, `outputs/tables/Pooling_Tables.xlsx`, three pooling figures |
| C | Appendix figures and tables: traffic state, event timing, detector performance | `outputs/figures/`, `outputs/tables/Appendix_Figure_Data.xlsx` |

The three camera-view stills of the sites are taken from the raw videos and are not regenerated here.

## Part A — Build datasets `data01` and `data02`

Rebuilds the analysis data from the two primary sources:

1. **Manual extraction**: `data/raw/Data Compiled (extracted observations).xlsx`, sheet `data`. This holds the four timestamps per following event, the vehicle labels and the behaviour flags.
2. **YOLO crossing logs**: `data/raw/yolo_crossings/*.xlsx`. Each row is a vehicle crossing the count line, with its class and time.

Outputs (in `data/`):
- `data01.xlsx`: all 898 events, with the main variables recomputed from the raw timestamps and the flow and composition recomputed from the YOLO logs.
- `data02.xlsx`: the cleaned analysis set. It drops the 4 events where the target reaches line 1 before the leader, leaving n = 894.

All paths are relative to this notebook, and every rule used is recorded in the `PARAMS` cell and in `LOG.md`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json, datetime
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Font
FONT_NAME = "Cambria"            # house font for all figures and Excel outputs
def cambria_xlsx(path):
    # set every cell of every sheet in an .xlsx to Cambria (header row bold)
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT_NAME, bold=(c.row == 1), size=11)
    wb.save(path)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
OUT = ROOT / "outputs_TRA" / "data"; OUT.mkdir(parents=True, exist_ok=True)
print("project root:", ROOT)

### PARAMS: every rule used in this notebook

In [ ]:
FPS = 30                      # video frame rate used to convert frame -> seconds (all three sessions are 30 fps)
WINDOW_SEC = 30.0             # rolling flow window width (centred), seconds
EDGE_FRACTION = 0.8           # window flagged as edge-truncated if effective width < 0.8 * WINDOW_SEC
FLOW_TIME_RULE = "floor_TT2"  # flow evaluated at floor(TT2), the whole second in which the target crosses line 2
                              # (this reproduces the prior compiled dataset exactly; see check below)

# Passenger-car equivalents per YOLO class (RHD 2005, as used in the prior YOLO flow notebook)
PCU = {"3W": 2.00, "4W": 1.00, "Bicycle": 0.50, "Bike": 0.75, "Bus": 3.00, "CNG": 0.75}

# Session (video) -> YOLO crossing log, site, segment length (m), road width (m)
SESSIONS = {
    "SP":       {"log": "SP_log_M.xlsx",        "site": "Shahjahanpur", "L": 4.5, "W": 9.400},
    "SP2":      {"log": "SP2_log.xlsx",         "site": "Shahjahanpur", "L": 4.2, "W": 9.400},
    "Tikatuli": {"log": "Tikatuli Merged.xlsx", "site": "Tikatuli",     "L": 8.3, "W": 13.885},
}

# Target class: Mod_BTW is pooled into BTW (pooling to be re-tested in the analysis notebooks)
TARGET_MAP = {"BTW": "BTW", "Mod_BTW": "BTW", "PR": "PR"}

# Leader class: raw label -> five engineering classes (same scheme as the TRB data3 file)
LEADER_MAP = {"Car": "4W", "SUV": "4W", "Utility_V": "4W", "Man_Hauler": "4W",
              "BTW": "MT_3W", "Mod_BTW": "MT_3W", "CNG": "MT_3W",
              "PR": "NMT_3W", "Van": "NMT_3W",
              "Bike": "MT_2W", "Bicycle": "NMT_2W"}

# data02 cleaning rule
DROP_TARGET_AHEAD_AT_L1 = True   # drop events with TT1 < LT1

### 1. Input fingerprints

In [ ]:
def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

inputs = [RAW / "Data Compiled (extracted observations).xlsx"] + \
         [RAW / "yolo_crossings" / s["log"] for s in SESSIONS.values()]
fingerprints = pd.DataFrame({"file": [str(p.relative_to(ROOT)) for p in inputs],
                             "sha256": [sha256(p) for p in inputs],
                             "bytes": [p.stat().st_size for p in inputs]})
fingerprints

### 2. Manual extraction: load and rebuild timestamps

In [ ]:
raw = pd.read_excel(RAW / "Data Compiled (extracted observations).xlsx", sheet_name="data", header=1)
raw.insert(0, "Row_Source", np.arange(3, len(raw) + 3))   # Excel row number in the source sheet (header is on row 2)
print(raw.shape)

def ts(m, s, f):
    return raw[m] * 60 + raw[s] + raw[f] / FPS

t = pd.DataFrame({
    "LT1": ts("Min",  "Sec",  "Frame"),     # leader crosses line 1
    "TT1": ts("Min2", "Sec3", "Frame4"),    # target crosses line 1
    "LT2": ts("Min5", "Sec6", "Frame7"),    # leader crosses line 2
    "TT2": ts("Min8", "Sec9", "Frame10"),   # target crosses line 2
})

# Check against the timestamps stored in the source sheet
chk = {"LT1": "LT1", "TT1": "TT1", "LT2": "LT2", "TT2": "Video_Time_Stamp"}
for k, col in chk.items():
    err = (t[k] - raw[col]).abs().max()
    print(f"{k}: max |recomputed - stored| = {err:.2e}")
    assert err < 1e-6

### 3. Main variables, recomputed from timestamps

In [ ]:
d = pd.DataFrame({"Row_Source": raw["Row_Source"], "Session": raw["Video"]})
d["Site"] = d["Session"].map({k: v["site"] for k, v in SESSIONS.items()})
d["V_Target_Raw"] = raw["V_Target"]
d["Target_Class"] = raw["V_Target"].map(TARGET_MAP)
d["V_Leading_Raw"] = raw["V_Leading"]
d["Leader_Class"] = raw["V_Leading"].map(LEADER_MAP)
assert d[["Target_Class", "Leader_Class"]].notna().all().all(), "unmapped label"
d["Pair"] = d["Target_Class"] + "_following_" + d["Leader_Class"]

d[["LT1", "TT1", "LT2", "TT2"]] = t.values
d["Segment_Length_m"] = d["Session"].map({k: v["L"] for k, v in SESSIONS.items()})
assert np.allclose(d["Segment_Length_m"], raw["Segment_Length"])

d["Time_Headway_s"]        = d["TT2"] - d["LT2"]             # headway at line 2 (front-to-front passage)
d["Headway_L1_s"]          = d["TT1"] - d["LT1"]             # headway at line 1 (kept for reference/QA)
d["Target_Travel_Time_s"]  = d["TT2"] - d["TT1"]
d["Leader_Travel_Time_s"]  = d["LT2"] - d["LT1"]
d["Target_Speed_kmh"]      = d["Segment_Length_m"] / d["Target_Travel_Time_s"] * 3.6
d["Leading_Speed_kmh"]     = d["Segment_Length_m"] / d["Leader_Travel_Time_s"] * 3.6
d["Speed_Difference_kmh"]  = d["Target_Speed_kmh"] - d["Leading_Speed_kmh"]

# Manually coded behaviour / site variables (carried over unchanged)
d["Off_Centered"]        = raw["Off_centeredness"].astype(bool)
d["Occupied"]            = raw["Occupancy"].astype(bool)
d["Roadside_Parking"]    = raw["Road_Side_Parking"].astype(bool)
d["Road_Width_m"]        = d["Session"].map({k: v["W"] for k, v in SESSIONS.items()})
d["Parking_Width_m"]     = raw["Space_Consumed_by_Parking"]
d["Effective_Width_m"]   = d["Road_Width_m"] - d["Parking_Width_m"]

# Checks against the source sheet's derived columns
pairs = [("Time_Headway_s", "Time_Headway"), ("Target_Travel_Time_s", "Target_Gap"),
         ("Leader_Travel_Time_s", "Leading_Gap"), ("Target_Speed_kmh", "Target_Speed_km/hr"),
         ("Leading_Speed_kmh", "Leading_Speed_km/hr"), ("Speed_Difference_kmh", "Speed_Difference"),
         ("Road_Width_m", "Road_Width"), ("Effective_Width_m", "Available_Carriage_Way")]
for a, b in pairs:
    err = (d[a] - raw[b]).abs().max()
    print(f"{a:24s} vs {b:22s} max err = {err:.2e}")
    assert err < 1e-6

### 4. Rolling flow and composition from the YOLO crossing logs

For each session, the flow at time *t* is the PCU sum of the crossings in the centred window
[t − 15, t + 15] s, scaled to an hourly rate:

$$q(t) = \frac{3600}{w_{\text{eff}}(t)} \sum_{j:\,t_j\in[t-15,\,t+15]} \text{PCU}_{c_j}$$

The window is truncated at the start and end of the recording, so $w_{\text{eff}}\le 30$ s; windows narrower than 24 s are flagged as edge windows.
Composition shares are PCU fractions, $\text{share}_k = \text{PCU}_k / \text{PCU}_{\text{total}}$, all taken from the same window.
Flow per metre is $q / W_{\text{eff}}$, where $W_{\text{eff}}$ is the road width minus any parking encroachment.

In [ ]:
CLASSES = list(PCU)

def rolling_flow(log, tq):
    log = log.sort_values("time_sec")
    tt = log["time_sec"].to_numpy(float)
    cl = log["class_name"].to_numpy()
    unknown = set(cl) - set(PCU)
    assert not unknown, unknown
    t0, t1 = tt.min(), tt.max()
    lo = np.clip(tq - WINDOW_SEC / 2, t0, None)
    hi = np.clip(tq + WINDOW_SEC / 2, None, t1)
    eff = hi - lo
    il = np.searchsorted(tt, lo, "left")
    ih = np.searchsorted(tt, hi, "right")
    out = {"Flow_Window_Eff_s": eff, "Flow_Edge_Window": eff < EDGE_FRACTION * WINDOW_SEC}
    pcu = np.array([PCU[c] for c in cl])
    cum_pcu = np.r_[0, np.cumsum(pcu)]
    cum_veh = np.r_[0, np.arange(1, len(tt) + 1)]
    out["Flow_pcu_h"] = (cum_pcu[ih] - cum_pcu[il]) * 3600 / eff
    out["Flow_veh_h"] = (cum_veh[ih] - cum_veh[il]) * 3600 / eff
    tot = cum_pcu[ih] - cum_pcu[il]
    for k in CLASSES:
        ck = np.r_[0, np.cumsum(np.where(cl == k, pcu, 0.0))]
        out[f"Share_{k}"] = np.divide(ck[ih] - ck[il], tot, out=np.zeros_like(tot), where=tot > 0)
    return pd.DataFrame(out)

logs, flow_parts = {}, []
for s, cfg in SESSIONS.items():
    logs[s] = pd.read_excel(RAW / "yolo_crossings" / cfg["log"])
    idx = d.index[d["Session"] == s]
    tq = np.floor(d.loc[idx, "TT2"].to_numpy()) if FLOW_TIME_RULE == "floor_TT2" else d.loc[idx, "TT2"].to_numpy()
    f = rolling_flow(logs[s], tq)
    f.index = idx
    flow_parts.append(f)
    L = logs[s]
    print(f"{s:9s} crossings={len(L):5d}  span={L.time_sec.min():7.1f}-{L.time_sec.max():7.1f} s  "
          f"events={len(idx)}  event TT2 span={d.loc[idx,'TT2'].min():.1f}-{d.loc[idx,'TT2'].max():.1f} s")
flow = pd.concat(flow_parts).sort_index()
d = d.join(flow)
d["Share_3W_All"] = d["Share_3W"] + d["Share_CNG"]          # all three-wheelers (YOLO 3W + CNG)
d["Share_2W"] = d["Share_Bike"] + d["Share_Bicycle"]
d["Flow_pcu_h_m"] = d["Flow_pcu_h"] / d["Effective_Width_m"]

#### 4.1 Reproduction check against the prior compiled flow and shares

In [ ]:
repro = [("Flow_pcu_h", "Flow_pcu/hr"), ("Flow_pcu_h_m", "Flow/m"), ("Share_4W", "Share_4W"),
         ("Share_Bus", "Share_Bus"), ("Share_CNG", "Share_CNG"), ("Share_Bike", "Share_Bike"),
         ("Share_Bicycle", "Share_Bicycle"), ("Share_3W", "Share_3W_Except_CNG"),
         ("Share_3W_All", "Share_3W"), ("Share_2W", "Share_2W")]
rep = pd.DataFrame([{"new": a, "prior": b,
                     "n_match": int(np.isclose(d[a], raw[b], rtol=1e-6, atol=1e-9).sum()),
                     "max_abs_diff": float((d[a] - raw[b]).abs().max())} for a, b in repro])
rep["n"] = len(d)
print(rep.to_string(index=False))
assert (rep["n_match"] == len(d)).all(), "flow/share reproduction failed"

### 5. QA flags (kept in data01, used for the data02 rule)

In [ ]:
frames = raw[["Frame", "Frame4", "Frame7", "Frame10"]]
d["QA_Frame_GE_FPS"]         = (frames >= FPS).any(axis=1)          # retained: valid entry (see LOG)
d["QA_Target_Ahead_At_L1"]   = d["TT1"] < d["LT1"]                  # dropped in data02
d["QA_Shared_Leader"]        = d.duplicated(["Session", "LT1", "LT2"], keep=False)  # informational only
qa = d[["QA_Frame_GE_FPS", "QA_Target_Ahead_At_L1", "QA_Shared_Leader", "Flow_Edge_Window"]].sum()
print(qa)
d.loc[d["QA_Target_Ahead_At_L1"], ["Row_Source", "Session", "Pair", "LT1", "TT1", "LT2", "TT2", "Time_Headway_s", "Headway_L1_s"]]

### 6. Cross-check with the prior TRB analysis file `data3.xlsx` (leader classes and pairs)

In [ ]:
prior3 = RAW / "prior_trb" / "data3.xlsx"
if prior3.exists():
    p3 = pd.read_excel(prior3, sheet_name="Sheet1")
    # match rows on (headway, target speed, leader speed); compare pair labels as multisets per key
    key = lambda df, cols: df[cols].round(6).astype(str).agg("|".join, axis=1)
    ka = key(d, ["Time_Headway_s", "Target_Speed_kmh", "Leading_Speed_kmh"])
    kb = key(p3, ["Time_Headway", "Target_Speed_km/hr", "Leading_Speed_km/hr"])
    A = d.groupby(ka)["Pair"].apply(lambda s: tuple(sorted(s)))
    B = p3.groupby(kb)["Pair"].apply(lambda s: tuple(sorted(s)))
    common = A.index.intersection(B.index)
    agree = (A[common] == B[common]).mean()
    print(f"keys: data01={len(A)} data3={len(B)} common={len(common)}  pair-label agreement on common keys: {agree:.4f}")
    assert agree == 1.0
    print(pd.concat([d["Pair"].value_counts().rename("data01"), p3["Pair"].value_counts().rename("data3")], axis=1))
else:
    print("data3.xlsx not found; skipped (optional check)")

### 7. Assemble data01 and data02

In [ ]:
d.insert(0, "Obs_ID", np.arange(1, len(d) + 1))

COLS = ["Obs_ID", "Row_Source", "Session", "Site",
        "V_Target_Raw", "Target_Class", "V_Leading_Raw", "Leader_Class", "Pair",
        "LT1", "TT1", "LT2", "TT2", "Segment_Length_m",
        "Time_Headway_s", "Headway_L1_s", "Target_Travel_Time_s", "Leader_Travel_Time_s",
        "Target_Speed_kmh", "Leading_Speed_kmh", "Speed_Difference_kmh",
        "Off_Centered", "Occupied", "Roadside_Parking",
        "Road_Width_m", "Parking_Width_m", "Effective_Width_m",
        "Flow_pcu_h", "Flow_veh_h", "Flow_pcu_h_m", "Flow_Window_Eff_s", "Flow_Edge_Window",
        "Share_3W", "Share_CNG", "Share_3W_All", "Share_4W", "Share_Bus", "Share_Bike", "Share_Bicycle", "Share_2W",
        "QA_Frame_GE_FPS", "QA_Target_Ahead_At_L1", "QA_Shared_Leader"]
data01 = d[COLS].copy()

mask_drop = data01["QA_Target_Ahead_At_L1"] if DROP_TARGET_AHEAD_AT_L1 else pd.Series(False, index=data01.index)
dropped = data01.loc[mask_drop, ["Obs_ID", "Row_Source", "Session", "Pair", "Time_Headway_s"]].assign(
    Reason="Target crossed line 1 before leader (TT1 < LT1)")
data02 = data01.loc[~mask_drop].drop(columns=["QA_Target_Ahead_At_L1"]).reset_index(drop=True)
print("data01:", data01.shape, " data02:", data02.shape)

### 8. Variable dictionary

In [ ]:
DESC = {
 "Obs_ID": "Sequential ID in source order (stable across data01/data02)",
 "Row_Source": "Excel row in source sheet 'data' of Data Compiled (extracted observations).xlsx",
 "Session": "Video session: SP, SP2 (Shahjahanpur), Tikatuli (two halves merged on one timeline)",
 "Site": "Study corridor",
 "V_Target_Raw": "Target (follower) label as coded: PR, BTW, Mod_BTW",
 "Target_Class": "PR or BTW (Mod_BTW pooled into BTW)",
 "V_Leading_Raw": "Leader label as coded (11 classes)",
 "Leader_Class": "4W (Car, SUV, Utility_V, Man_Hauler); MT_3W (BTW, Mod_BTW, CNG); NMT_3W (PR, Van); MT_2W (Bike); NMT_2W (Bicycle)",
 "Pair": "Target_Class + '_following_' + Leader_Class",
 "LT1": "Leader passes reference line 1 [s, session timeline] = min*60 + sec + frame/30",
 "TT1": "Target passes reference line 1 [s]",
 "LT2": "Leader passes reference line 2 [s]",
 "TT2": "Target passes reference line 2 [s]",
 "Segment_Length_m": "Longitudinal distance between reference lines [m]",
 "Time_Headway_s": "TT2 - LT2: time headway at line 2 [s] (analysis outcome)",
 "Headway_L1_s": "TT1 - LT1: time headway at line 1 [s] (reference/QA only)",
 "Target_Travel_Time_s": "TT2 - TT1 [s] (named Target_Gap in prior file)",
 "Leader_Travel_Time_s": "LT2 - LT1 [s] (named Leading_Gap in prior file)",
 "Target_Speed_kmh": "Space-mean speed of target over the segment [km/h]",
 "Leading_Speed_kmh": "Space-mean speed of leader over the segment [km/h]",
 "Speed_Difference_kmh": "Target minus leader speed [km/h]; negative = target slower (opening)",
 "Off_Centered": "Target laterally offset from leader (manual coding)",
 "Occupied": "Target carrying passenger(s) (manual coding)",
 "Roadside_Parking": "Roadside parking present at the section (manual coding)",
 "Road_Width_m": "Carriageway width [m]",
 "Parking_Width_m": "Width occupied by roadside parking at event time [m]",
 "Effective_Width_m": "Road_Width_m - Parking_Width_m [m]",
 "Flow_pcu_h": "Rolling flow, centred 30-s window at floor(TT2), from YOLO crossings [pcu/h]",
 "Flow_veh_h": "Same window, vehicle count [veh/h]",
 "Flow_pcu_h_m": "Flow_pcu_h / Effective_Width_m [pcu/h/m]",
 "Flow_Window_Eff_s": "Effective window width after truncation at recording edges [s]",
 "Flow_Edge_Window": "True if effective window < 24 s",
 "Share_3W": "PCU share of YOLO class 3W (PR, BTW, van and other non-CNG three-wheelers) in the window",
 "Share_CNG": "PCU share of CNG auto-rickshaws",
 "Share_3W_All": "Share_3W + Share_CNG",
 "Share_4W": "PCU share of 4W (cars, utility, etc.)",
 "Share_Bus": "PCU share of buses",
 "Share_Bike": "PCU share of motorcycles",
 "Share_Bicycle": "PCU share of bicycles",
 "Share_2W": "Share_Bike + Share_Bicycle",
 "QA_Frame_GE_FPS": "A frame field >= 30 in source (retained; valid entry)",
 "QA_Target_Ahead_At_L1": "TT1 < LT1: target reached line 1 before leader (dropped in data02)",
 "QA_Shared_Leader": "Same leader (LT1, LT2) used by more than one target; normal in non-lane traffic, informational",
}
dictionary = pd.DataFrame({"Variable": COLS, "Description": [DESC[c] for c in COLS]})
params = pd.DataFrame({"Parameter": ["FPS", "WINDOW_SEC", "EDGE_FRACTION", "FLOW_TIME_RULE", "PCU",
                                     "TARGET_MAP", "LEADER_MAP", "DROP_TARGET_AHEAD_AT_L1", "built_at"],
                       "Value": [FPS, WINDOW_SEC, EDGE_FRACTION, FLOW_TIME_RULE, json.dumps(PCU),
                                 json.dumps(TARGET_MAP), json.dumps(LEADER_MAP), DROP_TARGET_AHEAD_AT_L1,
                                 datetime.datetime.now().isoformat(timespec="seconds")]})

### 9. Export

In [ ]:
def write(path, main, name, extra):
    with pd.ExcelWriter(path, engine="openpyxl") as xw:
        main.to_excel(xw, sheet_name=name, index=False)
        for sh, df in extra.items():
            df.to_excel(xw, sheet_name=sh, index=False)
        for ws in xw.book.worksheets:
            ws.freeze_panes = "B2"
            for col in ws.columns:
                width = max(len(str(c.value)) if c.value is not None else 0 for c in list(col)[:50])
                ws.column_dimensions[col[0].column_letter].width = min(max(10, width + 2), 60)
    cambria_xlsx(path)
    print("wrote", path.relative_to(ROOT))

counts = lambda df: (df.groupby(["Site", "Session", "Target_Class"]).size().unstack(fill_value=0)
                       .assign(Total=lambda x: x.sum(axis=1)).reset_index())
write(OUT / "data01.xlsx", data01, "data01",
      {"dictionary": dictionary, "params": params, "flow_reproduction": rep,
       "qa_summary": qa.rename("n_flagged").reset_index().rename(columns={"index": "flag"}),
       "counts": counts(data01), "inputs": fingerprints})
write(OUT / "data02.xlsx", data02, "data02",
      {"dictionary": dictionary[dictionary.Variable != "QA_Target_Ahead_At_L1"], "params": params,
       "dropped_from_data01": dropped, "counts": counts(data02),
       "pair_counts": data02.groupby(["Target_Class", "Leader_Class"]).size().unstack(fill_value=0).reset_index()})

### 9b. data03: main analysis variables, single sheet, before pooling

The class labels stay **as coded**. Pooling (Mod_BTW into BTW, and 11 leader labels into 5 classes) is tested and applied in the pooling section of this notebook, which produces data04.

In [ ]:
DATA03_COLS = ["Obs_ID", "Site", "Session", "V_Target_Raw", "V_Leading_Raw",
               "Time_Headway_s", "Target_Speed_kmh", "Leading_Speed_kmh", "Speed_Difference_kmh",
               "Off_Centered", "Occupied", "Roadside_Parking", "Effective_Width_m",
               "Flow_pcu_h", "Flow_pcu_h_m",
               "Share_3W", "Share_CNG", "Share_4W", "Share_Bus", "Share_Bike", "Share_Bicycle"]
data03 = data02[DATA03_COLS].rename(columns={"V_Target_Raw": "V_Target", "V_Leading_Raw": "V_Leading"})
data03.to_excel(OUT / "data03.xlsx", sheet_name="data03", index=False)
cambria_xlsx(OUT / "data03.xlsx")
print("wrote data/data03.xlsx", data03.shape)

### 10. Summary

In [ ]:
print(counts(data02).to_string(index=False))
print()
print(data02.groupby("Target_Class")[["Time_Headway_s", "Target_Speed_kmh", "Leading_Speed_kmh", "Flow_pcu_h_m"]]
      .agg(["count", "mean", "std", "min", "max"]).round(3).T.to_string())

## Part B — Class pooling: data03 → data04

This notebook tests and applies the two pooling steps:

- **A. Target subtypes:** whether Mod_BTW can be pooled with BTW into one BTW class. PR is kept separate.
- **B. Leader labels:** whether the 11 coded leader labels can be collapsed into five engineering classes. The scheme is defined *a priori* from vehicle size, propulsion and wheel count, and the tests here are supporting evidence for it.

- Input: `data/data03.xlsx`.
- Outputs: `public/data/data04.xlsx` (the public analysis dataset, single sheet), plus `outputs/pooling/Pooling_Tables.xlsx` and two appendix figures in `outputs/figures/` (PNG, 600 dpi).

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import json, datetime, warnings
import numpy as np
import pandas as pd
from scipy import stats
import pingouin as pg
import matplotlib.pyplot as plt

from openpyxl import load_workbook
from openpyxl.styles import Font
FONT_NAME = "Cambria"            # house font for all figures and Excel outputs
def cambria_xlsx(path):
    # set every cell of every sheet in an .xlsx to Cambria (header row bold)
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT_NAME, bold=(c.row == 1), size=11)
    wb.save(path)
warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTP = ROOT / "outputs_TRA" / "tables"; OUTP.mkdir(parents=True, exist_ok=True)
OUTA = ROOT / "outputs_TRA" / "figures"; OUTA.mkdir(parents=True, exist_ok=True)   # pooling figures (appendix)
PUB = ROOT / "outputs_TRA" / "data"; PUB.mkdir(parents=True, exist_ok=True)
d = pd.read_excel(ROOT / "outputs_TRA" / "data" / "data03.xlsx", sheet_name="data03")
print(d.shape)

### PARAMS

In [ ]:
SEED = 20261106
N_BOOT = 2000                 # bootstrap replicates for the Cliff's delta CI
ALPHA = 0.05
TOST_BOUND = {"Time_Headway_s": 0.30,      # s; equivalence margin for headway (same as the CSTP pipeline)
              "Target_Speed_kmh": 2.0}     # km/h; equivalence margin for target speed (new, to be confirmed)
NEGLIGIBLE_DELTA = 0.147      # |Cliff's delta| < 0.147 = negligible (Romano et al. 2006)
TARGET_POOL = {"BTW": "BTW", "Mod_BTW": "BTW", "PR": "PR"}
LEADER_POOL = {"Car": "4W", "SUV": "4W", "Utility_V": "4W", "Man_Hauler": "4W",
               "BTW": "BTW", "Mod_BTW": "BTW", "CNG": "CNG",   # CNG kept separate (Tahsin, 2026-11-06)
               "PR": "NMT", "Van": "NMT", "Bicycle": "NMT",   # non-motorised merged (Tahsin, 2026-11-06)
               "Bike": "MT_2W"}
LEADER_ORDER = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
# Pooling rule for target subtypes (A): pool if headway equivalence holds (TOST p < ALPHA)
# AND |delta| is negligible for both headway and target speed.

### Helper statistics

In [ ]:
rng = np.random.default_rng(SEED)

def cliffs_delta(x, y):
    x, y = np.asarray(x), np.asarray(y)
    gt = (x[:, None] > y[None, :]).sum(); lt = (x[:, None] < y[None, :]).sum()
    return (gt - lt) / (len(x) * len(y))

def delta_ci(x, y, B=N_BOOT):
    x, y = np.asarray(x), np.asarray(y)
    bs = [cliffs_delta(rng.choice(x, len(x)), rng.choice(y, len(y))) for _ in range(B)]
    return np.percentile(bs, [2.5, 97.5])

def magnitude(dl):
    a = abs(dl)
    return "negligible" if a < 0.147 else "small" if a < 0.33 else "medium" if a < 0.474 else "large"

def two_group(x, y, label_x, label_y, var, bound=None):
    x, y = np.asarray(x, float), np.asarray(y, float)
    welch = stats.ttest_ind(x, y, equal_var=False)
    mw = stats.mannwhitneyu(x, y, alternative="two-sided")
    ks = stats.ks_2samp(x, y)
    ad = stats.anderson_ksamp([x, y], method=stats.PermutationMethod(n_resamples=2000, random_state=SEED))
    dl = cliffs_delta(x, y); lo, hi = delta_ci(x, y)
    bf10 = float(pg.ttest(x, y, correction=True)["BF10"].iloc[0])
    row = {"variable": var, "group_1": label_x, "group_2": label_y, "n_1": len(x), "n_2": len(y),
           "mean_1": x.mean(), "mean_2": y.mean(), "median_1": np.median(x), "median_2": np.median(y),
           "welch_t": welch.statistic, "welch_p": welch.pvalue,
           "MW_U": mw.statistic, "MW_p": mw.pvalue,
           "KS_D": ks.statistic, "KS_p": ks.pvalue,
           "AD_stat": ad.statistic, "AD_p": ad.pvalue,
           "cliffs_delta": dl, "delta_CI_lo": lo, "delta_CI_hi": hi, "delta_magnitude": magnitude(dl),
           "BF10": bf10, "BF01": 1 / bf10}
    if bound is not None:
        t = pg.tost(x, y, bound=bound)
        row.update({"TOST_bound": bound, "TOST_p": float(t["pval"].iloc[0])})
    return row

### A. Target subtypes: BTW vs Mod_BTW

Two outcomes are tested: time headway, which is the modelled outcome, and target speed. Each is tested on all data and within each site.
Distributional tests (KS, k-sample AD) and effect sizes (Cliff's δ with a bootstrap CI) describe the difference.
**TOST carries the pooling decision**, because failing to reject "no difference" is not evidence of equivalence. The JZS Bayes factor BF01 is reported as a supporting measure.

In [ ]:
rowsA = []
for scope, g in [("All", d)] + list(d.groupby("Site")):
    for var in ["Time_Headway_s", "Target_Speed_kmh"]:
        x = g.loc[g.V_Target == "BTW", var]; y = g.loc[g.V_Target == "Mod_BTW", var]
        rowsA.append({"scope": scope, **two_group(x, y, "BTW", "Mod_BTW", var, TOST_BOUND[var])})
A = pd.DataFrame(rowsA)

# Reference contrast: PR vs pooled BTW
rowsR = []
for scope, g in [("All", d)] + list(d.groupby("Site")):
    for var in ["Time_Headway_s", "Target_Speed_kmh"]:
        x = g.loc[g.V_Target == "PR", var]; y = g.loc[g.V_Target.isin(["BTW", "Mod_BTW"]), var]
        rowsR.append({"scope": scope, **two_group(x, y, "PR", "BTW+Mod_BTW", var)})
R = pd.DataFrame(rowsR)

show = ["scope", "variable", "n_1", "n_2", "mean_1", "mean_2", "welch_p", "MW_p", "KS_D", "KS_p", "AD_p",
        "cliffs_delta", "delta_CI_lo", "delta_CI_hi", "TOST_p", "BF01"]
print(A[show].round(4).to_string(index=False))
print()
print(R[[c for c in show if c != "TOST_p"]].round(4).to_string(index=False))

In [ ]:
a_all = A[A.scope == "All"].set_index("variable")
pool_target = bool((a_all.loc["Time_Headway_s", "TOST_p"] < ALPHA)
                   and (a_all["cliffs_delta"].abs() < NEGLIGIBLE_DELTA).all())
print("Pool Mod_BTW into BTW:", pool_target)

### B. Leader classes

**B1. Within-class homogeneity:** for each pooled class built from two or more raw labels, the raw labels are compared on leader speed and on the target's headway, using Kruskal–Wallis, then Holm-adjusted pairwise Mann–Whitney tests with Cliff's δ.

**B2. Between-class separation:** the five classes are compared on leader speed using Kruskal–Wallis (ε²), then pairwise Mann–Whitney (Holm), Cliff's δ and KS. For pairs that are not significantly different, a TOST is also run on leader speed with a ±2 km/h margin.

In [ ]:
d["Leader_Class"] = d["V_Leading"].map(LEADER_POOL)
assert d["Leader_Class"].notna().all()

def kw(groups):
    h, p = stats.kruskal(*groups)
    n = sum(len(g) for g in groups); k = len(groups)
    return h, p, (h - k + 1) / (n - k)          # epsilon-squared (Tomczak & Tomczak 2014)

def pairwise(df, group_col, var, order):
    rows = []
    for i, a in enumerate(order):
        for b in order[i + 1:]:
            x = df.loc[df[group_col] == a, var]; y = df.loc[df[group_col] == b, var]
            mw = stats.mannwhitneyu(x, y, alternative="two-sided"); ks = stats.ks_2samp(x, y)
            dl = cliffs_delta(x, y)
            rows.append({"group_1": a, "group_2": b, "n_1": len(x), "n_2": len(y),
                         "median_1": x.median(), "median_2": y.median(), "MW_p": mw.pvalue,
                         "KS_D": ks.statistic, "KS_p": ks.pvalue,
                         "cliffs_delta": dl, "delta_magnitude": magnitude(dl)})
    out = pd.DataFrame(rows)
    out["MW_p_holm"] = pg.multicomp(out["MW_p"].values, method="holm")[1]
    return out

# B1 within-class homogeneity
B1_omni, B1_pair = [], []
for cls in LEADER_ORDER:
    labs = sorted(k for k, v in LEADER_POOL.items() if v == cls)
    if len(labs) < 2:
        continue
    g = d[d.Leader_Class == cls]
    for var in ["Leading_Speed_kmh", "Time_Headway_s"]:
        h, p, e2 = kw([g.loc[g.V_Leading == l, var] for l in labs])
        B1_omni.append({"class": cls, "labels": ", ".join(labs), "n": len(g), "variable": var,
                        "KW_H": h, "KW_p": p, "epsilon2": e2})
        B1_pair.append(pairwise(g, "V_Leading", var, labs).assign(**{"class": cls, "variable": var}))
B1_omni = pd.DataFrame(B1_omni); B1_pair = pd.concat(B1_pair, ignore_index=True)
print(B1_omni.round(4).to_string(index=False))
print()
print(B1_pair[B1_pair.MW_p_holm < ALPHA][["class", "variable", "group_1", "group_2", "median_1", "median_2",
                                           "MW_p_holm", "cliffs_delta", "delta_magnitude"]].round(4).to_string(index=False))

In [ ]:
# B2 between-class separation on leader speed (and headway for reference)
B2_omni, B2_pair = [], []
for var in ["Leading_Speed_kmh", "Time_Headway_s"]:
    h, p, e2 = kw([d.loc[d.Leader_Class == c, var] for c in LEADER_ORDER])
    B2_omni.append({"variable": var, "KW_H": h, "KW_p": p, "epsilon2": e2, "n": len(d)})
    B2_pair.append(pairwise(d, "Leader_Class", var, LEADER_ORDER).assign(variable=var))
B2_omni = pd.DataFrame(B2_omni); B2_pair = pd.concat(B2_pair, ignore_index=True)

LEADER_TOST_BOUND = 2.0   # km/h
ns = B2_pair[(B2_pair.variable == "Leading_Speed_kmh") & (B2_pair.MW_p_holm >= ALPHA)]
tost_rows = []
for _, r in ns.iterrows():
    x = d.loc[d.Leader_Class == r.group_1, "Leading_Speed_kmh"]; y = d.loc[d.Leader_Class == r.group_2, "Leading_Speed_kmh"]
    t = pg.tost(x, y, bound=LEADER_TOST_BOUND); bf10 = float(pg.ttest(x, y)["BF10"].iloc[0])
    tost_rows.append({"group_1": r.group_1, "group_2": r.group_2, "TOST_bound_kmh": LEADER_TOST_BOUND,
                      "TOST_p": float(t["pval"].iloc[0]), "BF01": 1 / bf10})
B2_tost = pd.DataFrame(tost_rows)
print(B2_omni.round(4).to_string(index=False)); print()
print(B2_pair[B2_pair.variable == "Leading_Speed_kmh"].drop(columns="variable").round(4).to_string(index=False)); print()
print(B2_tost.round(4).to_string(index=False))

### Appendix figures (saved to outputs/figures)

In [ ]:
plt.rcParams.update({"font.family": FONT_NAME, "mathtext.fontset": "custom", "mathtext.rm": FONT_NAME,
                     "mathtext.it": FONT_NAME + ":italic", "mathtext.bf": FONT_NAME + ":bold", "font.size": 9})
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
for lab, c, ls in [("PR", PAL["target_sub"]["PR"], "-"), ("BTW", PAL["target_sub"]["BTW"], "-"), ("Mod_BTW", PAL["target_sub"]["Mod_BTW"], "--")]:
    for j, var in enumerate(["Time_Headway_s", "Target_Speed_kmh"]):
        x = np.sort(d.loc[d.V_Target == lab, var]); ax[j].step(x, np.arange(1, len(x) + 1) / len(x), where="post",
                                                                 color=c, ls=ls, lw=1.3, label=f"{lab} (n={len(x)})")
ax[0].set(xlabel="Time headway (s)", ylabel="ECDF"); ax[1].set(xlabel="Target speed (km/h)")
ax[0].legend(frameon=False, fontsize=8)
for a in ax: a.grid(alpha=.3)
fig.tight_layout(); (OUTA / "target_subtypes_ECDF.png").unlink(missing_ok=True); fig.savefig(OUTA / "target_subtypes_ECDF.png", dpi=1000)

fig, ax = plt.subplots(figsize=(7.2, 3.0))
labs = [l for c in LEADER_ORDER for l in sorted(k for k, v in LEADER_POOL.items() if v == c)]
data_b = [d.loc[d.V_Leading == l, "Leading_Speed_kmh"] for l in labs]
bp = ax.boxplot(data_b, tick_labels=labs, showfliers=False, patch_artist=True)
cmap = {c: PAL["leader"][c] for c in LEADER_ORDER}
for patch, l in zip(bp["boxes"], labs):
    patch.set_facecolor(cmap[LEADER_POOL[l]]); patch.set_alpha(.6)
ax.set(ylabel="Leader speed (km/h)"); ax.tick_params(axis="x", rotation=40); ax.grid(axis="y", alpha=.3)
handles = [plt.Rectangle((0, 0), 1, 1, color=cmap[c], alpha=.6) for c in LEADER_ORDER]
ax.legend(handles, LEADER_ORDER, ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.0))
fig.tight_layout(); (OUTA / "leader_speed_by_label.png").unlink(missing_ok=True); fig.savefig(OUTA / "leader_speed_by_label.png", dpi=1000)

# Time headway by leader label, separately for BTW and PR followers (target class strongly shifts headway)
fig, axes = plt.subplots(2, 1, figsize=(7.2, 5.4), sharex=True)
for ax, tgt, tag in zip(axes, ["BTW", "PR"], "ab"):
    sub = d[d.V_Target.map(TARGET_POOL) == tgt]
    data_h = [sub.loc[sub.V_Leading == l, "Time_Headway_s"] for l in labs]
    bp = ax.boxplot(data_h, tick_labels=labs, showfliers=False, patch_artist=True)
    for patch, l in zip(bp["boxes"], labs):
        patch.set_facecolor(cmap[LEADER_POOL[l]]); patch.set_alpha(.6)
    for k, x in enumerate(data_h, start=1):
        ax.text(k, 5.12, f"n={len(x)}", ha="center", va="bottom", fontsize=6.5, color="0.25")
    ax.set(ylabel="Time headway (s)", ylim=(0.4, 5.5))
    ax.set_title(f"({tag}) {tgt} followers (n={len(sub)})", loc="left", fontsize=8.5)
    ax.grid(axis="y", alpha=.3)
axes[-1].tick_params(axis="x", rotation=40)
axes[0].legend(handles, LEADER_ORDER, ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 1.08))
fig.tight_layout(); (OUTA / "headway_by_leader_label.png").unlink(missing_ok=True); fig.savefig(OUTA / "headway_by_leader_label.png", dpi=1000)
plt.show()

### Export the pooling tables and data04

In [ ]:
params = pd.DataFrame({"Parameter": ["SEED", "N_BOOT", "ALPHA", "TOST_BOUND", "NEGLIGIBLE_DELTA", "LEADER_TOST_BOUND",
                                     "TARGET_POOL", "LEADER_POOL", "pool_target_decision", "built_at"],
                       "Value": [SEED, N_BOOT, ALPHA, json.dumps(TOST_BOUND), NEGLIGIBLE_DELTA, LEADER_TOST_BOUND,
                                 json.dumps(TARGET_POOL), json.dumps(LEADER_POOL), pool_target,
                                 datetime.datetime.now().isoformat(timespec="seconds")]})
with pd.ExcelWriter(OUTP / "Pooling_Tables.xlsx") as xw:
    A.to_excel(xw, sheet_name="A_BTW_vs_ModBTW", index=False)
    R.to_excel(xw, sheet_name="A_ref_PR_vs_BTW", index=False)
    B1_omni.to_excel(xw, sheet_name="B1_within_omnibus", index=False)
    B1_pair.to_excel(xw, sheet_name="B1_within_pairwise", index=False)
    B2_omni.to_excel(xw, sheet_name="B2_between_omnibus", index=False)
    B2_pair.to_excel(xw, sheet_name="B2_between_pairwise", index=False)
    B2_tost.to_excel(xw, sheet_name="B2_equivalence", index=False)
    params.to_excel(xw, sheet_name="params", index=False)
cambria_xlsx(OUTP / "Pooling_Tables.xlsx")

assert pool_target, "Target pooling not supported; data04 not written (revisit decision)"
d04 = d.rename(columns={"V_Target": "Target_Type", "V_Leading": "Leader_Type"})
d04.insert(d04.columns.get_loc("Target_Type") + 1, "Target_Class", d04["Target_Type"].map(TARGET_POOL))
d04 = d04.drop(columns="Leader_Class")
d04.insert(d04.columns.get_loc("Leader_Type") + 1, "Leader_Class", d04["Leader_Type"].map(LEADER_POOL))
d04.insert(d04.columns.get_loc("Leader_Class") + 1, "Pair", d04["Target_Class"] + "_following_" + d04["Leader_Class"])
DROP_FROM_D04 = ["Roadside_Parking", "Effective_Width_m", "Flow_pcu_h", "Flow_pcu_h_m",
                 "Share_3W", "Share_CNG", "Share_4W", "Share_Bus", "Share_Bike", "Share_Bicycle"]  # not used further (Tahsin)
d04 = d04.drop(columns=DROP_FROM_D04)
d04.to_excel(PUB / "data04.xlsx", sheet_name="data04", index=False)
cambria_xlsx(PUB / "data04.xlsx")
print("data04:", d04.shape); print(list(d04.columns))
print(d04.groupby(["Target_Class", "Leader_Class"]).size().unstack())

## Part C — Appendix figures (named, not numbered)

| Figure | Content | Sources |
|---|---|---|
| traffic_state_rolling_flow | Traffic state during each survey session: rolling 30-s flow stacked by YOLO class, composition shares, and the sampled events | YOLO crossing logs and data02 |
| yolo_detector_performance | YOLO detector performance on the validation split: normalised confusion matrix, PR curves and training history | model `best.pt`, labelled data, `results.csv` |
| event_timing | Timing of the sampled events within each session, by target class | data02 |

The pooling figures (target_subtypes_ECDF, leader_speed_by_label) are produced by the pooling section of this notebook into the same folder. All outputs are PNG at 600 dpi in Cambria, written to `outputs/figures/`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import json, shutil, datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from openpyxl import load_workbook
from openpyxl.styles import Font

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROJ = ROOT
OUTA = ROOT / "outputs_TRA" / "figures"; OUTA.mkdir(parents=True, exist_ok=True)
OUTT = ROOT / "outputs_TRA" / "tables"; OUTT.mkdir(parents=True, exist_ok=True)
RAW = ROOT / "data" / "raw"
YOLO_DIR = RAW / "yolo"            # training history and cached validation results of the YOLO26n detector

FONT_NAME = "Cambria"
plt.rcParams.update({"font.family": FONT_NAME, "mathtext.fontset": "custom", "mathtext.rm": FONT_NAME,
                     "mathtext.it": FONT_NAME + ":italic", "mathtext.bf": FONT_NAME + ":bold", "font.size": 9})
def save(fig, name):
    p = OUTA / f"{name}.png"; p.unlink(missing_ok=True)
    fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.relative_to(ROOT))
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT_NAME, bold=(c.row == 1), size=11)
    wb.save(path)

### PARAMS

In [ ]:
WINDOW_SEC = 30.0; EDGE_FRACTION = 0.8          # identical to 01_build_data
PCU = {"3W": 2.00, "4W": 1.00, "Bicycle": 0.50, "Bike": 0.75, "Bus": 3.00, "CNG": 0.75}
CLASS_ORDER = ["3W", "CNG", "4W", "Bus", "Bike", "Bicycle"]
CLASS_COL = PAL["yolo_class"]
TARGET_COL = PAL["target"]
SESSIONS = {"SP": {"log": "SP_log_M.xlsx", "label": "Shahjahanpur, session 1 (SP)", "video": "SP.mp4"},
            "SP2": {"log": "SP2_log.xlsx", "label": "Shahjahanpur, session 2 (SP2)", "video": "SP2 (6-7-26).mp4"},
            "Tikatuli": {"log": "Tikatuli Merged.xlsx", "label": "Tikatuli", "video": "Tikatuli 1.mp4"}}

# Camera views: YOLO count lines (from the tracking code; confirmed by the logged crossing centroids,
# which all sit just past these y-values: SP cy>=601, SP2 cy>=701, Tikatuli cy>=901)
COUNT_LINE = {"SP": ((100, 600), (1800, 600)), "SP2": ((200, 700), (1800, 700)), "Tikatuli": ((400, 900), (2200, 900))}
FRAME_TIME = {"SP": "12:45", "SP2": "13:35", "Tikatuli": "15:57"}   # mm:ss in each video file (Tahsin); Tikatuli from "Tikatuli 1.mp4"

# YOLO evaluation
MODEL_PT = YOLO_DIR / "best.pt"   # optional: weights from the YOLO repository, only needed to re-evaluate
LABELLED = YOLO_DIR / "Training Data (labelled)"   # optional: labelled images, only needed to re-evaluate
YOLO_EVAL_CACHE = YOLO_DIR / "yolo_eval_cache.json"
RUN_YOLO_EVAL = not YOLO_EVAL_CACHE.exists()    # re-evaluate only when no cache exists (delete the json to force)
EVAL_KW = dict(imgsz=1280, batch=4, conf=0.001, iou=0.7, plots=True, verbose=False, device=0)
CM_CONF = 0.26               # detection confidence used in tracking; the confusion matrix is computed at this threshold

### Shared data

In [ ]:
d02 = pd.read_excel(ROOT / "outputs_TRA" / "data" / "data02.xlsx", sheet_name="data02")

def per_second_flow(log):
    log = log.sort_values("time_sec")
    tt = log.time_sec.to_numpy(float); cl = log.class_name.to_numpy()
    t0, t1 = tt.min(), tt.max()
    secs = np.arange(np.floor(t0), np.ceil(t1) + 1)
    lo = np.clip(secs - WINDOW_SEC / 2, t0, None); hi = np.clip(secs + WINDOW_SEC / 2, None, t1); eff = hi - lo
    il = np.searchsorted(tt, lo, "left"); ih = np.searchsorted(tt, hi, "right")
    out = pd.DataFrame({"t_sec": secs, "eff": eff, "edge": eff < EDGE_FRACTION * WINDOW_SEC})
    for k in CLASS_ORDER:
        ck = np.r_[0, np.cumsum(np.where(cl == k, PCU[k], 0.0))]
        out[k] = (ck[ih] - ck[il]) * 3600 / eff
    out["total"] = out[CLASS_ORDER].sum(axis=1)
    return out

flows = {s: per_second_flow(pd.read_excel(RAW / "yolo_crossings" / c["log"])) for s, c in SESSIONS.items()}
summary = []
for s, f in flows.items():
    core = f[~f.edge]
    ev = d02[d02.Session == s]
    summary.append({"Session": s, "duration_min": (f.t_sec.max() - f.t_sec.min()) / 60,
                    "flow_mean_pcu_h": core.total.mean(), "flow_sd": core.total.std(),
                    "flow_p05": core.total.quantile(.05), "flow_p95": core.total.quantile(.95),
                    "flow_max": core.total.max(), "n_events": len(ev),
                    "first_event_s": ev.TT2.min(), "last_event_s": ev.TT2.max(),
                    **{f"mean_share_{k}": (core[k] / core.total).mean() for k in CLASS_ORDER}})
summary = pd.DataFrame(summary)
summary.round(3)

### Traffic state during the survey sessions

In [ ]:
durs = np.array([flows[s].t_sec.max() - flows[s].t_sec.min() for s in SESSIONS])
fig = plt.figure(figsize=(7.2, 8.6))
outer = gridspec.GridSpec(3, 1, hspace=0.42, figure=fig)
for i, (s, cfg) in enumerate(SESSIONS.items()):
    f = flows[s]; m = f.t_sec / 60
    inner = gridspec.GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[i], height_ratios=[2.2, 1], hspace=0.08)
    a = fig.add_subplot(inner[0]); b = fig.add_subplot(inner[1], sharex=a)
    a.stackplot(m, *[f[k] for k in CLASS_ORDER], colors=[CLASS_COL[k] for k in CLASS_ORDER], alpha=.85, lw=0)
    a.plot(m, f.total, color="k", lw=.6)
    for side in [f[f.edge & (f.t_sec < f.t_sec.median())], f[f.edge & (f.t_sec > f.t_sec.median())]]:
        if len(side): a.axvspan(side.t_sec.min() / 60, side.t_sec.max() / 60, color="0.85", zorder=0, lw=0)
    ymax = f.total.max() * 1.12
    ev = d02[d02.Session == s]
    for tc, yoff in [("BTW", 1.0), ("PR", 0.955)]:
        e = ev[ev.Target_Class == tc]
        a.plot(e.TT2 / 60, np.full(len(e), ymax * yoff), "|", ms=4.5, mew=.6, color=TARGET_COL[tc])
    a.set_ylim(0, ymax * 1.03); a.set_ylabel("Flow (pcu/h)")
    a.set_title(f"({'abc'[i]}) {cfg['label']}: {len(ev)} sampled events", loc="left", fontsize=9)
    shares = f[CLASS_ORDER].div(f.total.replace(0, np.nan), axis=0).fillna(0)
    b.stackplot(m, *[shares[k] for k in CLASS_ORDER], colors=[CLASS_COL[k] for k in CLASS_ORDER], alpha=.85, lw=0)
    b.set_ylim(0, 1); b.set_ylabel("PCU share"); b.set_xlim(m.min(), m.max())
    plt.setp(a.get_xticklabels(), visible=False)
    if i == 2: b.set_xlabel("Time from start of session (min)")
    for ax in (a, b): ax.grid(alpha=.25, lw=.5)
handles = [plt.Rectangle((0, 0), 1, 1, color=CLASS_COL[k], alpha=.85) for k in CLASS_ORDER] + \
          [Line2D([], [], color="k", lw=.8), Line2D([], [], marker="|", ls="", color=TARGET_COL["BTW"], ms=7),
           Line2D([], [], marker="|", ls="", color=TARGET_COL["PR"], ms=7), plt.Rectangle((0, 0), 1, 1, color="0.85")]
labels = CLASS_ORDER + ["Total", "Sampled event: BTW", "Sampled event: PR", "Edge (window < 24 s)"]
fig.legend(handles, labels, ncol=5, frameon=False, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, 0.0))
fig.subplots_adjust(bottom=0.09, top=0.97, left=0.1, right=0.98)
save(fig, "traffic_state_rolling_flow"); plt.show()

### Site views: one still image per session at the chosen time (no overlays)

### YOLO detector performance

The trained model is evaluated on the **validation split** (287 images, 3,700 instances).
The `Test` folder is a byte-identical copy of `Val`, so there is no independent test set. Because `best.pt` was selected on Val, these figures are validation metrics, not held-out performance.
Neither split shares any image with Train.
The results are cached in `yolo_eval_cache.json`, so later runs only redraw the figure. Delete the json file to re-evaluate.

In [ ]:
def evaluate():
    from ultralytics import YOLO
    import yaml, tempfile
    cfg = {"path": str(LABELLED).replace("\\", "/"), "train": "Train/images", "val": "Val/images", "test": "Test/images",
           "nc": 6, "names": {0: "Bicycle", 1: "Bike", 2: "3W", 3: "CNG", 4: "4W", 5: "Bus"}}
    tmp = Path(tempfile.mkdtemp()); yml = tmp / "data_local.yaml"; yml.write_text(yaml.safe_dump(cfg))
    model = YOLO(str(MODEL_PT)); res = {}
    for split in ["val"]:
        m = model.val(data=str(yml), split=split, project=str(tmp), name=split, **EVAL_KW)
        names = [m.names[i] for i in range(len(m.names))]
        curves = {r[3] + "_vs_" + r[2]: {"x": np.asarray(r[0]).tolist(), "y": np.asarray(r[1]).tolist()}
                  for r in m.box.curves_results}
        # confusion matrix at the tracking confidence threshold
        mc = model.val(data=str(yml), split=split, project=str(tmp), name=split + "_cm",
                       **{**EVAL_KW, "conf": CM_CONF})
        res[split] = {"names": names, "mAP50": float(m.box.map50), "mAP50_95": float(m.box.map),
                      "precision": float(m.box.mp), "recall": float(m.box.mr),
                      "per_class_AP50": dict(zip(names, map(float, m.box.ap50))),
                      "per_class_AP50_95": dict(zip(names, map(float, m.box.ap))),
                      "confusion_matrix": mc.confusion_matrix.matrix.tolist(), "curves": curves}
    shutil.rmtree(tmp, ignore_errors=True)
    res["meta"] = {"model": str(MODEL_PT.name), "eval_kw": EVAL_KW, "cm_conf": CM_CONF,
                   "evaluated_at": datetime.datetime.now().isoformat(timespec="seconds")}
    YOLO_EVAL_CACHE.write_text(json.dumps(res))
    return res

ev = evaluate() if RUN_YOLO_EVAL else json.loads(YOLO_EVAL_CACHE.read_text())
for split in ["val"]:
    r = ev[split]
    print(f"{split}: mAP50={r['mAP50']:.3f} mAP50-95={r['mAP50_95']:.3f} P={r['precision']:.3f} R={r['recall']:.3f}")
    print("   AP50 by class:", {k: round(v, 3) for k, v in r["per_class_AP50"].items()})
print(list(ev["val"]["curves"]))

In [ ]:
r = ev["val"]; names = r["names"]
cm = np.array(r["confusion_matrix"])            # ultralytics layout: rows = predicted, cols = true; last = background
labels_cm = names + ["Background"]
cm_true = cm.T                                   # rows = true class
row_sum = cm_true.sum(axis=1, keepdims=True)
cmn = np.divide(cm_true, row_sum, out=np.zeros_like(cm_true, dtype=float), where=row_sum > 0)

fig = plt.figure(figsize=(7.2, 6.0))
gs = gridspec.GridSpec(2, 2, height_ratios=[1.15, 1], hspace=0.68, wspace=0.32, figure=fig)
a = fig.add_subplot(gs[0, 0])
show = cmn[:-1, :]                               # true background row has no meaning for recall
im = a.imshow(show, cmap="Blues", vmin=0, vmax=1)
for i in range(show.shape[0]):
    for j in range(show.shape[1]):
        if show[i, j] >= 0.005:
            a.text(j, i, f"{show[i, j]:.2f}", ha="center", va="center", fontsize=6.5,
                   color="w" if show[i, j] > .55 else "k")
a.set_xticks(range(len(labels_cm))); a.set_xticklabels(labels_cm, rotation=45, ha="right", fontsize=7)
a.set_yticks(range(len(names))); a.set_yticklabels(names, fontsize=7)
a.set_xlabel("Predicted"); a.set_ylabel("True")
a.set_title(f"(a) Confusion matrix, validation split (conf = {CM_CONF})", loc="left", fontsize=8.5)

b = fig.add_subplot(gs[0, 1])
pr_key = [k for k in r["curves"] if k.lower().startswith("precision_vs_recall")][0]
x = np.array(r["curves"][pr_key]["x"]); Y = np.array(r["curves"][pr_key]["y"])
cols = plt.cm.tab10(np.arange(len(names)))
for k, nm in enumerate(names):
    b.plot(x, Y[k], lw=1, color=cols[k], label=f"{nm} ({r['per_class_AP50'][nm]:.2f})")
b.plot(x, Y.mean(axis=0), lw=2, color="k", label=f"All classes ({r['mAP50']:.2f})")
b.set(xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
b.set_title("(b) Precision-recall, validation split (AP@0.5)", loc="left", fontsize=8.5)
b.legend(fontsize=6.5, frameon=False, loc="lower left"); b.grid(alpha=.25)

c = fig.add_subplot(gs[1, :])
hist = pd.read_csv(YOLO_DIR / "results.csv"); hist.columns = [s.strip() for s in hist.columns]
c.plot(hist.epoch, hist["metrics/mAP50(B)"], color=PAL["training"]["mAP50"], lw=1.3, label="mAP@0.5 (validation)")
c.plot(hist.epoch, hist["metrics/mAP50-95(B)"], color=PAL["training"]["mAP50_95"], lw=1.3, label="mAP@0.5:0.95 (validation)")
c.plot(hist.epoch, hist["metrics/precision(B)"], color="0.4", lw=.9, ls="--", label="Precision (validation)")
c.plot(hist.epoch, hist["metrics/recall(B)"], color="0.4", lw=.9, ls=":", label="Recall (validation)")
c.set(xlabel="Training epoch", ylabel="Metric", ylim=(0, 1), xlim=(1, hist.epoch.max()))
c.set_title(f"(c) Training history (best.pt on validation: mAP@0.5 = {r['mAP50']:.3f}, mAP@0.5:0.95 = {r['mAP50_95']:.3f})",
            loc="left", fontsize=8.5)
c.legend(fontsize=7, frameon=False, ncol=4, loc="lower right"); c.grid(alpha=.25)
save(fig, "yolo_detector_performance"); plt.show()

### Timing of the sampled events

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(7.2, 4.6), sharex=False)
for ax, (s, cfg) in zip(axes, SESSIONS.items()):
    f = flows[s]; ev_s = d02[d02.Session == s]
    bins = np.arange(0, f.t_sec.max() / 60 + 2, 2)       # 2-min bins
    ax.hist([ev_s.loc[ev_s.Target_Class == "BTW", "TT2"] / 60, ev_s.loc[ev_s.Target_Class == "PR", "TT2"] / 60],
            bins=bins, stacked=True, color=[TARGET_COL["BTW"], TARGET_COL["PR"]], alpha=.85,
            label=["BTW", "PR"], edgecolor="w", lw=.4)
    ax.set_xlim(0, f.t_sec.max() / 60); ax.set_ylabel("Events / 2 min")
    nb, npr = (ev_s.Target_Class == "BTW").sum(), (ev_s.Target_Class == "PR").sum()
    ax.set_title(f"({'abc'[list(SESSIONS).index(s)]}) {cfg['label']}: BTW {nb}, PR {npr}", loc="left", fontsize=8.5)
    ax.grid(alpha=.25, axis="y")
axes[0].legend(frameon=False, fontsize=8, ncol=2, loc="upper right")
axes[-1].set_xlabel("Time from start of session (min)")
fig.tight_layout()
save(fig, "event_timing"); plt.show()

### Export the appendix figure data

In [ ]:
yolo_tbl = pd.DataFrame([{"split": sp, "class": "All", "AP50": ev[sp]["mAP50"], "AP50_95": ev[sp]["mAP50_95"],
                          "precision": ev[sp]["precision"], "recall": ev[sp]["recall"]} for sp in ["val"]] +
                        [{"split": sp, "class": k, "AP50": ev[sp]["per_class_AP50"][k], "AP50_95": ev[sp]["per_class_AP50_95"][k]}
                         for sp in ["val"] for k in ev[sp]["names"]])
cm_df = pd.DataFrame(cmn, index=["True_" + l for l in labels_cm], columns=["Pred_" + l for l in labels_cm]).reset_index()
xl = OUTT / "Appendix_Figure_Data.xlsx"
with pd.ExcelWriter(xl) as xw:
    summary.to_excel(xw, sheet_name="traffic_state_summary", index=False)
    yolo_tbl.to_excel(xw, sheet_name="yolo_metrics", index=False)
    cm_df.to_excel(xw, sheet_name="yolo_confusion_val_rownorm", index=False)
    pd.DataFrame({"Parameter": ["WINDOW_SEC", "EDGE_FRACTION", "PCU", "COUNT_LINE", "FRAME_TIME",
                                "EVAL_KW", "CM_CONF", "yolo_eval_at"],
                  "Value": [WINDOW_SEC, EDGE_FRACTION, json.dumps(PCU), json.dumps(COUNT_LINE), json.dumps(FRAME_TIME), json.dumps(EVAL_KW), CM_CONF, ev["meta"]["evaluated_at"]]}
                 ).to_excel(xw, sheet_name="params", index=False)
cambria_xlsx(xl); print("wrote", xl.relative_to(ROOT))
print(summary[["Session", "duration_min", "flow_mean_pcu_h", "flow_sd", "flow_p05", "flow_p95", "flow_max", "n_events",
               "first_event_s", "last_event_s"]].round(1).to_string(index=False))

# Part 2: copy of `2_headway_models.ipynb`

Code copied unchanged except for the output folder (`outputs_TRA`) and the chart resolution (1000 dpi).

# Notebook 2 — Headway distributions and leader effects

Run from the `notebooks/` folder. Inputs come from `../data/raw/`; everything this notebook makes is written to `../outputs/`
(`data/`, `tables/`, `figures/`). Figures are PNG at 600 dpi in Cambria with the house colour scheme of `style.py`; tables are Excel in Cambria.
Run the notebooks in order: **1 → 2 → 3**.

Needs `outputs/data/data04.xlsx` from notebook 1. Steps 1 and 2 run 999 refitted bootstrap replicates per family and case (about 15 minutes each).

| Step | Content |
|---|---|
| 1 | Truncated distribution fits (lognormal, log-logistic, Weibull, gamma, generalized gamma) for the four site × target-class cases |
| 2 | Leader-conditioned truncated AFT model; one family for all cases (Weibull); time ratios by leader class |
| 2b | How each leader class shifts the headway distribution relative to the case's main distribution |
| 3 | Off-centred following, occupancy and leader speed as time ratios, crude and leader-adjusted |

## Step 1 — Truncated distribution fitting of time headway (4 cases)

**Cases:** each site (Shahjahanpur, Tikatuli) × target class (PR, BTW), fitted separately. The sites are never merged.

**Window:** events were extracted only when $0.5 \le H \le 5$ s. Above 5 s is free flow, not following, in sub-saturated traffic; below 0.5 s is most likely a coding error. Every likelihood is therefore **truncated** to this window,

$$\ell(\theta)=\sum_i \Big[\log f(H_i;\theta)-\log\{F(5;\theta)-F(0.5;\theta)\}\Big],$$

and every fitted quantity is conditional on following within the window.

**Families:** taken from the headway literature, each written in accelerated-failure-time (AFT) form $H = e^{\mu}Z$ so the same code carries leader conditioning in step 2. Model code is in `hwdist.py`.

| Family | Shape | Notes |
|---|---|---|
| Lognormal | σ | special case of the generalized gamma with Q = 0 |
| Log-logistic | σ | |
| Weibull | σ | special case of the generalized gamma with Q = 1 |
| Gamma | k | special case of the generalized gamma with Q = σ |
| Generalized gamma | σ, Q | Prentice parameterisation |
| Exponential | none | classical random-arrival reference only |

**Checks:** AIC, AICc and BIC with Akaike weights; refitted parametric-bootstrap KS, AD and CvM tests on the PIT values inside the window; likelihood-ratio tests of the three nested families against the generalized gamma; observed vs fitted short- and long-headway shares.

**Outputs:** `outputs/tables/01_truncated_distribution_fit.xlsx`, plus figures `truncated_fit_density.png` and `truncated_fit_cdf_difference.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0            # s, extraction window (truncation bounds)
N_BOOT = 999                       # refitted parametric-bootstrap replicates per case x family
SEED = 20261106
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
FAMS = hd.FAMILIES                 # 5 candidates + exponential reference
CANDIDATES = [f for f in FAMS if f != "exponential"]
FAM_COL = PAL["family"]

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
assert d.Time_Headway_s.between(LOWER, UPPER).all(), "observation outside the extraction window"
data = {c: d[(d.Site == c[0]) & (d.Target_Class == c[1])].Time_Headway_s.to_numpy(float) for c in CASES}
print({f"{s} {t}": len(x) for (s, t), x in data.items()})

### Fit all families and bootstrap the goodness of fit

In [ ]:
fits, rows = {}, []
t0 = time.time()
for ci, case in enumerate(CASES):
    x = data[case]
    for fi, fam in enumerate(FAMS):
        r = hd.fit(x, fam, lower=LOWER, upper=UPPER); fits[(case, fam)] = r
        obs, p, valid = hd.bootstrap_gof(x, r, B=N_BOOT, seed=SEED + 100 * ci + fi)
        mu = r["beta"][0]; sh = r["shape"]
        q = hd.ppf(hd.cdf(LOWER, mu, fam, sh) + np.array([0.25, 0.5, 0.75]) *
                   (hd.cdf(UPPER, mu, fam, sh) - hd.cdf(LOWER, mu, fam, sh)), mu, fam, sh)
        grid = np.linspace(LOWER, UPPER, 4001)
        mean_w = np.trapezoid(grid * hd.trunc_pdf(grid, mu, fam, sh, LOWER, UPPER), grid)
        row = {"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "family_key": fam, "n": r["n"], "n_params": r["k"],
               "mu": mu, **{("k_shape" if nm == "k" else nm): v for nm, v in zip(hd.SHAPE_NAMES[fam], sh)},
               "logLik": r["loglik"], "AIC": r["AIC"], "AICc": r["AICc"], "BIC": r["BIC"],
               "fitted_mean_in_window_s": mean_w, "fitted_Q1_s": q[0], "fitted_median_s": q[1], "fitted_Q3_s": q[2],
               "obs_mean_s": x.mean(), "obs_median_s": np.median(x),
               "KS_D": obs[0], "KS_p": p[0], "AD_A2": obs[1], "AD_p": p[1], "CvM_W2": obs[2], "CvM_p": p[2],
               "boot_valid": valid}
        for t in (1.5, 2.0):
            row[f"obs_P(H<{t})"] = (x < t).mean(); row[f"fit_P(H<{t})"] = hd.trunc_cdf(t, mu, fam, sh, LOWER, UPPER)
        row["obs_P(H>4.0)"] = (x > 4.0).mean(); row["fit_P(H>4.0)"] = 1 - hd.trunc_cdf(4.0, mu, fam, sh, LOWER, UPPER)
        rows.append(row)
    print(f"{case} done  ({time.time() - t0:.0f} s)")
res = pd.DataFrame(rows)
cand = res.family_key.isin(CANDIDATES)
res["dAIC"] = np.nan; res["Akaike_weight"] = np.nan
for case, g in res[cand].groupby(["Site", "Class"]):
    da = g.AIC - g.AIC.min(); w = np.exp(-da / 2); w /= w.sum()
    res.loc[g.index, "dAIC"] = da; res.loc[g.index, "Akaike_weight"] = w
res.loc[~cand, "dAIC"] = res[~cand].AIC.values - res[cand].groupby(["Site", "Class"]).AIC.min().reindex(
    list(zip(res[~cand].Site, res[~cand].Class))).values
res["adequate_KS_AD_CvM"] = (res[["KS_p", "AD_p", "CvM_p"]] >= 0.05).all(axis=1)

### Ranking and adequacy by case

In [ ]:
cols = ["Site", "Class", "Family", "n_params", "logLik", "AIC", "dAIC", "Akaike_weight", "KS_p", "AD_p", "CvM_p", "adequate_KS_AD_CvM"]
for case, g in res.groupby(["Site", "Class"], sort=False):
    print(g.sort_values("AIC")[cols].round(3).to_string(index=False)); print()

### Parameters and fitted vs observed summaries (inside the window)

In [ ]:
pcols = ["Site", "Class", "Family", "mu", "sigma", "k_shape", "Q", "fitted_median_s", "obs_median_s", "fitted_mean_in_window_s", "obs_mean_s",
         "obs_P(H<1.5)", "fit_P(H<1.5)", "obs_P(H<2.0)", "fit_P(H<2.0)", "obs_P(H>4.0)", "fit_P(H>4.0)"]
print(res[res.family_key.isin(CANDIDATES)].reindex(columns=pcols).round(3).to_string(index=False))

### Nested likelihood-ratio tests against the generalized gamma

$Q=0$ gives the lognormal, $Q=1$ the Weibull and $Q=\sigma$ the gamma. Each test has 1 df (interior restriction). A small p-value means the two-parameter family is rejected in favour of the three-parameter generalized gamma.

In [ ]:
lr = []
for case in CASES:
    llg = fits[(case, "gengamma")]["loglik"]; Qhat = fits[(case, "gengamma")]["shape"][1]
    for fam in ["lognormal", "weibull", "gamma"]:
        stat = max(2 * (llg - fits[(case, fam)]["loglik"]), 0.0)
        lr.append({"Site": case[0], "Class": case[1], "Restricted": hd.LABELS[fam], "GG_Q_hat": Qhat,
                   "LR_stat": stat, "df": 1, "p": stats.chi2.sf(stat, 1)})
lr = pd.DataFrame(lr)
print(lr.round(4).to_string(index=False))

### Figure: fitted truncated densities

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.0), sharex=True)
grid = np.linspace(LOWER, UPPER, 400); bins = np.arange(LOWER, UPPER + 0.25, 0.25)
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    x = data[case]
    ax.hist(x, bins=bins, density=True, color="0.82", edgecolor="w", lw=.4)
    g = res[(res.Site == case[0]) & (res.Class == case[1])]
    for fam in FAMS:
        r = fits[(case, fam)]; dA = g[g.family_key == fam].dAIC.iloc[0]
        ax.plot(grid, hd.trunc_pdf(grid, r["beta"][0], fam, r["shape"], LOWER, UPPER), color=FAM_COL[fam],
                lw=1.0 if fam == "exponential" else 1.4, ls="--" if fam == "exponential" else "-",
                label=f"{hd.LABELS[fam]} ($\\Delta$AIC {dA:.1f})")
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={len(x)})", loc="left", fontsize=8.5)
    ax.legend(frameon=False, fontsize=6.3, loc="upper right"); ax.grid(alpha=.25, lw=.5)
for ax in axes[1]: ax.set_xlabel("Time headway (s)")
for ax in axes[:, 0]: ax.set_ylabel("Density (truncated to 0.5–5 s)")
fig.tight_layout(); save_fig(fig, "truncated_fit_density"); plt.show()

### Figure: where each family misfits

Empirical CDF minus fitted truncated CDF. A family that fits stays near zero everywhere. The grey band is the approximate 95% KS band for a fully specified model, $\pm 1.36/\sqrt{n}$, shown only for orientation; the bootstrap p-values in the table account for estimation.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 6.0), sharex=True)
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    x = np.sort(data[case]); n = len(x); ecdf = np.arange(1, n + 1) / n
    ax.axhspan(-1.36 / np.sqrt(n), 1.36 / np.sqrt(n), color="0.9", zorder=0)
    ax.axhline(0, color="0.4", lw=.7)
    for fam in CANDIDATES:
        r = fits[(case, fam)]
        ax.plot(x, ecdf - hd.trunc_cdf(x, r["beta"][0], fam, r["shape"], LOWER, UPPER), color=FAM_COL[fam], lw=1.2,
                label=hd.LABELS[fam])
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={n})", loc="left", fontsize=8.5)
    ax.grid(alpha=.25, lw=.5)
axes[0, 0].legend(frameon=False, fontsize=6.8, loc="lower left", ncol=2)
for ax in axes[1]: ax.set_xlabel("Time headway (s)")
for ax in axes[:, 0]: ax.set_ylabel("Empirical − fitted CDF")
fig.tight_layout(); save_fig(fig, "truncated_fit_cdf_difference"); plt.show()

### Export

In [ ]:
out = TAB / "01_truncated_distribution_fit.xlsx"
summary = (res[res.family_key.isin(CANDIDATES)].sort_values(["Site", "Class", "AIC"])
           .groupby(["Site", "Class"], sort=False)
           .apply(lambda g: pd.Series({"Best_by_AIC": g.Family.iloc[0], "Within_2_AIC": ", ".join(g[g.dAIC < 2].Family),
                                       "Adequate_families": ", ".join(g[g.adequate_KS_AD_CvM].Family) or "none"}),
                  include_groups=False).reset_index())
print(summary.to_string(index=False))
with pd.ExcelWriter(out) as xw:
    summary.to_excel(xw, sheet_name="summary", index=False)
    res.drop(columns="family_key").to_excel(xw, sheet_name="fits_gof", index=False)
    lr.to_excel(xw, sheet_name="nested_LR", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "N_BOOT", "SEED", "families", "data04_sha256"],
                  "Value": [LOWER, UPPER, N_BOOT, SEED, json.dumps(FAMS), hashlib.sha256(DATA.read_bytes()).hexdigest()]}
                 ).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 2 — Leader-conditioned AFT models of time headway (4 cases)

**Model:** for each site × target class case and each family from step 1, the location of the truncated AFT model depends only on the class of the leader,

$$\log H_i = \beta_0 + \sum_{c \ne \text{4W}} \beta_c\,\mathbb{1}[\text{leader}_i = c] + \sigma\,\varepsilon_i,\qquad 0.5 \le H_i \le 5\ \text{s},$$

with five leader classes (4W, BTW, CNG, MT_2W, NMT) and 4W as the reference. $e^{\beta_c}$ is a **time ratio**: the factor by which the whole headway distribution is stretched (> 1) or compressed (< 1) when following leader class $c$ instead of a four-wheeler. The shape is shared across leaders, so the leader effect is a pure scale shift. With n = 77 in the smallest case, nothing beyond leader class is fitted here; other variables are added one at a time in step 3.

**Truncation:** the likelihood is truncated to the 0.5–5 s extraction window, exactly as in step 1, so every result is conditional on following within the window.

**Steps:**
1. Fit the six families (five candidates + exponential reference) with and without leader class.
2. Likelihood-ratio test of leader class (4 df) and ΔAIC against the unconditional fit of step 1.
3. Choose one family for all four cases from the conditional fits (summed AIC across cases, with per-case ΔAIC and adequacy shown).
4. Check adequacy: refitted parametric-bootstrap KS, AD and CvM on the conditional PIT values, and Cox–Snell residual plots.
5. Report time ratios with 95% Wald CIs and model-based median headway by leader class.

**Outputs:** `outputs/tables/02_leader_conditioned_aft.xlsx`, and figures `leader_aft_time_ratios.png`, `leader_aft_cox_snell.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0            # s, extraction window (truncation bounds)
N_BOOT = 999                       # refitted parametric-bootstrap replicates per case x family
SEED = 20261107
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
REF = "4W"                         # reference leader class (largest class in 3 of 4 cases)
FAMS = hd.FAMILIES
CANDIDATES = [f for f in FAMS if f != "exponential"]
LEADER_COL = PAL["leader"]

### Data and design matrices

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
assert d.Time_Headway_s.between(LOWER, UPPER).all()
assert set(d.Leader_Class) == set(LEADERS)
NONREF = [c for c in LEADERS if c != REF]
data = {}
for case in CASES:
    g = d[(d.Site == case[0]) & (d.Target_Class == case[1])]
    X = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == c).to_numpy(float) for c in NONREF])
    data[case] = (g.Time_Headway_s.to_numpy(float), X, g.Leader_Class.to_numpy())
print(pd.crosstab([d.Site, d.Target_Class], d.Leader_Class).reindex(columns=LEADERS))

### Fit every family with and without leader class

In [ ]:
fits0, fits1, rows = {}, {}, []
for case in CASES:
    x, X, _ = data[case]
    for fam in FAMS:
        r0 = hd.fit(x, fam, lower=LOWER, upper=UPPER)
        r1 = hd.fit(x, fam, X=X, lower=LOWER, upper=UPPER)
        if r1["loglik"] < r0["loglik"]:        # safeguard: start the conditional fit from the nested solution
            r1 = hd.fit(x, fam, X=X, lower=LOWER, upper=UPPER, start=np.r_[r0["beta"], np.zeros(len(NONREF)), r0["theta"][1:]])
        fits0[(case, fam)], fits1[(case, fam)] = r0, r1
        lr = max(2 * (r1["loglik"] - r0["loglik"]), 0.0); df = len(NONREF)
        rows.append({"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "family_key": fam, "n": r1["n"],
                     "n_params_uncond": r0["k"], "n_params_leader": r1["k"],
                     "logLik_uncond": r0["loglik"], "logLik_leader": r1["loglik"],
                     "AIC_uncond": r0["AIC"], "AIC_leader": r1["AIC"], "AICc_leader": r1["AICc"], "BIC_leader": r1["BIC"],
                     "AIC_change": r1["AIC"] - r0["AIC"], "LR_stat": lr, "LR_df": df, "LR_p": stats.chi2.sf(lr, df),
                     **{("k_shape" if nm == "k" else nm): v for nm, v in zip(hd.SHAPE_NAMES[fam], r1["shape"])}})
comp = pd.DataFrame(rows)
cand = comp.family_key.isin(CANDIDATES)
comp["dAIC_leader"] = np.nan; comp["Akaike_weight_leader"] = np.nan
for _, g in comp[cand].groupby(["Site", "Class"]):
    da = g.AIC_leader - g.AIC_leader.min(); w = np.exp(-da / 2); w /= w.sum()
    comp.loc[g.index, "dAIC_leader"] = da; comp.loc[g.index, "Akaike_weight_leader"] = w
cols = ["Site", "Class", "Family", "AIC_uncond", "AIC_leader", "AIC_change", "LR_stat", "LR_p", "dAIC_leader", "Akaike_weight_leader"]
for _, g in comp.groupby(["Site", "Class"], sort=False):
    print(g.sort_values("AIC_leader")[cols].round(4).to_string(index=False)); print()

### Choice of one family for all four cases

The same family is used in every case so the time ratios are comparable across sites and target classes.

**Rule (parsimony with a nested check):**
1. Rank the families by leader-conditioned AIC summed over the four cases (equivalent to a joint model with case-specific parameters).
2. Lognormal (Q = 0), Weibull (Q = 1) and gamma (Q = σ) are nested in the generalized gamma. For each, a joint likelihood-ratio test against the generalized gamma sums the per-case LR statistics (4 df).
3. The chosen family is the best-ranked two-parameter family that the joint test does not reject at 5%. The generalized gamma is used only if every nested family is rejected.

The extra shape parameter is not carried unless the data demand it. In a small case it can also sit on a weakly identified ridge (|Q| large, σ near 0), so the fitted Q is shown.

In [ ]:
gg_ll = {case: fits1[(case, "gengamma")]["loglik"] for case in CASES}
gg_Q = {case: fits1[(case, "gengamma")]["shape"][1] for case in CASES}
comp["GG_Q_leader"] = [gg_Q[(a, b)] for a, b in zip(comp.Site, comp.Class)]
comp["LR_vs_GG"] = [max(2 * (gg_ll[(a, b)] - ll), 0.0) if k in ("lognormal", "weibull", "gamma") else np.nan
                    for a, b, k, ll in zip(comp.Site, comp.Class, comp.family_key, comp.logLik_leader)]
comp["LR_vs_GG_p"] = stats.chi2.sf(comp.LR_vs_GG, 1)
print("generalized gamma Q (leader-conditioned):", {f"{a} {b}": round(q, 3) for (a, b), q in gg_Q.items()})
choice = (comp[cand].groupby("Family")
          .agg(family_key=("family_key", "first"), sum_AIC_leader=("AIC_leader", "sum"), max_dAIC_in_a_case=("dAIC_leader", "max"),
               cases_best=("dAIC_leader", lambda s: int((s == 0).sum())), cases_within_2=("dAIC_leader", lambda s: int((s < 2).sum())),
               joint_LR_vs_GG=("LR_vs_GG", lambda s: s.sum() if s.notna().all() else np.nan))
          .sort_values("sum_AIC_leader"))
choice["n_shape"] = [hd.N_SHAPE[k] for k in choice.family_key]
choice["d_sum_AIC"] = choice.sum_AIC_leader - choice.sum_AIC_leader.min()
choice["joint_LR_df"] = np.where(choice.joint_LR_vs_GG.notna(), len(CASES), np.nan)
choice["joint_LR_p"] = stats.chi2.sf(choice.joint_LR_vs_GG, len(CASES))
choice["rejected_by_GG"] = choice.joint_LR_p < 0.05
choice = choice.reset_index()
print(choice.drop(columns="family_key").round(4).to_string(index=False))
two = choice[(choice.n_shape == 1) & ~choice.rejected_by_GG].sort_values("sum_AIC_leader")
CHOSEN = two.family_key.iloc[0] if len(two) else "gengamma"
print("\nchosen family:", hd.LABELS[CHOSEN])

### Adequacy of the conditional fits: refitted parametric bootstrap on PIT values

In [ ]:
gof = []; t0 = time.time()
for ci, case in enumerate(CASES):
    x, X, _ = data[case]
    for fi, fam in enumerate(CANDIDATES):
        obs, p, valid = hd.bootstrap_gof(x, fits1[(case, fam)], B=N_BOOT, seed=SEED + 100 * ci + fi)
        gof.append({"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "KS_D": obs[0], "KS_p": p[0],
                    "AD_A2": obs[1], "AD_p": p[1], "CvM_W2": obs[2], "CvM_p": p[2], "boot_valid": valid})
    print(f"{case} done ({time.time() - t0:.0f} s)")
gof = pd.DataFrame(gof)
gof["adequate_KS_AD_CvM"] = (gof[["KS_p", "AD_p", "CvM_p"]] >= 0.05).all(axis=1)
comp = comp.merge(gof, on=["Site", "Class", "Family"], how="left")
print(gof.round(3).to_string(index=False))

### Time ratios by leader class (all candidate families)

$e^{\hat\beta_c}$ with 95% Wald intervals from the inverse observed information. The chosen family is reported in the manuscript; the others show how sensitive the leader effects are to the family.

In [ ]:
tr = []
for case in CASES:
    x, X, lead = data[case]
    for fam in CANDIDATES:
        r = fits1[(case, fam)]; V = hd.covariance(r, x); se = np.sqrt(np.clip(np.diag(V)[:X.shape[1]], 0, None))
        for j, c in enumerate(NONREF, start=1):
            b = r["beta"][j]
            tr.append({"Site": case[0], "Class": case[1], "Family": hd.LABELS[fam], "Leader": c, "n_leader": int((lead == c).sum()),
                       "n_ref": int((lead == REF).sum()), "beta": b, "SE": se[j], "z": b / se[j], "p_Wald": 2 * stats.norm.sf(abs(b / se[j])),
                       "TR": np.exp(b), "TR_lo95": np.exp(b - 1.96 * se[j]), "TR_hi95": np.exp(b + 1.96 * se[j])})
tr = pd.DataFrame(tr)
print(tr[tr.Family == hd.LABELS[CHOSEN]].round(3).to_string(index=False))

### Model-based headway by leader class (chosen family)

Median and mean of the fitted truncated distribution for each leader class, beside the observed values. Because the window truncates the distribution, the fitted median ratio between two leaders is close to, but not exactly, the time ratio.

In [ ]:
grid = np.linspace(LOWER, UPPER, 4001); med = []
for case in CASES:
    x, X, lead = data[case]; r = fits1[(case, CHOSEN)]
    for j, c in enumerate(LEADERS):
        mu = r["beta"][0] + (r["beta"][NONREF.index(c) + 1] if c != REF else 0.0)
        Fa, Fb = hd.cdf(LOWER, mu, CHOSEN, r["shape"]), hd.cdf(UPPER, mu, CHOSEN, r["shape"])
        fmed = hd.ppf(Fa + 0.5 * (Fb - Fa), mu, CHOSEN, r["shape"])
        fmean = np.trapezoid(grid * hd.trunc_pdf(grid, mu, CHOSEN, r["shape"], LOWER, UPPER), grid)
        xs = x[lead == c]
        med.append({"Site": case[0], "Class": case[1], "Leader": c, "n": len(xs), "obs_median_s": np.median(xs), "fit_median_s": fmed,
                    "obs_mean_s": xs.mean(), "fit_mean_s": fmean, "obs_P(H<2.0)": (xs < 2).mean(),
                    "fit_P(H<2.0)": hd.trunc_cdf(2.0, mu, CHOSEN, r["shape"], LOWER, UPPER)})
med = pd.DataFrame(med)
print(med.round(3).to_string(index=False))

### Figure: time ratios by leader class (chosen family)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(7.4, 2.9), sharey=True, sharex=True)
sub = tr[tr.Family == hd.LABELS[CHOSEN]]
ypos = np.arange(len(NONREF))[::-1]
for ax, case, tag in zip(axes, CASES, "abcd"):
    g = sub[(sub.Site == case[0]) & (sub.Class == case[1])].set_index("Leader").loc[NONREF]
    ax.axvline(1, color="0.4", lw=.7)
    for y, c in zip(ypos, NONREF):
        ax.plot([g.loc[c, "TR_lo95"], g.loc[c, "TR_hi95"]], [y, y], color=LEADER_COL[c], lw=1.4)
        ax.plot(g.loc[c, "TR"], y, "o", color=LEADER_COL[c], ms=4)
    ax.set_title(f"({tag}) {case[0]}, {case[1]}\n(n={len(data[case][0])})", loc="left", fontsize=8)
    ax.set_xscale("log"); ax.set_xticks([0.5, 0.7, 1, 1.4]); ax.set_xticklabels(["0.5", "0.7", "1", "1.4"])
    ax.minorticks_off(); ax.grid(axis="x", alpha=.25, lw=.5)
axes[0].set_yticks(ypos); axes[0].set_yticklabels(NONREF)
axes[0].set_ylabel("Leader class (reference: 4W)")
fig.supxlabel(f"Time ratio vs 4W leader, {hd.LABELS[CHOSEN]} AFT (95% CI, log scale)", fontsize=9)
fig.tight_layout(); save_fig(fig, "leader_aft_time_ratios"); plt.show()

### Figure: Cox–Snell residuals (chosen family)

$r_i = -\log\{1-\hat F_{\text{trunc}}(H_i \mid \text{leader}_i)\}$. If the conditional model is right, the $r_i$ are a unit-exponential sample, so their Nelson–Aalen cumulative hazard follows the 45° line.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(6.0, 5.6))
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    x, X, _ = data[case]; u = np.clip(hd.pit(x, fits1[(case, CHOSEN)]), 1e-12, 1 - 1e-12)
    rcs = np.sort(-np.log1p(-u)); n = len(rcs); na = np.cumsum(1 / (n - np.arange(n)))
    m = max(rcs.max(), na.max()) * 1.03
    ax.plot([0, m], [0, m], color="0.4", lw=.8, ls="--")
    ax.step(rcs, na, where="post", color=PAL["family"][CHOSEN], lw=1.2)
    ax.set_xlim(0, m); ax.set_ylim(0, m); ax.set_aspect("equal")
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={n})", loc="left", fontsize=8.5); ax.grid(alpha=.25, lw=.5)
for ax in axes[1]: ax.set_xlabel("Cox–Snell residual")
for ax in axes[:, 0]: ax.set_ylabel("Nelson–Aalen cumulative hazard")
fig.tight_layout(); save_fig(fig, "leader_aft_cox_snell"); plt.show()

### Export

In [ ]:
out = TAB / "02_leader_conditioned_aft.xlsx"
ch = comp[comp.family_key == CHOSEN]
summary = ch[["Site", "Class", "n", "AIC_uncond", "AIC_leader", "AIC_change", "LR_stat", "LR_df", "LR_p", "dAIC_leader",
              "KS_p", "AD_p", "CvM_p", "adequate_KS_AD_CvM"]].copy()
summary.insert(2, "Family", hd.LABELS[CHOSEN])
summary["Best_conditional_by_AIC"] = [comp[(comp.Site == s) & (comp.Class == c) & cand].sort_values("AIC_leader").Family.iloc[0]
                                      for s, c in zip(summary.Site, summary.Class)]
print(summary.round(4).to_string(index=False))
with pd.ExcelWriter(out) as xw:
    summary.to_excel(xw, sheet_name="summary", index=False)
    choice.drop(columns="family_key").to_excel(xw, sheet_name="family_choice", index=False)
    comp.drop(columns="family_key").to_excel(xw, sheet_name="fits_LR_gof", index=False)
    tr[tr.Family == hd.LABELS[CHOSEN]].to_excel(xw, sheet_name="time_ratios", index=False)
    tr.to_excel(xw, sheet_name="time_ratios_all_families", index=False)
    med.to_excel(xw, sheet_name="headway_by_leader", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "N_BOOT", "SEED", "families", "leader_classes", "reference_leader", "chosen_family",
                                "data04_sha256"],
                  "Value": [LOWER, UPPER, N_BOOT, SEED, json.dumps(FAMS), json.dumps(LEADERS), REF, hd.LABELS[CHOSEN],
                            hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 2b — How each leader class shifts the headway distribution (4 cases)

Notebook 02 gives time ratios against a four-wheeler leader. Here every leader class is compared with the **main distribution** of its case, which is the headway distribution of all events in that site × target class. The comparison answers two questions.

**1. Type of effect.** In the leader-conditioned Weibull AFT, a leader can only **shift** the distribution along the time axis, compressing it (shorter headways) or stretching it (longer headways). The shape is held the same. To check that assumption, the model is extended so the Weibull shape σ also depends on the leader, which would let a leader make headways more or less **dispersed** as well. A likelihood-ratio test (4 df) per case says whether that extra freedom is needed.

**2. Extent of effect.** For each leader class, against the main distribution:
- the median ratio (leader median ÷ main median, close to a time ratio against the case average);
- the change in median and mean headway (s and %);
- the change in the share of short headways, P(H < 2 s), in percentage points;
- the Kolmogorov distance between the two distributions (maximum CDF gap).

Under the same fitted model, the main distribution is the mixture of the five leader-specific distributions, weighted by the observed leader shares. This keeps leader and main on one consistent footing. It is checked against the unconditional Weibull fit of step 1.

**Uncertainty:** 95% intervals from 4,000 draws of the parameter vector from its asymptotic normal distribution (inverse observed information).

**Classification:** compresses if the median-ratio interval lies wholly below 1; stretches if it lies wholly above 1; otherwise no clear shift.

**Outputs:** `outputs/tables/02b_leader_distribution_shifts.xlsx`, and figures `leader_shifted_densities.png`, `leader_shift_vs_main.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from scipy import stats, optimize
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0
FAMILY = "weibull"                 # chosen in notebook 02 (checked below)
_chosen02 = pd.read_excel(TAB / "02_leader_conditioned_aft.xlsx", sheet_name="settings").set_index("Parameter").Value["chosen_family"]
assert _chosen02 == hd.LABELS[FAMILY], f"notebook 02 chose {_chosen02}"
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]; REF = "4W"
NONREF = [c for c in LEADERS if c != REF]
SHORT = 2.0                        # s, short-headway threshold
N_DRAW = 4000; SEED = 20261107
LEADER_COL = PAL["leader"]
GRID = np.linspace(LOWER, UPPER, 2001)

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
data = {}
for case in CASES:
    g = d[(d.Site == case[0]) & (d.Target_Class == case[1])]
    X = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == c).to_numpy(float) for c in NONREF])
    share = g.Leader_Class.value_counts(normalize=True).reindex(LEADERS).to_numpy()
    data[case] = (g.Time_Headway_s.to_numpy(float), X, g.Leader_Class.to_numpy(), share)

### Type of effect: does the leader also change the spread?

Base model: location depends on leader, one shared σ (step 2). Extended model: $\log\sigma_i = \alpha_0 + \sum_c \alpha_c\,\mathbb{1}[\text{leader}_i=c]$ as well. If the LR test is not significant, the leader effect is a pure shift (compression or stretch) of a common shape.

In [ ]:
def nll_het(theta, x, X):
    p = X.shape[1]; mu = X @ theta[:p]; s = np.exp(X @ theta[p:])
    with np.errstate(all="ignore"):
        lf = hd.logpdf(x, mu, FAMILY, [s])
        Z = hd.cdf(UPPER, mu, FAMILY, [s]) - hd.cdf(LOWER, mu, FAMILY, [s])
        if not np.all(np.isfinite(Z)) or Z.min() < 1e-10:
            return 1e12
        v = -(lf.sum() - np.log(Z).sum())
    return v if np.isfinite(v) else 1e12

fits, spread = {}, []
for case in CASES:
    x, X, lead, share = data[case]
    r = hd.fit(x, FAMILY, X=X, lower=LOWER, upper=UPPER); fits[case] = r
    t0 = np.r_[r["beta"], r["theta"][-1], np.zeros(len(NONREF))]
    o = optimize.minimize(nll_het, t0, args=(x, X), method="L-BFGS-B")
    o = optimize.minimize(nll_het, o.x, args=(x, X), method="Nelder-Mead", options={"maxiter": 40000, "xatol": 1e-9, "fatol": 1e-11})
    ll1 = -o.fun; lr = max(2 * (ll1 - r["loglik"]), 0.0)
    sig = np.exp(o.x[X.shape[1]:][0] + np.r_[0, o.x[X.shape[1] + 1:]])
    spread.append({"Site": case[0], "Class": case[1], "n": len(x), "logLik_shift_only": r["loglik"], "logLik_shift_and_spread": ll1,
                   "LR_stat": lr, "df": len(NONREF), "LR_p": stats.chi2.sf(lr, len(NONREF)),
                   "AIC_shift_only": r["AIC"], "AIC_shift_and_spread": 2 * (r["k"] + len(NONREF)) - 2 * ll1,
                   "sigma_shared": r["shape"][0], **{f"sigma_{c}": s_ for c, s_ in zip(LEADERS, sig)}})
spread = pd.DataFrame(spread)
spread["dAIC_spread_minus_shift"] = spread.AIC_shift_and_spread - spread.AIC_shift_only
print(spread.round(3).to_string(index=False))

### Extent of effect: each leader against the main distribution

In [ ]:
def tcdf(t, mu, s):  return hd.trunc_cdf(t, mu, FAMILY, [s], LOWER, UPPER)
def tpdf(t, mu, s):  return hd.trunc_pdf(t, mu, FAMILY, [s], LOWER, UPPER)

def summaries(beta, s, share):
    # leader-specific and mixture (main) summaries for one parameter vector
    mus = beta[0] + np.r_[0.0, beta[1:]]
    C = np.array([tcdf(GRID, m, s) for m in mus]); P = np.array([tpdf(GRID, m, s) for m in mus])
    Cm, Pm = share @ C, share @ P
    q = lambda cdfv, u: np.interp(u, cdfv, GRID)
    out = {"main": {"median": q(Cm, .5), "mean": np.trapezoid(GRID * Pm, GRID), "pshort": np.interp(SHORT, GRID, Cm),
                    "q10": q(Cm, .1), "q90": q(Cm, .9)}}
    for c, Ci, Pi in zip(LEADERS, C, P):
        out[c] = {"median": q(Ci, .5), "mean": np.trapezoid(GRID * Pi, GRID), "pshort": np.interp(SHORT, GRID, Ci),
                  "q10": q(Ci, .1), "q90": q(Ci, .9), "KS": np.abs(Ci - Cm).max()}
    return out, C, P, Cm, Pm

rng = np.random.default_rng(SEED)
rows, curves, main_rows = [], {}, []
for case in CASES:
    x, X, lead, share = data[case]; r = fits[case]
    est, C, P, Cm, Pm = summaries(r["beta"], r["shape"][0], share); curves[case] = (P, Pm)
    V = hd.covariance(r, x); draws = rng.multivariate_normal(r["theta"], V, size=N_DRAW)
    sims = [summaries(t[:X.shape[1]], np.exp(t[-1]), share)[0] for t in draws]
    r01 = hd.fit(x, FAMILY, lower=LOWER, upper=UPPER)          # unconditional fit (notebook 01) as a check on the mixture
    Fa, Fb = hd.cdf(LOWER, r01["beta"][0], FAMILY, r01["shape"]), hd.cdf(UPPER, r01["beta"][0], FAMILY, r01["shape"])
    main_rows.append({"Site": case[0], "Class": case[1], "n": len(x), "obs_median_s": np.median(x), "mixture_median_s": est["main"]["median"],
                      "uncond_fit_median_s": hd.ppf(Fa + .5 * (Fb - Fa), r01["beta"][0], FAMILY, r01["shape"]),
                      "obs_mean_s": x.mean(), "mixture_mean_s": est["main"]["mean"], f"obs_P(H<{SHORT})": (x < SHORT).mean(),
                      f"mixture_P(H<{SHORT})": est["main"]["pshort"], "mixture_Q10_s": est["main"]["q10"], "mixture_Q90_s": est["main"]["q90"]})
    for c in LEADERS:
        e, m = est[c], est["main"]
        def ci(f):
            v = np.array([f(sd[c], sd["main"]) for sd in sims]); return np.percentile(v, [2.5, 97.5])
        mr = e["median"] / m["median"]; mr_ci = ci(lambda a, b: a["median"] / b["median"])
        xs = x[lead == c]
        rows.append({"Site": case[0], "Class": case[1], "Leader": c, "n_leader": len(xs), "share": share[LEADERS.index(c)],
                     "median_s": e["median"], "main_median_s": m["median"], "median_ratio": mr, "median_ratio_lo95": mr_ci[0], "median_ratio_hi95": mr_ci[1],
                     "d_median_s": e["median"] - m["median"], "d_median_pct": 100 * (mr - 1),
                     "d_mean_s": e["mean"] - m["mean"], "d_mean_pct": 100 * (e["mean"] / m["mean"] - 1),
                     **dict(zip(["d_mean_pct_lo95", "d_mean_pct_hi95"], 100 * (ci(lambda a, b: a["mean"] / b["mean"]) - 1))),
                     f"P(H<{SHORT})": e["pshort"], f"main_P(H<{SHORT})": m["pshort"], "d_P_short_pp": 100 * (e["pshort"] - m["pshort"]),
                     **dict(zip(["d_P_short_pp_lo95", "d_P_short_pp_hi95"], 100 * ci(lambda a, b: a["pshort"] - b["pshort"]))),
                     "Q10_s": e["q10"], "Q90_s": e["q90"], "KS_distance_to_main": e["KS"],
                     "obs_median_s": np.median(xs), f"obs_P(H<{SHORT})": (xs < SHORT).mean()})
shift = pd.DataFrame(rows)
shift["Effect_type"] = np.select([shift.median_ratio_hi95 < 1, shift.median_ratio_lo95 > 1],
                                 ["compresses (shorter headways)", "stretches (longer headways)"], "no clear shift")
main = pd.DataFrame(main_rows)
print(main.round(3).to_string(index=False)); print()
print(shift[["Site", "Class", "Leader", "n_leader", "median_s", "main_median_s", "median_ratio", "median_ratio_lo95", "median_ratio_hi95",
             "d_median_s", "d_P_short_pp", "d_P_short_pp_lo95", "d_P_short_pp_hi95", "KS_distance_to_main", "Effect_type"]].round(3).to_string(index=False))

### Summary by leader class across the four cases

In [ ]:
summary = (shift.pivot_table(index="Leader", columns=["Site", "Class"], values="d_median_pct", sort=False).round(1)
           .reindex(LEADERS))
summary.columns = [f"{a} {b}: median change %" for a, b in summary.columns]
types = shift.pivot_table(index="Leader", columns=["Site", "Class"], values="Effect_type", aggfunc="first", sort=False).reindex(LEADERS)
types.columns = [f"{a} {b}: effect" for a, b in types.columns]
summary = pd.concat([summary, types], axis=1).reset_index()
print(summary.to_string(index=False))

### Figure: leader-specific headway distributions against the main distribution

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(7.2, 5.8), sharex=True)
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    P, Pm = curves[case]
    for c, Pi in zip(LEADERS, P):
        ax.plot(GRID, Pi, color=LEADER_COL[c], lw=1.2, label=c)
    ax.plot(GRID, Pm, color="k", lw=2.0, ls="--", label="Main (all leaders)")
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={len(data[case][0])})", loc="left", fontsize=8.5)
    ax.grid(alpha=.25, lw=.5); ax.set_ylim(bottom=0)
axes[0, 1].legend(frameon=False, fontsize=7, loc="upper right")
for ax in axes[1]: ax.set_xlabel("Time headway (s)")
for ax in axes[:, 0]: ax.set_ylabel("Density (truncated to 0.5–5 s)")
fig.tight_layout(); save_fig(fig, "leader_shifted_densities"); plt.show()

### Figure: extent of each leader's shift against the main distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4))
ypos = {case: i for i, case in enumerate(CASES[::-1])}; off = dict(zip(LEADERS, np.linspace(0.28, -0.28, len(LEADERS))))
for c in LEADERS:
    s_ = shift[shift.Leader == c]
    y = np.array([ypos[(a, b)] for a, b in zip(s_.Site, s_.Class)]) + off[c]
    for ax, v, lo, hi in [(axes[0], s_.d_median_pct, 100 * (s_.median_ratio_lo95 - 1), 100 * (s_.median_ratio_hi95 - 1)),
                          (axes[1], s_.d_P_short_pp, s_.d_P_short_pp_lo95, s_.d_P_short_pp_hi95)]:
        ax.errorbar(v, y, xerr=[v - lo, hi - v], fmt="o", ms=3.5, color=LEADER_COL[c], elinewidth=1.1, capsize=0, label=c)
for ax, lab in zip(axes, ["Change in median headway vs main (%)", f"Change in P(H < {SHORT:g} s) vs main (pp)"]):
    ax.axvline(0, color="0.4", lw=.7); ax.set_xlabel(lab); ax.grid(axis="x", alpha=.25, lw=.5)
    ax.set_yticks(list(ypos.values())); ax.set_yticklabels([f"{a}, {b}" for a, b in ypos])
    for k in range(len(CASES) - 1): ax.axhline(k + 0.5, color="0.85", lw=.6)
axes[1].set_yticklabels([])
axes[0].set_title("(a) Shift of the centre", loc="left", fontsize=8.5); axes[1].set_title("(b) Shift in short headways", loc="left", fontsize=8.5)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, frameon=False, fontsize=7.5, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.04))
fig.tight_layout(rect=(0, 0.05, 1, 1)); save_fig(fig, "leader_shift_vs_main"); plt.show()

### Export

In [ ]:
out = TAB / "02b_leader_distribution_shifts.xlsx"
with pd.ExcelWriter(out) as xw:
    summary.to_excel(xw, sheet_name="summary", index=False)
    shift.to_excel(xw, sheet_name="leader_vs_main", index=False)
    main.to_excel(xw, sheet_name="main_distribution", index=False)
    spread.to_excel(xw, sheet_name="type_shift_vs_spread", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "family", "leader_classes", "main_distribution", "short_headway_threshold_s",
                                "N_DRAW", "SEED", "data04_sha256"],
                  "Value": [LOWER, UPPER, hd.LABELS[FAMILY], json.dumps(LEADERS), "mixture of leader-specific fits, observed leader shares",
                            SHORT, N_DRAW, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 3 — Other variables as time ratios: crude and leader-adjusted Weibull AFT (4 cases)

**One variable at a time:** with n = 77 in the smallest case, the variables are not entered jointly. Each is fitted in two ways, separately for each site × target class case, with the truncated Weibull AFT chosen in step 2:

- **Crude (single-variable):** the variable alone, $\log H_i = \beta_0 + \gamma\,z_i + \sigma\varepsilon_i$. This is the plain association of the variable with headway, whatever the leader.
- **Leader-adjusted:** the variable added to the leader-only model, $\log H_i = \beta_0 + \sum_c \beta_c\,\mathbb{1}[\text{leader}_i=c] + \gamma\,z_i + \sigma\varepsilon_i$ (reference leader 4W). This is the association among events with the same leader class.

Both are truncated to $0.5\le H_i\le 5$ s. Each is reported as a time ratio $e^{\hat\gamma}$ with a 95% Wald CI and a 1-df likelihood-ratio p-value against the same model without the variable, Holm-adjusted across the three variables within each case and model.

**How to read the two:** if a crude effect shrinks once leader class is added, it was carried by the leader mix (for example, faster leaders are more often four-wheelers). If it holds, it acts within leader classes.

| Variable | Coding | Time ratio means |
|---|---|---|
| Off_Centered | 1 = follower laterally offset from the leader's path | headway when offset ÷ headway when in line |
| Occupied | 1 = follower carrying passenger(s) | headway when occupied ÷ headway when empty |
| Leader speed | km/h, centred at the case mean, per 5 km/h | headway change for a leader 5 km/h faster |

**Not entered:** the follower's own speed. Headway is spacing divided by speed, so follower speed is part of the outcome; it is handled exactly by the decomposition in step 4.

**Checks:** how much each variable moves the leader time ratios of step 2 (confounding check), and whether the time ratios hold across the other candidate families (sensitivity, both models).

**Outputs:** `outputs/tables/03_covariate_time_ratios.xlsx`, and figure `covariate_time_ratios.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
LOWER, UPPER = 0.5, 5.0
FAMILY = "weibull"                 # chosen in notebook 02 (checked against its settings sheet below)
_chosen02 = pd.read_excel(HERE.parent / "outputs_TRA" / "tables" / "02_leader_conditioned_aft.xlsx", sheet_name="settings").set_index("Parameter").Value["chosen_family"]
assert _chosen02 == hd.LABELS[FAMILY], f"notebook 02 chose {_chosen02}"
SENS_FAMILIES = ["lognormal", "loglogistic", "gamma", "gengamma"]
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]; REF = "4W"
NONREF = [c for c in LEADERS if c != REF]
SPEED_UNIT = 5.0                   # km/h per leader-speed step
VARS = {"Off_Centered": "Off-centred following", "Occupied": "Occupied (passengers)",
        "Leader_speed_5kmh": "Leader speed (+5 km/h)"}
VAR_COL = PAL["variable"]

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
assert d.Time_Headway_s.between(LOWER, UPPER).all()
data, desc = {}, []
for case in CASES:
    g = d[(d.Site == case[0]) & (d.Target_Class == case[1])].copy()
    g["Leader_speed_5kmh"] = (g.Leading_Speed_kmh - g.Leading_Speed_kmh.mean()) / SPEED_UNIT
    base = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == c).to_numpy(float) for c in NONREF])
    z = {"Off_Centered": g.Off_Centered.to_numpy(float), "Occupied": g.Occupied.to_numpy(float),
         "Leader_speed_5kmh": g.Leader_speed_5kmh.to_numpy(float)}
    data[case] = (g.Time_Headway_s.to_numpy(float), base, z)
    desc.append({"Site": case[0], "Class": case[1], "n": len(g), "n_Off_Centered": int(g.Off_Centered.sum()),
                 "n_Occupied": int(g.Occupied.sum()), "Leader_speed_mean_kmh": g.Leading_Speed_kmh.mean(),
                 "Leader_speed_sd_kmh": g.Leading_Speed_kmh.std()})
desc = pd.DataFrame(desc); print(desc.round(2).to_string(index=False))
MODELS = {"crude": "Crude (variable alone)", "leader_adjusted": "Leader-adjusted"}
def base_X(X0, model):
    return X0[:, :1] if model == "crude" else X0

### One-at-a-time additions: crude and leader-adjusted

In [ ]:
def fit_added(x, fam, X1, r0):
    # fit base + one variable from a warm start and from the default starts; keep the higher likelihood
    p0 = r0["X"].shape[1]
    cands = [hd.fit(x, fam, X=X1, lower=LOWER, upper=UPPER, start=np.r_[r0["beta"], 0.0, r0["theta"][p0:]]),
             hd.fit(x, fam, X=X1, lower=LOWER, upper=UPPER)]
    r1 = max(cands, key=lambda r: r["loglik"])
    assert r1["loglik"] >= r0["loglik"] - 1e-6, f"added-variable fit below base ({fam})"
    return r1

def wald(r, x, j):
    V = hd.covariance(r, x); se = float(np.sqrt(max(V[j, j], 0)))
    b = float(r["beta"][j]); return b, se

rows, leader_shift = [], []
for case in CASES:
    x, X0full, z = data[case]
    for model in MODELS:
        X0 = base_X(X0full, model)
        r0 = hd.fit(x, FAMILY, X=X0, lower=LOWER, upper=UPPER)
        for v in VARS:
            X1 = np.column_stack([X0, z[v]])
            r1 = fit_added(x, FAMILY, X1, r0)
            lr = max(2 * (r1["loglik"] - r0["loglik"]), 0.0)
            b, se = wald(r1, x, X1.shape[1] - 1)
            rows.append({"Site": case[0], "Class": case[1], "Model": model, "Variable": v, "Label": VARS[v], "n": len(x),
                         "gamma": b, "SE": se, "TR": np.exp(b), "TR_lo95": np.exp(b - 1.96 * se), "TR_hi95": np.exp(b + 1.96 * se),
                         "p_Wald": 2 * stats.norm.sf(abs(b / se)), "LR_stat": lr, "LR_p": stats.chi2.sf(lr, 1),
                         "AIC_base": r0["AIC"], "AIC_with_var": r1["AIC"], "dAIC": r1["AIC"] - r0["AIC"],
                         "sigma_base": r0["shape"][0], "sigma_with_var": r1["shape"][0]})
            if model == "leader_adjusted":
                for c, a, bb in zip(NONREF, r0["beta"][1:], r1["beta"][1:1 + len(NONREF)]):
                    leader_shift.append({"Site": case[0], "Class": case[1], "Variable": v, "Leader": c,
                                         "TR_leader_base": np.exp(a), "TR_leader_with_var": np.exp(bb),
                                         "pct_change_TR": 100 * (np.exp(bb - a) - 1)})
res = pd.DataFrame(rows)
res["LR_p_Holm"] = np.nan
for _, g in res.groupby(["Site", "Class", "Model"]):
    res.loc[g.index, "LR_p_Holm"] = multipletests(g.LR_p, method="holm")[1]
leader_shift = pd.DataFrame(leader_shift)
print(res[["Site", "Class", "Model", "Label", "TR", "TR_lo95", "TR_hi95", "LR_p", "LR_p_Holm", "dAIC"]].round(4).to_string(index=False))

### Crude vs leader-adjusted, side by side

`attenuation_pct` is how much of the crude log time ratio is removed by adjusting for leader class, $100\,(1-\hat\gamma_{\text{adj}}/\hat\gamma_{\text{crude}})$. It is shown only where the crude effect is at least 2% ($|e^{\hat\gamma}-1|\ge 0.02$); below that the ratio is unstable.

In [ ]:
side = res.pivot_table(index=["Site", "Class", "Variable"], columns="Model", values=["TR", "TR_lo95", "TR_hi95", "LR_p_Holm"], sort=False)
side.columns = [f"{m}_{k}" for k, m in side.columns]
side = side.reset_index()
gc, ga = np.log(side.crude_TR), np.log(side.leader_adjusted_TR)
side["attenuation_pct"] = np.where(np.abs(side.crude_TR - 1) >= 0.02, 100 * (1 - ga / gc), np.nan)
side = side[["Site", "Class", "Variable", "crude_TR", "crude_TR_lo95", "crude_TR_hi95", "crude_LR_p_Holm",
             "leader_adjusted_TR", "leader_adjusted_TR_lo95", "leader_adjusted_TR_hi95", "leader_adjusted_LR_p_Holm", "attenuation_pct"]]
print(side.round(3).to_string(index=False))

### Check: how far each variable moves the leader time ratios

In [ ]:
shift = (leader_shift.assign(abs_pct=lambda t: t.pct_change_TR.abs())
         .groupby(["Site", "Class", "Variable"], sort=False).abs_pct.max().rename("max_abs_pct_change_leader_TR").reset_index())
print(shift.round(2).to_string(index=False))

### Sensitivity: the same time ratios under the other candidate families

In [ ]:
sens = []
for case in CASES:
    x, X0full, z = data[case]
    for model in MODELS:
        X0 = base_X(X0full, model)
        for fam in [FAMILY] + SENS_FAMILIES:
            r0 = hd.fit(x, fam, X=X0, lower=LOWER, upper=UPPER)
            for v in VARS:
                X1 = np.column_stack([X0, z[v]])
                r1 = fit_added(x, fam, X1, r0)
                b, se = wald(r1, x, X1.shape[1] - 1); lr = max(2 * (r1["loglik"] - r0["loglik"]), 0.0)
                sens.append({"Site": case[0], "Class": case[1], "Model": model, "Variable": v, "Family": hd.LABELS[fam], "TR": np.exp(b),
                             "TR_lo95": np.exp(b - 1.96 * se), "TR_hi95": np.exp(b + 1.96 * se), "LR_p": stats.chi2.sf(lr, 1),
                             "GG_Q": r1["shape"][1] if fam == "gengamma" else np.nan})
sens = pd.DataFrame(sens)
sens["weakly_identified"] = sens.GG_Q.abs() > 5      # generalized gamma on the high-|Q| ridge: time ratio not interpretable
print("weakly identified fits:", sens[sens.weakly_identified][["Site", "Class", "Model", "Variable", "GG_Q"]].to_dict("records"))
print(sens.pivot_table(index=["Site", "Class", "Model", "Variable"], columns="Family", values="TR", sort=False).round(3).to_string())

### Figure: time ratios of the added variables, crude and leader-adjusted (Weibull)

In [ ]:
from matplotlib.lines import Line2D
fig, axes = plt.subplots(2, 2, figsize=(6.2, 4.8), sharey=True, sharex=True)
vlist = list(VARS); ypos = np.arange(len(vlist))[::-1]; OFF = {"crude": 0.14, "leader_adjusted": -0.14}
DASH = (0, (2, 1.2))
YLAB = {"Off_Centered": "Off-centred\nfollowing", "Occupied": "Occupied\n(passengers)", "Leader_speed_5kmh": "Leader speed\n(+5 km/h)"}
for ax, case, tag in zip(axes.ravel(), CASES, "abcd"):
    ax.axvline(1, color="0.4", lw=.7)
    for model in MODELS:
        g = res[(res.Site == case[0]) & (res.Class == case[1]) & (res.Model == model)].set_index("Variable").loc[vlist]
        for y, v in zip(ypos + OFF[model], vlist):
            adj = model == "leader_adjusted"
            ax.plot([g.loc[v, "TR_lo95"], g.loc[v, "TR_hi95"]], [y, y], color=VAR_COL[v], lw=1.3, ls="-" if adj else DASH)
            ax.plot(g.loc[v, "TR"], y, "o", ms=4, color=VAR_COL[v], mfc=VAR_COL[v] if adj else "white", mew=1.1)
    ax.set_title(f"({tag}) {case[0]}, {case[1]} (n={len(data[case][0])})", loc="left", fontsize=8.5)
    ax.set_xscale("log"); ax.set_xlim(0.33, 1.7); ax.set_xticks([0.5, 0.7, 1, 1.4]); ax.set_xticklabels(["0.5", "0.7", "1", "1.4"])
    ax.minorticks_off(); ax.grid(axis="x", alpha=.25, lw=.5)
for ax in axes[:, 0]:
    ax.set_yticks(ypos); ax.set_yticklabels([YLAB[v] for v in vlist], fontsize=8.5)
hand = [Line2D([], [], color="0.25", ls=DASH, marker="o", mfc="white", ms=4, label="Crude (variable alone)"),
        Line2D([], [], color="0.25", ls="-", marker="o", ms=4, label="Leader-adjusted")]
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.text(0.55, 0.065, "Time ratio, truncated Weibull AFT (95% CI, log scale)", ha="center", fontsize=8.5)
fig.legend(handles=hand, frameon=False, fontsize=7.5, ncol=2, loc="lower center", bbox_to_anchor=(0.55, -0.01))
save_fig(fig, "covariate_time_ratios"); plt.show()

### Export

In [ ]:
out = TAB / "03_covariate_time_ratios.xlsx"
with pd.ExcelWriter(out) as xw:
    side.to_excel(xw, sheet_name="crude_vs_adjusted", index=False)
    res.to_excel(xw, sheet_name="time_ratios", index=False)
    desc.to_excel(xw, sheet_name="variables_by_case", index=False)
    leader_shift.to_excel(xw, sheet_name="leader_TR_shift", index=False)
    sens.to_excel(xw, sheet_name="family_sensitivity", index=False)
    pd.DataFrame({"Parameter": ["LOWER", "UPPER", "family", "reference_leader", "leader_speed_unit_kmh", "leader_speed_centring",
                                "sensitivity_families", "data04_sha256"],
                  "Value": [LOWER, UPPER, hd.LABELS[FAMILY], REF, SPEED_UNIT, "case mean", json.dumps(SENS_FAMILIES),
                            hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

# Part 3: copy of `3_mechanism_and_policy.ipynb`

Code copied unchanged except for the output folder (`outputs_TRA`) and the chart resolution (1000 dpi).

# Notebook 3 — Spacing–speed mechanism, policy scenarios and manuscript tables

Run from the `notebooks/` folder. Inputs come from `../data/raw/`; everything this notebook makes is written to `../outputs/`
(`data/`, `tables/`, `figures/`). Figures are PNG at 600 dpi in Cambria with the house colour scheme of `style.py`; tables are Excel in Cambria.
Run the notebooks in order: **1 → 2 → 3**.

Needs notebooks 1 and 2.

| Step | Content |
|---|---|
| 4 | PR–BTW headway contrast split exactly into spacing and speed parts |
| 5 | E-rickshaw speed limits (S.R.O. No. 384-Law/2026): 15 km/h zone limit and 25 km/h general limit; exposure to short headways |
| 6 | Dedicated BTW lane at 25 km/h |
| 7 | Sensitivity of the main results to the 0.5–5 s extraction window |
| Tables | All manuscript and appendix tables in one workbook, `outputs/Manuscript_Tables.xlsx` |

## Step 4 — Spacing–speed decomposition of the PR–BTW headway contrast (by site)

**Question:** within each site, PR followers keep longer time headways than BTW followers. Is that because they keep a larger **gap in space**, or because they are **slower** and need more time to cover a similar gap?

**Identity:** for every event, the spacing is $S_i = H_i\,v_i$, where $v_i$ is the follower's space-mean speed (m/s). $S_i$ is the distance the follower covers in its headway (front-to-front space headway at the follower's speed). Then exactly,

$$\log H_i = \log S_i - \log v_i .$$

**Contrast at the same leader mix:** PR and BTW followers meet different leader mixes. Within each site, the mean log of each quantity is computed per target class and leader class, then averaged over leader classes with one common set of weights, the site's pooled leader shares (direct standardisation). Because means are linear, the standardised contrast splits **exactly**:

$$\underbrace{\Delta\overline{\log H}}_{\text{headway}} = \underbrace{\Delta\overline{\log S}}_{\text{spacing part}} \;+\; \underbrace{\left(-\Delta\overline{\log v}\right)}_{\text{speed part}},\qquad \Delta = \text{PR} - \text{BTW}.$$

Exponentiated, these are ratios of geometric means: headway ratio = spacing ratio × (BTW speed ÷ PR speed). The **speed share** is $-\Delta\overline{\log v}\,/\,\Delta\overline{\log H}$.

**Uncertainty:** 95% percentile intervals from a stratified bootstrap (resampling events within site × target class × leader class cells), B = 4,000.

**Checks:**
- The identity holds to machine precision for every event.
- A regression version, OLS of each log quantity on target class plus leader dummies, gives the same exact split (shared design matrix).
- Per-leader contrasts show whether the split is the same behind every leader class.

**Scope:** the sites are never merged. Every quantity is conditional on following within the 0.5–5 s extraction window, as in steps 1–3.

**Outputs:** `outputs/tables/04_spacing_speed_decomposition.xlsx`, and figures `decomposition_contrast.png`, `spacing_speed_scatter.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
SITES = ["Shahjahanpur", "Tikatuli"]
TARGETS = ["PR", "BTW"]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
B = 4000; SEED = 20261107
QTY = {"logH": "Headway", "logS": "Spacing part", "neg_logv": "Speed part"}
QCOL = PAL["component"]
TCOL = PAL["target"]

### Data and the identity

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
d["v_ms"] = d.Target_Speed_kmh / 3.6
d["S_m"] = d.Time_Headway_s * d.v_ms
d["logH"], d["logS"], d["logv"] = np.log(d.Time_Headway_s), np.log(d.S_m), np.log(d.v_ms)
d["neg_logv"] = -d.logv
err = np.abs(d.logH - (d.logS - d.logv)).max(); print("max |log H - (log S - log v)| =", err); assert err < 1e-12
desc = (d.groupby(["Site", "Target_Class"])
        .agg(n=("Obs_ID", "size"), gm_headway_s=("logH", lambda s: np.exp(s.mean())), gm_spacing_m=("logS", lambda s: np.exp(s.mean())),
             gm_speed_kmh=("logv", lambda s: 3.6 * np.exp(s.mean())), median_spacing_m=("S_m", "median"),
             p10_spacing_m=("S_m", lambda s: s.quantile(.1)), p90_spacing_m=("S_m", lambda s: s.quantile(.9)))
        .reindex(pd.MultiIndex.from_product([SITES, TARGETS])).reset_index().rename(columns={"level_0": "Site", "level_1": "Target_Class"}))
print(desc.round(3).to_string(index=False))

### Standardised contrast and its exact split

In [ ]:
def cell_means(g):
    return g.groupby(["Target_Class", "Leader_Class"])[list(QTY)].mean()

def standardised(g, w):
    m = cell_means(g)
    out = {}
    for t in TARGETS:
        mt = m.loc[t].reindex(LEADERS)
        out[t] = (mt.mul(w, axis=0)).sum()
    return out["PR"] - out["BTW"], out

rng = np.random.default_rng(SEED)
rows, boot_store = [], {}
for site in SITES:
    g = d[d.Site == site]
    w = g.Leader_Class.value_counts(normalize=True).reindex(LEADERS)
    assert (g.groupby(["Target_Class", "Leader_Class"]).size().unstack().reindex(columns=LEADERS) > 0).all().all()
    est, lev = standardised(g, w)
    # stratified bootstrap within target x leader cells; weights held at the site's observed leader shares
    idx = [np.flatnonzero(((g.Target_Class == t) & (g.Leader_Class == c)).to_numpy()) for t in TARGETS for c in LEADERS]
    vals = g[list(QTY)].to_numpy(); keys = [(t, c) for t in TARGETS for c in LEADERS]
    bs = np.empty((B, len(QTY)))
    for b in range(B):
        mean = {k: vals[rng.choice(ix, size=len(ix), replace=True)].mean(axis=0) for k, ix in zip(keys, idx)}
        pr = sum(w[c] * mean[("PR", c)] for c in LEADERS); bt = sum(w[c] * mean[("BTW", c)] for c in LEADERS)
        bs[b] = pr - bt
    boot_store[site] = bs
    share_b = bs[:, 2] / bs[:, 0]
    for j, q in enumerate(QTY):
        lo, hi = np.percentile(bs[:, j], [2.5, 97.5])
        rows.append({"Site": site, "Component": QTY[q], "key": q, "log_contrast": est[q], "log_lo95": lo, "log_hi95": hi,
                     "ratio_PR_over_BTW": np.exp(est[q]), "ratio_lo95": np.exp(lo), "ratio_hi95": np.exp(hi),
                     "pct": 100 * (np.exp(est[q]) - 1), "pct_lo95": 100 * (np.exp(lo) - 1), "pct_hi95": 100 * (np.exp(hi) - 1),
                     "share_of_headway_contrast": est[q] / est["logH"] if q != "logH" else 1.0,
                     "share_lo95": np.percentile(bs[:, j] / bs[:, 0], 2.5) if q != "logH" else np.nan,
                     "share_hi95": np.percentile(bs[:, j] / bs[:, 0], 97.5) if q != "logH" else np.nan})
    print(site, " exact split check:", round(est["logH"] - est["logS"] - est["neg_logv"], 15))
dec = pd.DataFrame(rows)
print(dec.drop(columns="key").round(4).to_string(index=False))

### Check: the regression version gives the same split

OLS of each log quantity on a PR indicator plus leader dummies, within site. The three regressions share one design matrix, so the PR coefficients add up exactly: $b_H = b_S + b_{-v}$. This contrast weights leaders by precision instead of by the site's shares, so it is close to, but not identical with, the standardised contrast.

In [ ]:
ols = []
for site in SITES:
    g = d[d.Site == site]
    X = np.column_stack([np.ones(len(g)), (g.Target_Class == "PR").to_numpy(float)] +
                        [(g.Leader_Class == c).to_numpy(float) for c in LEADERS[1:]])
    coef = {q: np.linalg.lstsq(X, g[q].to_numpy(), rcond=None)[0][1] for q in QTY}
    ols.append({"Site": site, **{f"b_{q}": v for q, v in coef.items()}, "split_error": coef["logH"] - coef["logS"] - coef["neg_logv"],
                "speed_share": coef["neg_logv"] / coef["logH"], "headway_ratio": np.exp(coef["logH"])})
ols = pd.DataFrame(ols); print(ols.round(5).to_string(index=False))
assert ols.split_error.abs().max() < 1e-10

### Per-leader contrasts

In [ ]:
pl = []
for site in SITES:
    m = cell_means(d[d.Site == site]); n = d[d.Site == site].groupby(["Target_Class", "Leader_Class"]).size()
    for c in LEADERS:
        diff = m.loc[("PR", c)] - m.loc[("BTW", c)]
        pl.append({"Site": site, "Leader": c, "n_PR": int(n[("PR", c)]), "n_BTW": int(n[("BTW", c)]),
                   "headway_ratio": np.exp(diff.logH), "spacing_ratio": np.exp(diff.logS), "speed_ratio_BTW_over_PR": np.exp(diff.neg_logv),
                   "speed_share": diff.neg_logv / diff.logH})
pl = pd.DataFrame(pl); print(pl.round(3).to_string(index=False))

### Figure: the PR–BTW headway contrast split into spacing and speed

In [ ]:
# Waterfall on the log scale, where the split is additive: the speed part starts at 0, the spacing part starts where
# the speed part ends, and together they reach the headway bar. Tick labels are shown as % (PR relative to BTW).
fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.9), sharex=True)
Y = {"logH": 2, "logS": 1, "neg_logv": 0}
for ax, site, tag in zip(axes, SITES, "ab"):
    g = dec[dec.Site == site].set_index("key")
    sp = g.loc["neg_logv", "log_contrast"]
    start = {"logH": 0.0, "neg_logv": 0.0, "logS": sp}
    for q in QTY:
        v, lo, hi = g.loc[q, "log_contrast"], g.loc[q, "log_lo95"], g.loc[q, "log_hi95"]
        y, s0 = Y[q], start[q]
        ax.barh(y, v, left=s0, color=QCOL[q], alpha=.85, height=.55)
        ax.plot([s0 + lo, s0 + hi], [y, y], color="k", lw=1.1)
        ax.text(max(s0 + hi, s0 + v) + 0.02, y, f"{g.loc[q, 'pct']:+.0f}%", va="center", fontsize=8)
    ax.plot([sp, sp], [Y["neg_logv"] + .275, Y["logS"] - .275], color="0.35", lw=.8, ls=":")
    tot = g.loc["logH", "log_contrast"]
    ax.plot([tot, tot], [Y["logS"] + .275, Y["logH"] - .275], color="0.35", lw=.8, ls=":")
    share = g.loc["neg_logv", "share_of_headway_contrast"]
    ax.set_yticks([2, 1, 0])
    ax.set_yticklabels(["Headway\n(PR ÷ BTW)", "Spacing part", "Speed part\n(BTW ÷ PR speed)"] if tag == "a" else [])
    ax.axvline(0, color="0.4", lw=.7); ax.grid(axis="x", alpha=.25, lw=.5)
    ax.set_title(f"({tag}) {site}\nspeed part = {100 * share:.0f}% of the log headway contrast", loc="left", fontsize=8.5)
PCT = [0, 20, 40, 60, 80]
ax.set_xticks(np.log1p(np.array(PCT) / 100)); ax.set_xticklabels([f"+{p}%" if p else "0" for p in PCT])
ax.set_xlim(np.log(0.92), np.log(2.2))
fig.supxlabel("PR relative to BTW, same leader mix (log scale; segments add up; geometric means, 95% bootstrap CI)", fontsize=8.5)
fig.tight_layout(); save_fig(fig, "decomposition_contrast"); plt.show()

### Figure: spacing against speed

Each point is one following event. The grey lines are constant headway ($S = H\,v$ for H = 1, 2, 3, 4 s). If PR and BTW followers kept the same spacing at all speeds, the two clouds would spread along the same horizontal band, and the headway difference would come from where they sit on the speed axis.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=True)
VMAX, SMAX = 38, 30
vgrid = np.linspace(3, VMAX, 100)
for ax, site, tag in zip(axes, SITES, "ab"):
    for h in (1, 2, 3, 4):
        ax.plot(vgrid, h * vgrid / 3.6, color="0.75", lw=.7, zorder=0)
        v_end = min(VMAX, SMAX * 3.6 / h)                        # where the line leaves the plotting area
        ax.text(v_end - 0.4, h * v_end / 3.6 - 0.4, f"H = {h} s", fontsize=6.5, color="0.45", va="top", ha="right")
    for t in TARGETS:
        g = d[(d.Site == site) & (d.Target_Class == t)]
        ax.scatter(g.Target_Speed_kmh, g.S_m, s=5, alpha=.45, color=TCOL[t], lw=0, label=f"{t} (n={len(g)})")
        gm_v, gm_s = 3.6 * np.exp(g.logv.mean()), np.exp(g.logS.mean())
        ax.plot(gm_v, gm_s, marker="D", ms=6, color=TCOL[t], mec="k", mew=.8)
    ax.set_xlim(3, VMAX); ax.set_ylim(0, SMAX); ax.grid(alpha=.2, lw=.5)
    ax.set_title(f"({tag}) {site}", loc="left", fontsize=8.5); ax.set_xlabel("Follower speed (km/h)")
    ax.legend(frameon=False, fontsize=7, loc="upper left")
axes[0].set_ylabel("Spacing S = H · v (m)")
fig.tight_layout(); save_fig(fig, "spacing_speed_scatter"); plt.show()

### Export

In [ ]:
out = TAB / "04_spacing_speed_decomposition.xlsx"
with pd.ExcelWriter(out) as xw:
    dec.drop(columns="key").to_excel(xw, sheet_name="decomposition", index=False)
    desc.to_excel(xw, sheet_name="by_site_target", index=False)
    pl.to_excel(xw, sheet_name="per_leader", index=False)
    ols.to_excel(xw, sheet_name="regression_check", index=False)
    pd.DataFrame({"Parameter": ["spacing_definition", "speed", "standardisation_weights", "bootstrap", "B", "SEED", "data04_sha256"],
                  "Value": ["S = H x v_follower (m)", "follower space-mean speed over the segment (m/s)", "site pooled leader shares",
                            "stratified within site x target x leader", B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}
                 ).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 5 — E-rickshaw speed-limit scenarios (by site)

**Policy:** City Corporation Areas Three-Wheeler Slow-Speed Battery-Operated Rickshaw (E-Rickshaw) Operation Rules, 2026. S.R.O. No. 384-Law/2026, Bangladesh Gazette (extraordinary), 29 October 2026, Local Government Division.
- Rule 2(ঘ) limits the e-rickshaw by design to **25 km/h**.
- Rule 8(4) sets **15 km/h** near schools, hospitals, places of worship and crowded places.
- The rules apply to e-rickshaws (BTW) only, not to pedal rickshaws (PR).

**Scenarios:**
- **15 km/h (zone limit):** both study segments were crowded during data collection, so this is the limit that applies at both sites.
- **25 km/h (general limit):** the e-rickshaw design limit anywhere in a city corporation area.

**Sites (context for reading the results):** Shahjahanpur is signal-controlled and narrower, with prominent on-street parking. Tikatuli is a wider, single-lane segment without signal control and is the more crowded of the two.

**What a cap does to an observed BTW following event.** The cap only binds when the BTW was faster than the cap ($v > c$). What happens then depends on the leader:

| Case | Condition | Outcome under the cap |
|---|---|---|
| 1. Not affected | $v \le c$ | headway unchanged |
| 2. Following breaks up | $v > c$, leader is **not** an e-rickshaw and travels faster than the cap ($v_L > c$) | the leader keeps its speed and the capped BTW falls behind; the gap keeps opening, so there is no steady following headway |
| 3. Following continues at the capped speed | $v > c$, and the leader is a BTW (capped as well) or already at or below the cap ($v_L \le c$) | the BTW follows at $v' = c$ with the same spacing, $S' = S$, so $H' = H\,v/c$ |

Case 3 holds the gap in space constant (step 4: PR and BTW keep similar spacing, and the headway contrast is mostly speed). A case-3 event whose new headway exceeds 5 s also leaves the following window.

**PR followers** are not capped. A PR behind a BTW leader that was faster than the cap now has a slower leader. These events are counted as **indirectly affected**. No headway is modelled for them, because leader speed had no clear effect on PR headways (step 3).

**Leader-speed sensitivity (Shahjahanpur BTW):** BTW followers at Shahjahanpur keep longer headways behind faster leaders (leader-adjusted time ratio ≈ 1.09 per 5 km/h, step 3). For case-3 events behind a capped BTW leader, the headway is also multiplied by $\text{TR}^{(v_L'-v_L)/5}$.

**Reported per site × cap:**
- the shares of the three cases;
- for the BTW events that keep following (cases 1 and 3 within 5 s), the median, mean and $P(H<2\ \text{s})$, compared with the observed BTW and PR distributions.

**Two effects in the change for the events that keep following.** The change from the observed BTW distribution mixes:
- a **selection effect**: the events that break up are those behind fast leaders, mostly four-wheelers, which had longer headways;
- a **speed effect**: case-3 events now run at the capped speed.

Each change is split exactly (in log-median and in percentage points): *selection* compares the surviving events at their observed headways with all observed events; *speed* compares the surviving events after and before the cap.

These are counterfactual transformations of the observed events, not a traffic simulation.

**Uncertainty:** 95% percentile intervals from a bootstrap that resamples events within site × target × leader-class cells (B = 2,000).

**Outputs:** `outputs/tables/05_speed_cap_scenarios.xlsx`, and figures `speed_cap_outcome_shares.png`, `speed_cap_headway_ecdf.png`, `speed_cap_short_headway_exposure.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
SITES = ["Shahjahanpur", "Tikatuli"]
CAPS = {15: "15 km/h (zone limit, rule 8(4))", 25: "25 km/h (general limit, rule 2(ঘ))"}
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
UPPER = 5.0                        # s, upper edge of the extraction window
B = 2000; SEED = 20261107
CASES = {1: "Not affected (v ≤ cap)", 2: "Following breaks up (faster uncapped leader)", 3: "Follows at the capped speed"}
CASE_COL = PAL["outcome"]
CAP_COL = PAL["cap"]
_tr = pd.read_excel(TAB / "03_covariate_time_ratios.xlsx", sheet_name="time_ratios")
_row = _tr[(_tr.Site == "Shahjahanpur") & (_tr.Class == "BTW") & (_tr.Model == "leader_adjusted") & (_tr.Variable == "Leader_speed_5kmh")].iloc[0]
TR_LS = {"est": float(_row.TR), "lo": float(_row.TR_lo95), "hi": float(_row.TR_hi95)}
print("Shahjahanpur BTW leader-speed TR per 5 km/h:", {k: round(v, 4) for k, v in TR_LS.items()})

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
speed = d.groupby(["Site", "Target_Class"]).Target_Speed_kmh.describe(percentiles=[.1, .5, .9]).round(2).reset_index()
print(speed.to_string(index=False))

### Scenario engine

In [ ]:
def classify(g, cap):
    v, vl = g.Target_Speed_kmh.to_numpy(), g.Leading_Speed_kmh.to_numpy()
    lead_btw = (g.Leader_Class == "BTW").to_numpy()
    case = np.ones(len(g), int)
    binds = v > cap
    case[binds & ~lead_btw & (vl > cap)] = 2
    case[binds & (lead_btw | (vl <= cap))] = 3
    return case

def btw_scenario(g, cap, leader_tr=None):
    # returns the case of each event and the new headway (NaN where following breaks up)
    H, v, vl = g.Time_Headway_s.to_numpy(), g.Target_Speed_kmh.to_numpy(), g.Leading_Speed_kmh.to_numpy()
    case = classify(g, cap)
    Hn = H.copy().astype(float)
    m3 = case == 3
    Hn[m3] = H[m3] * v[m3] / cap
    if leader_tr is not None:
        lb = m3 & (g.Leader_Class == "BTW").to_numpy() & (vl > cap)
        Hn[lb] = Hn[lb] * leader_tr ** ((cap - vl[lb]) / 5.0)
    Hn[case == 2] = np.nan
    return case, Hn

def summarise(case, Hn, H0):
    keep = np.isfinite(Hn) & (Hn <= UPPER)
    h = Hn[keep]
    h_sel = H0[keep]                                       # same surviving events, headways as observed
    sel = {"median_selection_only_s": np.median(h_sel), "P(H<2)_selection_only": (h_sel < 2).mean(),
           "d_median_pct_selection": 100 * (np.median(h_sel) / np.median(H0) - 1),
           "d_median_pct_speed": 100 * (np.median(h) / np.median(h_sel) - 1),
           "d_P(H<2)_pp_selection": 100 * ((h_sel < 2).mean() - (H0 < 2).mean()),
           "d_P(H<2)_pp_speed": 100 * ((h < 2).mean() - (h_sel < 2).mean())}
    return {**sel, **exposure(case, Hn, H0), "share_case1": (case == 1).mean(), "share_case2": (case == 2).mean(), "share_case3": (case == 3).mean(),
            "share_case3_above_5s": ((case == 3) & (Hn > UPPER)).mean(), "share_still_following": keep.mean(),
            "n_still_following": int(keep.sum()), "median_s": np.median(h), "mean_s": h.mean(), "P(H<2)": (h < 2).mean(),
            "d_median_pct": 100 * (np.median(h) / np.median(H0) - 1), "d_P(H<2)_pp": 100 * ((h < 2).mean() - (H0 < 2).mean())}

THRESH = [2.0, 1.5]                # s, short-headway thresholds for exposure

def exposure(case, Hn, H0):
    # exposure to short headways, per observed BTW event and among the events still following.
    # Accounting per threshold t (counts per 100 observed events):
    #   observed short = removed by break-up + lifted above t by the slower speed + still short
    N = len(H0); keep = np.isfinite(Hn) & (Hn <= UPPER); out = {}
    for t in THRESH:
        k = f"{t:g}"
        obs_short = H0 < t
        still = keep & (Hn < t)
        removed = (case == 2) & obs_short
        lifted = obs_short & ~still & ~removed
        out.update({f"exp{k}_obs_per100": 100 * obs_short.mean(), f"exp{k}_after_per100": 100 * still.mean(),
                    f"exp{k}_removed_breakup_per100": 100 * removed.mean(), f"exp{k}_lifted_speed_per100": 100 * lifted.mean(),
                    f"exp{k}_change_pct": 100 * (still.sum() / max(obs_short.sum(), 1) - 1),
                    f"exp{k}_share_among_following": still.sum() / max(keep.sum(), 1)})
    return out

def pr_indirect(p, cap):
    affected = (p.Leader_Class == "BTW").to_numpy() & (p.Leading_Speed_kmh.to_numpy() > cap)
    return {"PR_share_behind_capped_BTW_leader": affected.mean(),
            "PR_share_behind_capped_BTW_leader_and_PR_faster_than_cap": (affected & (p.Target_Speed_kmh.to_numpy() > cap)).mean()}

def strat_resample(g, rng):
    parts = [g[g.Leader_Class == c] for c in LEADERS]
    return pd.concat([q.iloc[rng.integers(0, len(q), len(q))] for q in parts], ignore_index=True)

### Scenarios with bootstrap intervals

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
for s in SITES:
    g = d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True)
    p = d[(d.Site == s) & (d.Target_Class == "PR")].reset_index(drop=True)
    est = {}
    for cap in CAPS:
        case, Hn = btw_scenario(g, cap)
        est[cap] = {**summarise(case, Hn, g.Time_Headway_s.to_numpy()), **pr_indirect(p, cap)}
    boot = {cap: [] for cap in CAPS}
    for _ in range(B):
        gb, pb = strat_resample(g, rng), strat_resample(p, rng)
        for cap in CAPS:
            case, Hn = btw_scenario(gb, cap)
            boot[cap].append({**summarise(case, Hn, gb.Time_Headway_s.to_numpy()), **pr_indirect(pb, cap)})
    obs = g.Time_Headway_s.to_numpy(); pr_h = p.Time_Headway_s.to_numpy()
    rows.append({"Site": s, "Cap_kmh": "observed", "Scenario": "observed (no cap)", "share_case1": 1.0, "share_case2": 0.0, "share_case3": 0.0,
                 "share_case3_above_5s": 0.0, "share_still_following": 1.0, "n_still_following": len(obs), "median_s": np.median(obs),
                 "mean_s": obs.mean(), "P(H<2)": (obs < 2).mean(), "d_median_pct": 0.0, "d_P(H<2)_pp": 0.0,
                 "PR_median_s": np.median(pr_h), "PR_P(H<2)": (pr_h < 2).mean(), "n_BTW": len(g), "n_PR": len(p),
                 **exposure(np.ones(len(obs), int), obs.astype(float), obs)})
    for cap in CAPS:
        bb = pd.DataFrame(boot[cap]); r = {"Site": s, "Cap_kmh": cap, "Scenario": CAPS[cap], **est[cap]}
        for k in ["share_case1", "share_case2", "share_case3", "share_still_following", "median_s", "P(H<2)", "d_median_pct", "d_P(H<2)_pp",
                  "d_median_pct_selection", "d_median_pct_speed", "d_P(H<2)_pp_selection", "d_P(H<2)_pp_speed",
                  "PR_share_behind_capped_BTW_leader"] + \
                 [f"exp{t:g}_{m}" for t in THRESH for m in ("after_per100", "removed_breakup_per100", "lifted_speed_per100",
                                                             "change_pct", "share_among_following")]:
            r[k + "_lo95"], r[k + "_hi95"] = np.percentile(bb[k], [2.5, 97.5])
        r.update({"PR_median_s": np.median(pr_h), "PR_P(H<2)": (pr_h < 2).mean(), "n_BTW": len(g), "n_PR": len(p)})
        rows.append(r)
scen = pd.DataFrame(rows)
show = ["Site", "Cap_kmh", "share_case1", "share_case2", "share_case2_lo95", "share_case2_hi95", "share_case3", "share_case3_above_5s",
        "share_still_following", "median_s", "d_median_pct", "d_median_pct_lo95", "d_median_pct_hi95", "P(H<2)", "d_P(H<2)_pp",
        "d_P(H<2)_pp_lo95", "d_P(H<2)_pp_hi95", "PR_median_s", "PR_P(H<2)", "PR_share_behind_capped_BTW_leader"]
print(scen.reindex(columns=show).round(3).to_string(index=False))
print()
split = ["Site", "Cap_kmh", "d_median_pct", "d_median_pct_selection", "d_median_pct_speed", "d_median_pct_speed_lo95", "d_median_pct_speed_hi95",
         "d_P(H<2)_pp", "d_P(H<2)_pp_selection", "d_P(H<2)_pp_speed", "d_P(H<2)_pp_speed_lo95", "d_P(H<2)_pp_speed_hi95"]
print(scen[scen.Cap_kmh != "observed"].reindex(columns=split).round(3).to_string(index=False))

### Exposure to short headways (< 2 s and < 1.5 s)

Exposure is counted **per 100 observed BTW following events**, so the scenarios share one denominator. An event whose following breaks up no longer runs at a short headway behind that leader and is not counted. Each observed short-headway event ends in one of three states, and these add up exactly:

*observed short = removed by break-up + lifted above the threshold by the slower speed + still short.*

The share **among the events still following** is also given; it answers a different question (how close the remaining followers run). The PR benchmark is the observed PR share.

In [ ]:
pr_exp = {s: {t: 100 * (d[(d.Site == s) & (d.Target_Class == "PR")].Time_Headway_s < t).mean() for t in THRESH} for s in SITES}
ex_rows = []
for _, r in scen.iterrows():
    for t in THRESH:
        k = f"{t:g}"
        row = {"Site": r.Site, "Cap_kmh": r.Cap_kmh, "Threshold_s": t, "observed_per100": r[f"exp{k}_obs_per100"],
               "after_per100": r[f"exp{k}_after_per100"], "removed_by_breakup_per100": r[f"exp{k}_removed_breakup_per100"],
               "lifted_by_speed_per100": r[f"exp{k}_lifted_speed_per100"], "change_in_exposed_events_pct": r[f"exp{k}_change_pct"],
               "share_among_still_following": r[f"exp{k}_share_among_following"], "PR_observed_per100": pr_exp[r.Site][t]}
        for m, col in [("after_per100", "after_per100"), ("change_pct", "change_in_exposed_events_pct"),
                       ("share_among_following", "share_among_still_following"),
                       ("removed_breakup_per100", "removed_by_breakup_per100"), ("lifted_speed_per100", "lifted_by_speed_per100")]:
            row[col + "_lo95"] = r.get(f"exp{k}_{m}_lo95", np.nan); row[col + "_hi95"] = r.get(f"exp{k}_{m}_hi95", np.nan)
        ex_rows.append(row)
expo = pd.DataFrame(ex_rows)
assert np.allclose(expo.observed_per100, expo.after_per100 + expo.removed_by_breakup_per100 + expo.lifted_by_speed_per100)
print(expo[["Site", "Cap_kmh", "Threshold_s", "observed_per100", "after_per100", "after_per100_lo95", "after_per100_hi95",
            "removed_by_breakup_per100", "lifted_by_speed_per100", "change_in_exposed_events_pct", "share_among_still_following",
            "share_among_still_following_lo95", "share_among_still_following_hi95", "PR_observed_per100"]].round(3).to_string(index=False))

### Figure: exposure to short headways per 100 observed BTW following events

In [ ]:
SEG = [("after_per100", "Still at a short headway", PAL["exposure"]["still"]), ("lifted_by_speed_per100", "Lifted above the threshold by the slower speed", PAL["exposure"]["lifted_speed"]),
       ("removed_by_breakup_per100", "Removed: following breaks up", PAL["exposure"]["removed"])]
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.4), sharey=True)
xlab = ["Obs.", "25", "15"]
for ax, s, tag in zip(axes, SITES, "ab"):
    for j, t in enumerate(THRESH):
        sub = expo[(expo.Site == s) & (expo.Threshold_s == t)].set_index("Cap_kmh").loc[["observed", 25, 15]]
        x0 = j * 4.0
        for i, (cap, r) in enumerate(sub.iterrows()):
            bottom = 0.0
            for col, lab, colr in SEG:
                ax.bar(x0 + i, r[col], bottom=bottom, color=colr, width=.75, label=lab if (j == 0 and i == 0) or (col != "after_per100" and j == 0 and i == 1) else None)
                bottom += r[col]
            ax.text(x0 + i, 2.5, f"{r.after_per100:.0f}", ha="center", va="bottom", fontsize=7, color="white")
        ax.text(x0 + 1, 71, f"H < {t:g} s", ha="center", va="center", fontsize=8.5)
    ax.set_xticks([0, 1, 2, 4, 5, 6]); ax.set_xticklabels(xlab * 2, fontsize=7.5)
    ax.set_ylim(0, 75); ax.grid(axis="y", alpha=.2, lw=.5); ax.set_xlabel("BTW speed cap (km/h)", fontsize=8.5)
    ax.set_title(f"({tag}) {s}", loc="left", fontsize=8.5)
axes[0].set_ylabel("Events per 100 observed BTW events")
h, l = axes[0].get_legend_handles_labels(); seen = {}
for hh, ll in zip(h, l): seen.setdefault(ll, hh)
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(seen.values(), seen.keys(), frameon=False, fontsize=7.2, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.02)); save_fig(fig, "speed_cap_short_headway_exposure"); plt.show()

### Which leaders break up the following (case 2), by leader class

In [ ]:
bl = []
for s in SITES:
    g = d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True)
    for cap in CAPS:
        case = classify(g, cap)
        for c in LEADERS:
            m = (g.Leader_Class == c).to_numpy()
            bl.append({"Site": s, "Cap_kmh": cap, "Leader": c, "n": int(m.sum()), "share_case1": (case[m] == 1).mean(),
                       "share_case2": (case[m] == 2).mean(), "share_case3": (case[m] == 3).mean()})
bl = pd.DataFrame(bl); print(bl.round(3).to_string(index=False))

### Sensitivity: Shahjahanpur BTW, capped BTW leaders shorten the headway behind them

In [ ]:
g = d[(d.Site == "Shahjahanpur") & (d.Target_Class == "BTW")].reset_index(drop=True)
sens = []
for cap in CAPS:
    case, H0 = btw_scenario(g, cap); base = summarise(case, H0, g.Time_Headway_s.to_numpy())
    row = {"Cap_kmh": cap, "median_s_no_leader_effect": base["median_s"], "P(H<2)_no_leader_effect": base["P(H<2)"],
           "exp2_per100_no_leader_effect": base["exp2_after_per100"], "exp1.5_per100_no_leader_effect": base["exp1.5_after_per100"],
           "share_case3_behind_capped_BTW_leader": ((case == 3) & (g.Leader_Class == "BTW").to_numpy() & (g.Leading_Speed_kmh.to_numpy() > cap)).mean()}
    for k in ("est", "lo", "hi"):
        c1, H1 = btw_scenario(g, cap, leader_tr=TR_LS[k]); s1 = summarise(c1, H1, g.Time_Headway_s.to_numpy())
        row[f"median_s_TR_{k}"], row[f"P(H<2)_TR_{k}"] = s1["median_s"], s1["P(H<2)"]
        row[f"exp2_per100_TR_{k}"], row[f"exp1.5_per100_TR_{k}"] = s1["exp2_after_per100"], s1["exp1.5_after_per100"]
    sens.append(row)
sens = pd.DataFrame(sens); print(sens.round(3).to_string(index=False))

### Figure: what the cap does to the observed BTW following events

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 2.6))
labels, y = [], 0
for s in SITES:
    for cap in [25, 15]:
        r = scen[(scen.Site == s) & (scen.Cap_kmh == cap)].iloc[0]; left = 0.0
        for k in (1, 3, 2):
            w = 100 * r[f"share_case{k}"]
            ax.barh(y, w, left=left, color=CASE_COL[k], height=.62, label=CASES[k] if y == 0 else None, edgecolor="white", lw=.5)
            if w >= 6:
                ax.text(left + w / 2, y, f"{w:.0f}%", ha="center", va="center", fontsize=7.5, color="white" if k != 1 else PAL["ink"])
            left += w
        labels.append(f"{s}, {cap} km/h"); y += 1
    y += 0.4
ys = [0, 1, 2.4, 3.4]
ax.set_yticks(ys); ax.set_yticklabels(labels); ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel("Share of observed BTW following events (%)")
ax.legend(frameon=False, fontsize=7.5, ncol=3, loc="upper center", bbox_to_anchor=(0.45, -0.28))
for sp in ("top", "right"): ax.spines[sp].set_visible(False)
fig.tight_layout(); save_fig(fig, "speed_cap_outcome_shares"); plt.show()

### Figure: headways of the BTW events that keep following, against observed BTW and PR

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)
for ax, s, tag in zip(axes, SITES, "ab"):
    g = d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True)
    p = np.sort(d[(d.Site == s) & (d.Target_Class == "PR")].Time_Headway_s.to_numpy())
    curves = [("observed", g.Time_Headway_s.to_numpy(), "BTW observed")]
    for cap in [25, 15]:
        case, Hn = btw_scenario(g, cap); h = Hn[np.isfinite(Hn) & (Hn <= UPPER)]
        curves.append((cap, h, f"BTW still following, cap {cap} km/h (n={len(h)})"))
    for key, h, lab in curves:
        x = np.sort(h); ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=CAP_COL[key], lw=1.5, label=lab)
    ax.step(p, np.arange(1, len(p) + 1) / len(p), where="post", color=PAL["target"]["PR"], lw=1.4, ls="--", label="PR observed")
    ax.axvline(2, color="0.6", lw=.6, ls=":")
    ax.set_xlim(0.4, 5.1); ax.grid(alpha=.2, lw=.5); ax.set_xlabel("Time headway (s)")
    ax.set_title(f"({tag}) {s}", loc="left", fontsize=8.5); ax.legend(frameon=False, fontsize=6.6, loc="lower right")
axes[0].set_ylabel("Cumulative share")
fig.tight_layout(); save_fig(fig, "speed_cap_headway_ecdf"); plt.show()

### Export

In [ ]:
out = TAB / "05_speed_cap_scenarios.xlsx"
with pd.ExcelWriter(out) as xw:
    scen.assign(Cap_kmh=scen.Cap_kmh.astype(str)).to_excel(xw, sheet_name="scenarios", index=False)
    expo.assign(Cap_kmh=expo.Cap_kmh.astype(str)).to_excel(xw, sheet_name="short_headway_exposure", index=False)
    bl.to_excel(xw, sheet_name="cases_by_leader", index=False)
    sens.to_excel(xw, sheet_name="SP_BTW_leader_speed_sens", index=False)
    speed.to_excel(xw, sheet_name="observed_speeds", index=False)
    pd.DataFrame({"Parameter": ["policy", "caps_kmh", "cap_applies_to", "case_1", "case_2", "case_3", "following_window_upper_s",
                                "leader_speed_TR_SP_BTW", "bootstrap", "B", "SEED", "data04_sha256"],
                  "Value": ["S.R.O. No. 384-Law/2026, Bangladesh Gazette (extraordinary), 29 Oct 2026: rule 2(gha) 25 km/h design limit; rule 8(4) 15 km/h near schools, hospitals, places of worship and crowded places",
                            json.dumps(list(CAPS)), "BTW (e-rickshaw) only; PR not capped", "v <= cap: unchanged",
                            "v > cap and non-BTW leader with v_L > cap: following breaks up", "v > cap and (BTW leader or v_L <= cap): H' = H v / cap (spacing constant)",
                            UPPER, json.dumps({k: round(v, 4) for k, v in TR_LS.items()}), "events resampled within site x target x leader",
                            B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 6 — Dedicated e-rickshaw (BTW) lane scenario (by site)

**Scenario:** a lane reserved for e-rickshaws. Only BTWs use it, so:
- every leader is a BTW;
- no PR or other vehicle shares the lane;
- the general e-rickshaw limit of **25 km/h** applies (rule 2(ঘ), S.R.O. No. 384-Law/2026);
- following is centred, because there is no mixed traffic to stagger against.

**Basis in the data:** the observed BTW-follows-BTW events stand in for following inside such a lane. Two versions are reported side by side:

| Version | Events used | n (Shahjahanpur / Tikatuli) | Role |
|---|---|---|---|
| A. Dedicated lane | all BTW-led BTW events | 58 / 69 | main result; centring is part of the lane design and is not modelled |
| B. Dedicated lane, centred only | BTW-led BTW events that were centred | 10 / 37 | check; matches the scenario exactly, but Shahjahanpur rests on 10 events |

No off-centring effect is estimated or applied in either version.

**Cap:** both the follower and its BTW leader are held to 25 km/h, so following never breaks up. A follower faster than the cap runs at $v' = 25$ km/h with the same spacing, $H' = H\,v/25$ (as in step 5). An event pushed above 5 s leaves the following window.

**Compared with mixed traffic:** all observed BTW following events at the site (no cap), and mixed traffic under the same 25 km/h cap (step 5 logic: an event behind a faster uncapped leader breaks up and leaves the denominator).

**Reported:** median headway, and exposure to short headways ($H < 2$ s and $H < 1.5$ s) as the share of following events. For the lane every event keeps following, so this share is also exposure per 100 lane events.

**Leader-speed sensitivity (Shahjahanpur):** BTW followers there keep longer headways behind faster leaders (leader-adjusted time ratio 1.089 per 5 km/h, step 3). Capping BTW leaders at 25 km/h would therefore shorten the headways behind them; this is shown as a sensitivity check.

**Uncertainty:** 95% percentile intervals from a bootstrap that resamples BTW events within site × leader-class cells (B = 2,000) and takes the lane subset from each replicate, so lane and mixed traffic are compared on the same resample.

**Caution:** behaviour observed behind BTW leaders in mixed traffic is used as a proxy for behaviour inside a dedicated lane. A real lane may change speeds and spacing in ways these data cannot show.

**Outputs:** `outputs/tables/06_dedicated_btw_lane.xlsx`, and figures `dedicated_lane_exposure.png`, `dedicated_lane_headway_ecdf.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
SITES = ["Shahjahanpur", "Tikatuli"]
CAP = 25                           # km/h, general e-rickshaw limit (rule 2(gha))
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
THRESH = [2.0, 1.5]
UPPER = 5.0
B = 2000; SEED = 20261107
SCEN = {"mixed_observed": "Mixed traffic, observed", "mixed_cap25": "Mixed traffic, 25 km/h cap",
        "lane_A": "Dedicated lane, 25 km/h (BTW-led)", "lane_B": "Dedicated lane, 25 km/h (BTW-led, centred only)"}
SCOL = PAL["lane"]
_tr = pd.read_excel(TAB / "03_covariate_time_ratios.xlsx", sheet_name="time_ratios")
_row = _tr[(_tr.Site == "Shahjahanpur") & (_tr.Class == "BTW") & (_tr.Model == "leader_adjusted") & (_tr.Variable == "Leader_speed_5kmh")].iloc[0]
TR_LS = {"est": float(_row.TR), "lo": float(_row.TR_lo95), "hi": float(_row.TR_hi95)}
print("Shahjahanpur BTW leader-speed TR per 5 km/h:", {k: round(v, 4) for k, v in TR_LS.items()})

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
btw = {s: d[(d.Site == s) & (d.Target_Class == "BTW")].reset_index(drop=True) for s in SITES}
counts = []
for s in SITES:
    g = btw[s]; lead = g[g.Leader_Class == "BTW"]
    counts.append({"Site": s, "n_BTW_all": len(g), "n_BTW_led": len(lead), "n_BTW_led_centred": int((~lead.Off_Centered).sum()),
                   "share_follower_above_cap": (lead.Target_Speed_kmh > CAP).mean(), "share_leader_above_cap": (lead.Leading_Speed_kmh > CAP).mean(),
                   "mean_follower_speed_kmh": lead.Target_Speed_kmh.mean(), "mean_leader_speed_kmh": lead.Leading_Speed_kmh.mean()})
counts = pd.DataFrame(counts); print(counts.round(3).to_string(index=False))

### Scenario engine

In [ ]:
def capped(g, leader_tr=None):
    # both follower and BTW leader capped; spacing constant; returns new headways (NaN where following breaks up)
    H, v, vl = g.Time_Headway_s.to_numpy(float), g.Target_Speed_kmh.to_numpy(), g.Leading_Speed_kmh.to_numpy()
    lead_btw = (g.Leader_Class == "BTW").to_numpy()
    Hn = np.where(v > CAP, H * v / CAP, H)
    Hn = np.where((v > CAP) & ~lead_btw & (vl > CAP), np.nan, Hn)          # mixed traffic: faster uncapped leader, following breaks up
    if leader_tr is not None:
        lb = lead_btw & (vl > CAP)
        Hn = np.where(lb, Hn * leader_tr ** ((CAP - vl) / 5.0), Hn)
    return Hn

def summ(h):
    h = h[np.isfinite(h) & (h <= UPPER)]
    out = {"n_following": len(h), "median_s": np.median(h), "mean_s": h.mean()}
    for t in THRESH:
        out[f"share_H<{t:g}"] = (h < t).mean()
    return out

def scenarios(g, leader_tr=None):
    lead = (g.Leader_Class == "BTW").to_numpy(); cen = ~g.Off_Centered.to_numpy().astype(bool)
    Hc = capped(g, leader_tr)
    return {"mixed_observed": summ(g.Time_Headway_s.to_numpy(float)), "mixed_cap25": summ(Hc),
            "lane_A": summ(Hc[lead]), "lane_B": summ(Hc[lead & cen])}

def strat_resample(g, rng):
    parts = [g[g.Leader_Class == c] for c in LEADERS]
    return pd.concat([q.iloc[rng.integers(0, len(q), len(q))] for q in parts], ignore_index=True)

### Results with bootstrap intervals

In [ ]:
rng = np.random.default_rng(SEED)
rows = []
METRICS = ["median_s", "mean_s"] + [f"share_H<{t:g}" for t in THRESH]
for s in SITES:
    g = btw[s]; est = scenarios(g)
    boot = {k: [] for k in SCEN}; dboot = {k: [] for k in SCEN}
    for _ in range(B):
        sb = scenarios(strat_resample(g, rng))
        for k in SCEN:
            boot[k].append(sb[k]); dboot[k].append({m: sb[k][m] - sb["mixed_observed"][m] for m in METRICS})
    for k, lab in SCEN.items():
        bb, db = pd.DataFrame(boot[k]), pd.DataFrame(dboot[k])
        r = {"Site": s, "Scenario": lab, "key": k, **est[k]}
        for m in METRICS:
            r[m + "_lo95"], r[m + "_hi95"] = np.percentile(bb[m], [2.5, 97.5])
            r["d_" + m] = est[k][m] - est["mixed_observed"][m]
            r["d_" + m + "_lo95"], r["d_" + m + "_hi95"] = np.percentile(db[m], [2.5, 97.5])
        rows.append(r)
res = pd.DataFrame(rows)
show = ["Site", "Scenario", "n_following", "median_s", "median_s_lo95", "median_s_hi95", "share_H<2", "share_H<2_lo95", "share_H<2_hi95",
        "d_share_H<2", "d_share_H<2_lo95", "d_share_H<2_hi95", "share_H<1.5", "share_H<1.5_lo95", "share_H<1.5_hi95",
        "d_share_H<1.5", "d_share_H<1.5_lo95", "d_share_H<1.5_hi95"]
print(res[show].round(3).to_string(index=False))

### Sensitivity: Shahjahanpur, capped BTW leaders shorten the headways behind them

In [ ]:
g = btw["Shahjahanpur"]; sens = []
for k in ("est", "lo", "hi"):
    sc = scenarios(g, leader_tr=TR_LS[k])
    for key in ("lane_A", "lane_B"):
        sens.append({"Site": "Shahjahanpur", "TR_leader_speed": k, "TR_value": TR_LS[k], "Scenario": SCEN[key], **sc[key]})
base = scenarios(g)
for key in ("lane_A", "lane_B"):
    sens.append({"Site": "Shahjahanpur", "TR_leader_speed": "none", "TR_value": 1.0, "Scenario": SCEN[key], **base[key]})
sens = pd.DataFrame(sens); print(sens.round(3).to_string(index=False))

### Figure: exposure to short headways, mixed traffic vs dedicated lane

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.3), sharey=True)
keys = list(SCEN); w = 0.2
for ax, s, tag in zip(axes, SITES, "ab"):
    sub = res[res.Site == s].set_index("key")
    for i, k in enumerate(keys):
        x = np.arange(len(THRESH)) + (i - 1.5) * w
        y = [100 * sub.loc[k, f"share_H<{t:g}"] for t in THRESH]
        lo = [100 * sub.loc[k, f"share_H<{t:g}_lo95"] for t in THRESH]; hi = [100 * sub.loc[k, f"share_H<{t:g}_hi95"] for t in THRESH]
        ax.bar(x, y, width=w, color=SCOL[k], label=f"{SCEN[k]}" if tag == "a" else None, edgecolor="white", lw=.4)
        ax.errorbar(x, y, yerr=[np.array(y) - lo, np.array(hi) - y], fmt="none", ecolor="k", elinewidth=.8, capsize=1.8)
        for xx, yy in zip(x, y):
            ax.text(xx, 2, f"{yy:.0f}", ha="center", va="bottom", fontsize=6.5, color="white")
    ax.set_xticks(range(len(THRESH))); ax.set_xticklabels([f"H < {t:g} s" for t in THRESH])
    ax.set_title(f"({tag}) {s}  (lane n = {int(sub.loc['lane_A', 'n_following'])}; centred n = {int(sub.loc['lane_B', 'n_following'])})",
                 loc="left", fontsize=8.5)
    ax.set_ylim(0, 100); ax.grid(axis="y", alpha=.2, lw=.5)
axes[0].set_ylabel("Share of following events (%)")
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(frameon=False, fontsize=7.2, ncol=2, loc="lower center", bbox_to_anchor=(0.5, -0.02))
save_fig(fig, "dedicated_lane_exposure"); plt.show()

### Figure: headway distribution, mixed traffic vs dedicated lane

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharey=True)
for ax, s, tag in zip(axes, SITES, "ab"):
    g = btw[s]; lead = (g.Leader_Class == "BTW").to_numpy(); cen = ~g.Off_Centered.to_numpy().astype(bool); Hc = capped(g)
    curves = {"mixed_observed": g.Time_Headway_s.to_numpy(float), "lane_A": Hc[lead], "lane_B": Hc[lead & cen]}
    for k, h in curves.items():
        x = np.sort(h[np.isfinite(h) & (h <= UPPER)])
        ax.step(x, np.arange(1, len(x) + 1) / len(x), where="post", color=SCOL[k],
                lw=1.5, ls="-" if k != "lane_B" else "--", label=f"{SCEN[k]} (n={len(x)})")
    ax.axvline(2, color="0.6", lw=.6, ls=":"); ax.axvline(1.5, color="0.6", lw=.6, ls=":")
    ax.set_xlim(0.4, 5.1); ax.grid(alpha=.2, lw=.5); ax.set_xlabel("BTW follower time headway (s)")
    ax.set_title(f"({tag}) {s}", loc="left", fontsize=8.5); ax.legend(frameon=False, fontsize=6.6, loc="lower right")
axes[0].set_ylabel("Cumulative share")
fig.tight_layout(); save_fig(fig, "dedicated_lane_headway_ecdf"); plt.show()

### Export

In [ ]:
out = TAB / "06_dedicated_btw_lane.xlsx"
with pd.ExcelWriter(out) as xw:
    res.drop(columns="key").to_excel(xw, sheet_name="scenarios", index=False)
    counts.to_excel(xw, sheet_name="lane_events", index=False)
    sens.to_excel(xw, sheet_name="SP_leader_speed_sens", index=False)
    pd.DataFrame({"Parameter": ["policy", "cap_kmh", "lane_A", "lane_B", "headway_under_cap", "mixed_cap25", "thresholds_s",
                                "following_window_upper_s", "leader_speed_TR_SP_BTW", "bootstrap", "B", "SEED", "data04_sha256"],
                  "Value": ["S.R.O. No. 384-Law/2026, rule 2(gha): 25 km/h e-rickshaw limit", CAP,
                            "BTW followers behind BTW leaders, all lateral positions (main)", "BTW followers behind BTW leaders, centred only (check)",
                            "H' = H v / 25 for v > 25 (spacing constant); leader also capped, so no break-up in the lane",
                            "notebook 05 logic: v > 25 behind a faster uncapped leader breaks up", json.dumps(THRESH), UPPER,
                            json.dumps({k: round(v, 4) for k, v in TR_LS.items()}), "BTW events resampled within site x leader; lane subset per replicate",
                            B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Step 7 — Sensitivity to the 0.5–5 s extraction window

Events were extracted only when $0.5 \le H \le 5$ s, so the window can be **narrowed** but not widened. The key results are refitted on narrower windows, each with the likelihood truncated to its own bounds. This tests whether they depend on where the extraction rule draws the lines:
- trimming the shortest headways (the end most exposed to coding error);
- trimming the longest headways (the end closest to free flow).

| Window (s) | Removes |
|---|---|
| 0.5–5.0 | nothing (main analysis) |
| 0.75–5.0, 1.0–5.0 | the shortest headways |
| 0.5–4.5, 0.5–4.0 | the longest headways |
| 1.0–4.0 | both ends |

**Re-estimated for each window:**
1. Best distribution by AIC among the five candidates (step 1), and whether Weibull stays within 2 AIC.
2. Leader time ratios of the truncated Weibull AFT (step 2, reference 4W), 95% Wald intervals.
3. Share of the PR–BTW log-headway contrast due to speed (step 4), with a stratified bootstrap interval (B = 1,000).

Where a narrow window puts its upper bound close to the PR mode (about 3 s) and a leader cell has only about ten events, the leader-conditioned Weibull fit for PR followers is not identified. Such time ratios are marked as missing, not reported.

**Outputs:** `outputs/tables/07_window_sensitivity.xlsx`, and figure `window_sensitivity.png`.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import hashlib, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from openpyxl import load_workbook
from openpyxl.styles import Font
import hwdist as hd

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"; TAB.mkdir(parents=True, exist_ok=True)
FIG = HERE.parent / "outputs_TRA" / "figures"; FIG.mkdir(parents=True, exist_ok=True)
FONT = "Cambria"
plt.rcParams.update({"font.family": FONT, "mathtext.fontset": "custom", "mathtext.rm": FONT,
                     "mathtext.it": FONT + ":italic", "mathtext.bf": FONT + ":bold", "font.size": 9})
def save_fig(fig, name):
    p = FIG / f"{name}.png"; p.unlink(missing_ok=True); fig.savefig(p, dpi=1000, bbox_inches="tight"); print("saved", p.name)
def cambria_xlsx(path):
    wb = load_workbook(path)
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for c in row:
                c.font = Font(name=FONT, bold=(c.row == 1), size=11)
    wb.save(path)

### Settings

In [ ]:
WINDOWS = [(0.5, 5.0), (0.75, 5.0), (1.0, 5.0), (0.5, 4.5), (0.5, 4.0), (1.0, 4.0)]
WLAB = {w: f"{w[0]:g}–{w[1]:g} s" for w in WINDOWS}
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
SITES = ["Shahjahanpur", "Tikatuli"]
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]; NONREF = LEADERS[1:]
CANDIDATES = ["lognormal", "loglogistic", "weibull", "gamma", "gengamma"]
B = 1000; SEED = 20261107

### Data

In [ ]:
d = pd.read_excel(DATA, sheet_name="data04")
print("data04:", d.shape, " sha256:", hashlib.sha256(DATA.read_bytes()).hexdigest()[:16])
d["logH"] = np.log(d.Time_Headway_s); d["logv"] = np.log(d.Target_Speed_kmh / 3.6); d["logS"] = d.logH + d.logv
cnt = pd.DataFrame([{"Window": WLAB[w], **{f"{s} {t}": int(((d.Site == s) & (d.Target_Class == t) & d.Time_Headway_s.between(*w)).sum())
                                         for s, t in CASES}} for w in WINDOWS])
print(cnt.to_string(index=False))

### 1–2. Best distribution and Weibull leader time ratios in each window

In [ ]:
fam_rows, tr_rows = [], []
for w in WINDOWS:
    lo, hi = w
    for c in CASES:
        g = d[(d.Site == c[0]) & (d.Target_Class == c[1]) & d.Time_Headway_s.between(lo, hi)]
        x = g.Time_Headway_s.to_numpy(float)
        aic = {f: hd.fit(x, f, lower=lo, upper=hi)["AIC"] for f in CANDIDATES}
        best = min(aic, key=aic.get)
        fam_rows.append({"Window": WLAB[w], "Site": c[0], "Class": c[1], "n": len(x), "best_family": hd.LABELS[best],
                         "dAIC_weibull": aic["weibull"] - aic[best], "weibull_within_2": aic["weibull"] - aic[best] < 2})
        X = np.column_stack([np.ones(len(g))] + [(g.Leader_Class == L).to_numpy(float) for L in NONREF])
        r = hd.fit(x, "weibull", X=X, lower=lo, upper=hi); V = hd.covariance(r, x)
        for j, L in enumerate(NONREF, start=1):
            se = float(np.sqrt(max(V[j, j], 0))); b = float(r["beta"][j])
            tr_rows.append({"Window": WLAB[w], "Site": c[0], "Class": c[1], "Leader": L, "n_leader": int((g.Leader_Class == L).sum()),
                            "TR": np.exp(b), "TR_lo95": np.exp(b - 1.96 * se), "TR_hi95": np.exp(b + 1.96 * se)})
fam = pd.DataFrame(fam_rows); trw = pd.DataFrame(tr_rows)
# a fit is "not identified" when its interval is not finite or the ratio is implausible (narrow window near the PR mode, small leader cells)
trw["identified"] = np.isfinite(trw.TR_lo95) & np.isfinite(trw.TR_hi95) & trw.TR.between(0.1, 10) & (trw.TR_hi95 / trw.TR_lo95 < 100)
for col in ["TR", "TR_lo95", "TR_hi95"]:
    trw.loc[~trw.identified, col] = np.nan
print("leader time ratios not identified:")
print(trw[~trw.identified][["Window", "Site", "Class", "Leader", "n_leader"]].to_string(index=False))
print(fam.pivot_table(index="Window", columns=["Site", "Class"], values="best_family", aggfunc="first", sort=False).to_string())
print(fam.pivot_table(index="Window", columns=["Site", "Class"], values="dAIC_weibull", sort=False).round(2).to_string())

### 3. Speed share of the PR–BTW contrast in each window

In [ ]:
def speed_share(g, w_shares):
    m = g.groupby(["Target_Class", "Leader_Class"])[["logH", "logv"]].mean()
    dH = sum(w_shares[L] * (m.loc[("PR", L), "logH"] - m.loc[("BTW", L), "logH"]) for L in LEADERS)
    dv = sum(w_shares[L] * (m.loc[("PR", L), "logv"] - m.loc[("BTW", L), "logv"]) for L in LEADERS)
    return dH, -dv / dH

rng = np.random.default_rng(SEED); dec_rows = []
for w in WINDOWS:
    for s in SITES:
        g = d[(d.Site == s) & d.Time_Headway_s.between(*w)].reset_index(drop=True)
        ws = g.Leader_Class.value_counts(normalize=True).reindex(LEADERS)
        dH, sh = speed_share(g, ws)
        cells = [np.flatnonzero(((g.Target_Class == t) & (g.Leader_Class == L)).to_numpy()) for t in ["PR", "BTW"] for L in LEADERS]
        bs = []
        for _ in range(B):
            idx = np.concatenate([rng.choice(ix, size=len(ix), replace=True) for ix in cells])
            bs.append(speed_share(g.iloc[idx], ws)[1])
        dec_rows.append({"Window": WLAB[w], "Site": s, "n": len(g), "headway_ratio_PR_BTW": np.exp(dH), "speed_share": sh,
                         "speed_share_lo95": np.percentile(bs, 2.5), "speed_share_hi95": np.percentile(bs, 97.5)})
dec = pd.DataFrame(dec_rows); print(dec.round(3).to_string(index=False))

### Figure: leader time ratios and speed share across windows

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.4, 3.0), gridspec_kw={"width_ratios": [1, 1, 0.9]})
yl = np.arange(len(WINDOWS))[::-1]
for ax, c, tag in zip(axes[:2], [("Shahjahanpur", "BTW"), ("Tikatuli", "BTW")], "ab"):
    off = dict(zip(NONREF, np.linspace(0.27, -0.27, len(NONREF))))
    for L in NONREF:
        t = trw[(trw.Site == c[0]) & (trw.Class == c[1]) & (trw.Leader == L)].set_index("Window").loc[[WLAB[w] for w in WINDOWS]]
        y = yl + off[L]
        ax.errorbar(t.TR, y, xerr=[t.TR - t.TR_lo95, t.TR_hi95 - t.TR], fmt="o", ms=3, color=PAL["leader"][L], elinewidth=1, capsize=0,
                    label=L)
    ax.axvline(1, color="0.4", lw=.7); ax.set_xscale("log"); ax.set_xlim(0.4, 1.6); ax.set_xticks([0.5, 0.7, 1, 1.4]); ax.set_xticklabels(["0.5", "0.7", "1", "1.4"])
    ax.minorticks_off(); ax.grid(axis="x", alpha=.25, lw=.5); ax.set_xlabel("Time ratio vs 4W leader")
    ax.set_title(f"({tag}) {c[0]}, BTW followers", loc="left", fontsize=8.5)
    for k in range(len(WINDOWS) - 1): ax.axhline(k + 0.5, color="0.9", lw=.6)
axes[0].set_yticks(yl); axes[0].set_yticklabels([WLAB[w] for w in WINDOWS]); axes[1].set_yticks(yl); axes[1].set_yticklabels([])
axes[0].set_ylabel("Extraction window")
ax = axes[2]
for s, dy, mk in [("Shahjahanpur", 0.12, "o"), ("Tikatuli", -0.12, "s")]:
    t = dec[dec.Site == s].set_index("Window").loc[[WLAB[w] for w in WINDOWS]]
    ax.errorbar(100 * t.speed_share, yl + dy, xerr=[100 * (t.speed_share - t.speed_share_lo95), 100 * (t.speed_share_hi95 - t.speed_share)],
                fmt=mk, ms=3.5, color=PAL["ink"] if s == "Shahjahanpur" else PAL["muted"], mfc=PAL["ink"] if s == "Shahjahanpur" else "white", elinewidth=1, capsize=0, label=s)
ax.set_yticks(yl); ax.set_yticklabels([]); ax.axvline(100, color="0.6", lw=.6, ls=":"); ax.set_xlim(0, 220); ax.set_xticks([0, 50, 100, 150, 200]); ax.grid(axis="x", alpha=.25, lw=.5)
ax.set_xlabel("Speed share of PR–BTW contrast (%)"); ax.set_title("(c) Spacing–speed split", loc="left", fontsize=8.5)
for k in range(len(WINDOWS) - 1): ax.axhline(k + 0.5, color="0.9", lw=.6)
h1, l1 = axes[0].get_legend_handles_labels(); h3, l3 = ax.get_legend_handles_labels()
fig.tight_layout(rect=(0, 0.1, 1, 1))
fig.legend(h1 + h3, l1 + l3, frameon=False, fontsize=7.2, ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.01))
save_fig(fig, "window_sensitivity"); plt.show()

### Export

In [ ]:
out = TAB / "07_window_sensitivity.xlsx"
with pd.ExcelWriter(out) as xw:
    cnt.to_excel(xw, sheet_name="n_by_window", index=False)
    fam.to_excel(xw, sheet_name="best_family", index=False)
    trw.to_excel(xw, sheet_name="weibull_leader_TR", index=False)
    dec.to_excel(xw, sheet_name="speed_share", index=False)
    pd.DataFrame({"Parameter": ["windows_s", "B", "SEED", "data04_sha256"],
                  "Value": [json.dumps(WINDOWS), B, SEED, hashlib.sha256(DATA.read_bytes()).hexdigest()]}).to_excel(xw, sheet_name="settings", index=False)
cambria_xlsx(out); print("wrote", out.name)

## Manuscript tables — Manuscript and appendix tables

Collects the results of steps 1–7 into one workbook, `outputs/Manuscript_Tables.xlsx`, with one table per sheet. Each table keeps only the columns needed in the manuscript or appendix. The full result tables stay in `outputs/tables/`.

**Style**, following the TRB manuscript:
- estimates with 95% intervals in brackets;
- p-values to three decimals ("<0.001" below that);
- goodness-of-fit results marked (A) accepted or (R) rejected at 5%;
- percentages to one decimal;
- negative numbers with a true minus sign.

Each sheet has a title row, a header row, the table, and a notes row.

**Run order:** notebook 1, then 2, then 3.

In [ ]:
import sys; sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
from style import PAL, use_style
from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HERE = Path.cwd()
DATA = HERE.parent / "outputs_TRA" / "data" / "data04.xlsx"
TAB = HERE.parent / "outputs_TRA" / "tables"
FIG = HERE.parent / "outputs_TRA" / "figures"
OUT = HERE.parent / "outputs_TRA" / "Manuscript_Tables.xlsx"
need = ["01_truncated_distribution_fit", "02_leader_conditioned_aft", "02b_leader_distribution_shifts", "03_covariate_time_ratios",
        "04_spacing_speed_decomposition", "05_speed_cap_scenarios", "06_dedicated_btw_lane", "07_window_sensitivity"]
missing = [n for n in need if not (TAB / f"{n}.xlsx").exists()]
assert not missing, f"run these notebooks first: {missing}"
R = {n: pd.read_excel(TAB / f"{n}.xlsx", sheet_name=None) for n in need}
d = pd.read_excel(DATA, sheet_name="data04")
CASES = [("Shahjahanpur", "PR"), ("Shahjahanpur", "BTW"), ("Tikatuli", "PR"), ("Tikatuli", "BTW")]
CASE_LAB = {c: f"{c[0]}, {c[1]}" for c in CASES}
LEADERS = ["4W", "BTW", "CNG", "MT_2W", "NMT"]
FAM_ORDER = ["Lognormal", "Log-logistic", "Weibull", "Gamma", "Generalized gamma"]
print("inputs found:", len(R), "result workbooks")

### Formatting helpers

In [ ]:
MINUS = "−"
def num(x, d=3):
    if x is None or (isinstance(x, float) and np.isnan(x)): return "—"
    s = f"{x:.{d}f}"
    return s.replace("-", MINUS) if s.startswith("-") and float(s) != 0 else s.lstrip("-")
def ci(e, lo, hi, d=3):
    return f"{num(e, d)} ({num(lo, d)} to {num(hi, d)})"
def pct(x, d=1): return num(100 * x, d)
def pct_ci(e, lo, hi, d=1): return ci(100 * e, 100 * lo, 100 * hi, d)
def signed(x, d=1):
    s = num(x, d); return s if s.startswith(MINUS) or s in ("0.0", "0", "—") else "+" + s
def signed_ci(e, lo, hi, d=1): return f"{signed(e, d)} ({signed(lo, d)} to {signed(hi, d)})"
def pval(p):
    if p is None or (isinstance(p, float) and np.isnan(p)): return "—"
    return "<0.001" if p < 0.001 else f"{p:.3f}"
def gof(p): return f"{pval(p)} ({'A' if p >= 0.05 else 'R'})"
def n_pct(k, n): return f"{k} ({100 * k / n:.1f}%)"

TABLES = []        # (sheet, title, DataFrame, notes)
def add(sheet, title, df, notes):
    TABLES.append((sheet, title, df.reset_index(drop=True), notes))

### Table 1 — Sample description

In [ ]:
rows = []
def block(label, fn):
    for cat, f in fn:
        r = {"Variable": label, "Category": cat}
        for c in CASES:
            g = d[(d.Site == c[0]) & (d.Target_Class == c[1])]; r[CASE_LAB[c]] = f(g)
        rows.append(r); label = ""
block("Following events", [("n", lambda g: str(len(g)))])
block("Leader class", [(L, (lambda L: lambda g: n_pct(int((g.Leader_Class == L).sum()), len(g)))(L)) for L in LEADERS])
block("Lateral position", [("Off-centred", lambda g: n_pct(int(g.Off_Centered.sum()), len(g)))])
block("Occupancy", [("Occupied", lambda g: n_pct(int(g.Occupied.sum()), len(g)))])
ms = lambda col: (lambda g: f"{num(g[col].mean(), 2)} ± {num(g[col].std(), 2)}")
rg = lambda col: (lambda g: f"{num(g[col].min(), 2)} to {num(g[col].max(), 2)}")
block("Time headway (s)", [("Mean ± SD", ms("Time_Headway_s")), ("Median", lambda g: num(g.Time_Headway_s.median(), 2)),
                           ("Range", rg("Time_Headway_s"))])
block("Follower speed (km/h)", [("Mean ± SD", ms("Target_Speed_kmh"))])
block("Leader speed (km/h)", [("Mean ± SD", ms("Leading_Speed_kmh"))])
t1 = pd.DataFrame(rows)
add("T1_Sample", "Table 1. Following events by site and target class", t1,
    "Events extracted only when 0.5 ≤ H ≤ 5 s. Leader classes after pooling: 4W (car, SUV, utility, human hauler), BTW (BTW, modified BTW), "
    "CNG auto-rickshaw, MT_2W (motorcycle), NMT (PR, van, bicycle). Speeds are space-mean speeds over the segment.")
t1

### Table 2 — Truncated distribution fits (step 1)

In [ ]:
f = R["01_truncated_distribution_fit"]["fits_gof"]
f = f[f.Family.isin(FAM_ORDER)].copy()
rows = []
for c in CASES:
    g = f[(f.Site == c[0]) & (f.Class == c[1])].sort_values("AIC")
    for i, (_, r) in enumerate(g.iterrows()):
        rows.append({"Case": f"{CASE_LAB[c]} (n = {int(r.n)})" if i == 0 else "", "Distribution": r.Family, "AIC": num(r.AIC, 2),
                     "ΔAIC": num(r.dAIC, 2), "Akaike weight": num(r.Akaike_weight, 3),
                     "KS p": gof(r.KS_p), "AD p": gof(r.AD_p), "CvM p": gof(r.CvM_p)})
add("T2_Distribution_fit", "Table 2. Truncated headway distributions by site and target class", pd.DataFrame(rows),
    "Maximum likelihood truncated to the 0.5–5 s extraction window. p-values from a refitted parametric bootstrap (999 replicates) of the "
    "KS, AD and CvM statistics on the probability-integral transform; (A) not rejected, (R) rejected at 5%. Families ordered by AIC within each case.")
TABLES[-1][2].head(6)

### Table 3 — Leader-conditioned AFT: choice of family and fit (step 2)

In [ ]:
fc = R["02_leader_conditioned_aft"]["family_choice"].copy()
fl = R["02_leader_conditioned_aft"]["fits_LR_gof"]
cand = fl[fl.Family.isin(FAM_ORDER)]
adq = cand.groupby("Family").adequate_KS_AD_CvM.sum()
t3a = pd.DataFrame({"Distribution": fc.Family, "Σ AIC (4 cases)": fc.sum_AIC_leader.map(lambda x: num(x, 2)),
                    "ΔΣ AIC": fc.d_sum_AIC.map(lambda x: num(x, 2)),
                    "Joint LR vs generalized gamma (4 df), p": [("—" if np.isnan(a) else f"{num(a, 2)}, {pval(p)}") for a, p in zip(fc.joint_LR_vs_GG, fc.joint_LR_p)],
                    "Cases adequate (of 4)": [int(adq.get(fm, 0)) for fm in fc.Family]})
add("T3a_Family_choice", "Table 3a. Choice of one distribution for the leader-conditioned AFT model", t3a,
    "Leader-conditioned truncated AFT (location = intercept + four leader-class indicators, reference 4W). Chosen family: the best-ranked "
    "two-parameter family not rejected by the joint likelihood-ratio test against the generalized gamma (Weibull). Adequate = KS, AD and CvM all p ≥ 0.05.")
s = R["02_leader_conditioned_aft"]["summary"]
rows = []
for c in CASES:
    r = s[(s.Site == c[0]) & (s.Class == c[1])].iloc[0]
    rows.append({"Case": f"{CASE_LAB[c]} (n = {int(r.n)})", "AIC, no leader": num(r.AIC_uncond, 2), "AIC, leader class": num(r.AIC_leader, 2),
                 "ΔAIC": num(r.AIC_change, 2), "LR leader class (4 df), p": f"{num(r.LR_stat, 2)}, {pval(r.LR_p)}",
                 "KS p": gof(r.KS_p), "AD p": gof(r.AD_p), "CvM p": gof(r.CvM_p)})
add("T3b_Weibull_leader_model", "Table 3b. Leader-conditioned Weibull AFT by site and target class", pd.DataFrame(rows),
    "ΔAIC = AIC with leader class − AIC without. Goodness of fit from a refitted parametric bootstrap (999 replicates).")
TABLES[-1][2]

### Table 4 — Time ratios by leader class (step 2)

In [ ]:
tr = R["02_leader_conditioned_aft"]["time_ratios"]
rows = []
for L in ["BTW", "CNG", "MT_2W", "NMT"]:
    r = {"Leader class": L}
    for c in CASES:
        x = tr[(tr.Site == c[0]) & (tr.Class == c[1]) & (tr.Leader == L)].iloc[0]
        r[CASE_LAB[c]] = ci(x.TR, x.TR_lo95, x.TR_hi95) + (" *" if x.TR_hi95 < 1 or x.TR_lo95 > 1 else "")
    rows.append(r)
nref = {CASE_LAB[c]: str(int(tr[(tr.Site == c[0]) & (tr.Class == c[1])].n_ref.iloc[0])) for c in CASES}
rows.insert(0, {"Leader class": "4W (reference, n)", **nref})
add("T4_Leader_time_ratios", "Table 4. Time ratios of the headway distribution by leader class (Weibull AFT, reference 4W)", pd.DataFrame(rows),
    "Time ratio exp(β) with 95% Wald interval. A ratio below 1 compresses the whole headway distribution relative to following a four-wheeler. "
    "* interval excludes 1.")
TABLES[-1][2]

### Table 5 — Leader shifts against the main distribution (step 2b)

In [ ]:
lv = R["02b_leader_distribution_shifts"]["leader_vs_main"]
sp = R["02b_leader_distribution_shifts"]["type_shift_vs_spread"]
TYPE = {"compresses (shorter headways)": "compresses", "stretches (longer headways)": "stretches", "no clear shift": "—"}
rows = []
for L in LEADERS:
    r = {"Leader class": L}
    for c in CASES:
        x = lv[(lv.Site == c[0]) & (lv.Class == c[1]) & (lv.Leader == L)].iloc[0]
        r[CASE_LAB[c]] = f"{signed_ci(x.d_median_pct, 100 * (x.median_ratio_lo95 - 1), 100 * (x.median_ratio_hi95 - 1))} {TYPE[x.Effect_type]}".rstrip(" —")
    rows.append(r)
r = {"Leader class": "Shift-only vs shift and spread, LR (4 df) p"}
for c in CASES:
    x = sp[(sp.Site == c[0]) & (sp.Class == c[1])].iloc[0]; r[CASE_LAB[c]] = pval(x.LR_p)
rows.append(r)
add("T5_Leader_shifts", "Table 5. Change in median headway behind each leader class relative to the case's main distribution (%)", pd.DataFrame(rows),
    "Main distribution = mixture of the leader-specific Weibull fits weighted by the observed leader shares. 95% intervals from 4,000 draws of the "
    "parameter vector. 'compresses' / 'stretches' = interval wholly below / above 0. Last row: does the leader also change the spread (shape)?")
TABLES[-1][2]

### Table 6 — Other variables as time ratios (step 3)

In [ ]:
cv = R["03_covariate_time_ratios"]["crude_vs_adjusted"]
VAR = {"Off_Centered": "Off-centred following", "Occupied": "Occupied (passengers)", "Leader_speed_5kmh": "Leader speed (+5 km/h)"}
rows = []
for v, vl in VAR.items():
    for i, c in enumerate(CASES):
        x = cv[(cv.Site == c[0]) & (cv.Class == c[1]) & (cv.Variable == v)].iloc[0]
        rows.append({"Variable": vl if i == 0 else "", "Case": CASE_LAB[c],
                     "Crude TR (95% CI)": ci(x.crude_TR, x.crude_TR_lo95, x.crude_TR_hi95), "Crude Holm p": pval(x.crude_LR_p_Holm),
                     "Leader-adjusted TR (95% CI)": ci(x.leader_adjusted_TR, x.leader_adjusted_TR_lo95, x.leader_adjusted_TR_hi95),
                     "Leader-adjusted Holm p": pval(x.leader_adjusted_LR_p_Holm)})
add("T6_Covariate_time_ratios", "Table 6. Time ratios of off-centred following, occupancy and leader speed (truncated Weibull AFT)", pd.DataFrame(rows),
    "Each variable entered alone: crude = variable only; leader-adjusted = added to the leader-class model. Likelihood-ratio p-values, "
    "Holm-adjusted across the three variables within each case and model. Leader speed centred at the case mean.")
TABLES[-1][2].head(4)

### Table 7 — Spacing–speed decomposition (step 4)

In [ ]:
de = R["04_spacing_speed_decomposition"]["decomposition"]
bt = R["04_spacing_speed_decomposition"]["by_site_target"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    gp = bt[(bt.Site == site) & (bt.Target_Class == "PR")].iloc[0]; gb = bt[(bt.Site == site) & (bt.Target_Class == "BTW")].iloc[0]
    q = de[de.Site == site].set_index("Component")
    spec = [("Headway (s)", gp.gm_headway_s, gb.gm_headway_s, "Headway", 2),
            ("Spacing (m)", gp.gm_spacing_m, gb.gm_spacing_m, "Spacing part", 2),
            ("Speed (km/h)", gp.gm_speed_kmh, gb.gm_speed_kmh, "Speed part", 1)]
    for i, (lab, vp, vb, comp, dd) in enumerate(spec):
        x = q.loc[comp]
        rows.append({"Site": site if i == 0 else "", "Quantity": lab, "PR": num(vp, dd), "BTW": num(vb, dd),
                     "Contrast (%, 95% CI)": signed_ci(x.pct, x.pct_lo95, x.pct_hi95),
                     "Share of log headway contrast (%)": "100" if comp == "Headway" else pct_ci(x.share_of_headway_contrast, x.share_lo95, x.share_hi95, 0)})
add("T7_Decomposition", "Table 7. PR–BTW headway contrast split into spacing and speed parts (same leader mix)", pd.DataFrame(rows),
    "Geometric means. log H = log S − log v; contrasts PR − BTW, standardised to the site's leader shares; headway % = PR ÷ BTW − 1; "
    "spacing % = PR ÷ BTW spacing − 1; speed % = BTW ÷ PR speed − 1. The spacing and speed parts multiply to the headway contrast. "
    "Stratified bootstrap (4,000 replicates).")
TABLES[-1][2]

### Table 8 — E-rickshaw speed limits (step 5)

In [ ]:
sc = R["05_speed_cap_scenarios"]["scenarios"]; ex = R["05_speed_cap_scenarios"]["short_headway_exposure"]
sc["Cap_kmh"] = sc.Cap_kmh.astype(str); ex["Cap_kmh"] = ex.Cap_kmh.astype(str)
LIM = {"observed": "No limit (observed)", "25": "25 km/h (general)", "15": "15 km/h (zone)"}
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for i, cap in enumerate(["observed", "25", "15"]):
        x = sc[(sc.Site == site) & (sc.Cap_kmh == cap)].iloc[0]
        e2 = ex[(ex.Site == site) & (ex.Cap_kmh == cap) & (ex.Threshold_s == 2.0)].iloc[0]
        e15 = ex[(ex.Site == site) & (ex.Cap_kmh == cap) & (ex.Threshold_s == 1.5)].iloc[0]
        obs = cap == "observed"
        rows.append({"Site": site if i == 0 else "", "Speed limit": LIM[cap],
                     "Not affected (%)": pct(x.share_case1), "Follows at the limit (%)": pct(x.share_case3),
                     "Following breaks up (%)": "0.0" if obs else pct_ci(x.share_case2, x.share_case2_lo95, x.share_case2_hi95),
                     "Median, still following (s)": num(x.median_s, 2),
                     "Median change: selection / speed (%)": "—" if obs else f"{signed(x.d_median_pct_selection)} / {signed(x.d_median_pct_speed)}",
                     "H < 2 s per 100 events": num(e2.after_per100, 1) if obs else ci(e2.after_per100, e2.after_per100_lo95, e2.after_per100_hi95, 1),
                     "H < 1.5 s per 100 events": num(e15.after_per100, 1) if obs else ci(e15.after_per100, e15.after_per100_lo95, e15.after_per100_hi95, 1)})
add("T8_Speed_limits", "Table 8. BTW following under the e-rickshaw speed limits (S.R.O. No. 384-Law/2026)", pd.DataFrame(rows),
    "Limits: 25 km/h general (rule 2(gha)), 15 km/h near schools, hospitals, places of worship and crowded places (rule 8(4)); applied to BTW only. "
    "An event breaks up when the BTW is faster than the limit and follows a faster non-BTW leader; otherwise it follows at the limit with the same spacing "
    "(H' = H v / limit). Exposure per 100 observed BTW following events; events that break up are not exposed. Bootstrap (2,000 replicates).")
TABLES[-1][2]

### Table 9 — Dedicated BTW lane (step 6)

In [ ]:
dl = R["06_dedicated_btw_lane"]["scenarios"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for i, (_, x) in enumerate(dl[dl.Site == site].iterrows()):
        rows.append({"Site": site if i == 0 else "", "Scenario": x.Scenario, "n": int(x.n_following), "Median (s)": num(x.median_s, 2),
                     "H < 2 s (%, 95% CI)": pct_ci(x["share_H<2"], x["share_H<2_lo95"], x["share_H<2_hi95"]),
                     "H < 1.5 s (%, 95% CI)": pct_ci(x["share_H<1.5"], x["share_H<1.5_lo95"], x["share_H<1.5_hi95"])})
add("T9_Dedicated_lane", "Table 9. Short-headway exposure in mixed traffic and in a dedicated BTW lane at 25 km/h", pd.DataFrame(rows),
    "Dedicated lane: BTW followers behind BTW leaders, both limited to 25 km/h, so following never breaks up. The centred-only version uses the BTW-led "
    "events that were centred (small n at Shahjahanpur). Observed BTW-led following in mixed traffic stands in for in-lane behaviour.")
TABLES[-1][2]

### Appendix tables

In [ ]:
nl = R["01_truncated_distribution_fit"]["nested_LR"]
rows = []
for c in CASES:
    for i, (_, x) in enumerate(nl[(nl.Site == c[0]) & (nl.Class == c[1])].iterrows()):
        rows.append({"Case": CASE_LAB[c] if i == 0 else "", "Generalized gamma Q̂": num(x.GG_Q_hat, 3) if i == 0 else "",
                     "Restricted family": x.Restricted, "LR (1 df)": num(x.LR_stat, 2), "p": pval(x.p)})
add("A1_Nested_LR", "Table A1. Nested likelihood-ratio tests against the generalized gamma (unconditional fits)", pd.DataFrame(rows),
    "Lognormal: Q = 0; Weibull: Q = 1; gamma: Q = σ. A small p rejects the two-parameter family in favour of the generalized gamma.")

rows = []
for c in CASES:
    g = cand[(cand.Site == c[0]) & (cand.Class == c[1])].sort_values("AIC_leader")
    for i, (_, x) in enumerate(g.iterrows()):
        rows.append({"Case": CASE_LAB[c] if i == 0 else "", "Distribution": x.Family, "ΔAIC (within case)": num(x.dAIC_leader, 2),
                     "LR leader class (4 df) p": pval(x.LR_p), "KS p": gof(x.KS_p), "AD p": gof(x.AD_p), "CvM p": gof(x.CvM_p)})
add("A2_Leader_AFT_all_families", "Table A2. Leader-conditioned AFT fits for all five distributions", pd.DataFrame(rows),
    "Refitted parametric bootstrap, 999 replicates. In Shahjahanpur PR the generalized gamma sits on a weakly identified high-|Q| ridge.")

rows = []
for c in CASES:
    x = sp[(sp.Site == c[0]) & (sp.Class == c[1])].iloc[0]
    rows.append({"Case": CASE_LAB[c], "LR (4 df)": num(x.LR_stat, 2), "p": pval(x.LR_p), "ΔAIC (spread − shift only)": num(x.dAIC_spread_minus_shift, 2),
                 "σ shared": num(x.sigma_shared, 3), **{f"σ {L}": num(x[f"sigma_{L}"], 3) for L in LEADERS}})
add("A3_Shift_vs_spread", "Table A3. Does the leader change only the scale (shift) or also the spread of the headway distribution?", pd.DataFrame(rows),
    "Weibull AFT with leader-specific log σ against the shift-only model.")

pl = R["04_spacing_speed_decomposition"]["per_leader"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for i, (_, x) in enumerate(pl[pl.Site == site].iterrows()):
        rows.append({"Site": site if i == 0 else "", "Leader class": x.Leader, "n PR / BTW": f"{int(x.n_PR)} / {int(x.n_BTW)}",
                     "Headway PR ÷ BTW": num(x.headway_ratio, 2), "Spacing PR ÷ BTW": num(x.spacing_ratio, 2),
                     "Speed BTW ÷ PR": num(x.speed_ratio_BTW_over_PR, 2), "Speed share (%)": num(100 * x.speed_share, 0)})
add("A4_Decomposition_by_leader", "Table A4. Spacing–speed decomposition behind each leader class", pd.DataFrame(rows),
    "Geometric-mean ratios within each leader class; small cells, descriptive only.")

cl = R["05_speed_cap_scenarios"]["cases_by_leader"]
rows = []
for site in ["Shahjahanpur", "Tikatuli"]:
    for j, cap in enumerate([25, 15]):
        for i, (_, x) in enumerate(cl[(cl.Site == site) & (cl.Cap_kmh == cap)].iterrows()):
            rows.append({"Site": site if (i == 0 and j == 0) else "", "Speed limit": f"{cap} km/h" if i == 0 else "", "Leader class": x.Leader,
                         "n": int(x.n), "Not affected (%)": pct(x.share_case1), "Follows at the limit (%)": pct(x.share_case3),
                         "Following breaks up (%)": pct(x.share_case2)})
add("A5_Speed_limit_by_leader", "Table A5. Outcome of the speed limits for BTW followers by leader class", pd.DataFrame(rows),
    "Shares of the observed BTW following events behind each leader class.")

ss = R["05_speed_cap_scenarios"]["SP_BTW_leader_speed_sens"]; ls = R["06_dedicated_btw_lane"]["SP_leader_speed_sens"]
rows = []
for _, x in ss.iterrows():
    rows.append({"Scenario": f"Mixed traffic, {int(x.Cap_kmh)} km/h", "Without leader-speed effect": f"{num(x.median_s_no_leader_effect, 2)} s; "
                 f"{num(x.exp2_per100_no_leader_effect, 1)} / {num(x['exp1.5_per100_no_leader_effect'], 1)}",
                 "With leader-speed effect (TR 95% CI)": f"{num(x.median_s_TR_est, 2)} s ({num(x.median_s_TR_hi, 2)} to {num(x.median_s_TR_lo, 2)}); "
                 f"{num(x.exp2_per100_TR_est, 1)} / {num(x['exp1.5_per100_TR_est'], 1)}"})
for scen in ls.Scenario.unique():
    g = ls[ls.Scenario == scen].set_index("TR_leader_speed")
    rows.append({"Scenario": scen, "Without leader-speed effect": f"{num(g.loc['none', 'median_s'], 2)} s; {pct(g.loc['none', 'share_H<2'])}% / {pct(g.loc['none', 'share_H<1.5'])}%",
                 "With leader-speed effect (TR 95% CI)": f"{num(g.loc['est', 'median_s'], 2)} s ({num(g.loc['hi', 'median_s'], 2)} to {num(g.loc['lo', 'median_s'], 2)}); "
                 f"{pct(g.loc['est', 'share_H<2'])}% / {pct(g.loc['est', 'share_H<1.5'])}%"})
add("A6_SP_leader_speed_sensitivity", "Table A6. Shahjahanpur BTW: sensitivity to capped BTW leaders shortening the headway behind them", pd.DataFrame(rows),
    "Leader-adjusted time ratio 1.089 per 5 km/h of leader speed (notebook 03). Cells: median of the events still following; then short-headway "
    "exposure H < 2 s / H < 1.5 s (per 100 observed events for mixed traffic, share of lane events for the lane).")
print(len(TABLES), "tables")

### Appendix tables from the static appendix material

The pooling evidence, the traffic state and the detector metrics come from private inputs (data03, raw YOLO logs, videos). They are shipped as fixed results in `../appendix_static/` and copied into the workbook when present.

In [ ]:
pool_f, app_f = TAB / "Pooling_Tables.xlsx", TAB / "Appendix_Figure_Data.xlsx"   # written by notebook 1
VLAB = {"Time_Headway_s": "Time headway", "Target_Speed_kmh": "Follower speed", "Leading_Speed_kmh": "Leader speed"}
if pool_f.exists():
    P = pd.read_excel(pool_f, sheet_name=None)
    a = P["A_BTW_vs_ModBTW"]; rows = []
    for _, x in a.iterrows():
        rows.append({"Scope": x.scope, "Variable": VLAB[x.variable], "n BTW / modified BTW": f"{int(x.n_1)} / {int(x.n_2)}",
                     "Median BTW / modified BTW": f"{num(x.median_1, 2)} / {num(x.median_2, 2)}",
                     "Cliff's δ (95% CI)": ci(x.cliffs_delta, x.delta_CI_lo, x.delta_CI_hi), "KS p": pval(x.KS_p),
                     "TOST p (margin)": f"{pval(x.TOST_p)} (±{num(x.TOST_bound, 1)})", "BF01": num(x.BF01, 2)})
    add("A7_Target_pooling", "Table A7. Pooling of modified BTW with BTW as target vehicles", pd.DataFrame(rows),
        "Equivalence margins: ±0.3 s for headway, ±2 km/h for speed (TOST). BF01 > 3: evidence for no difference. Bootstrap CI of Cliff's δ.")
    om = P["B1_within_omnibus"]; rows = []
    for cls in om["class"].unique():
        g = om[om["class"] == cls].set_index("variable")
        rows.append({"Leader class": cls, "Pooled types": g.labels.iloc[0], "n": int(g.n.iloc[0]),
                     "Headway: KW p (ε²)": f"{pval(g.loc['Time_Headway_s', 'KW_p'])} ({num(g.loc['Time_Headway_s', 'epsilon2'], 3)})",
                     "Leader speed: KW p (ε²)": f"{pval(g.loc['Leading_Speed_kmh', 'KW_p'])} ({num(g.loc['Leading_Speed_kmh', 'epsilon2'], 3)})"})
    add("A8_Leader_pooling", "Table A8. Pooling of leader vehicle types into leader classes", pd.DataFrame(rows),
        "Kruskal–Wallis tests among the vehicle types pooled into each class. Pooling rests on the follower headway; bicycle leaders are faster "
        "than PR and van leaders, but the headway behind them does not differ. CNG and MT_2W kept as separate classes.")
if app_f.exists():
    A = pd.read_excel(app_f, sheet_name=None)
    ts = A["traffic_state_summary"]; rows = []
    for _, x in ts.iterrows():
        rows.append({"Session": x.Session, "Duration (min)": num(x.duration_min, 1), "Following events": int(x.n_events),
                     "Flow, mean ± SD (pcu/h)": f"{num(x.flow_mean_pcu_h, 0)} ± {num(x.flow_sd, 0)}",
                     "Flow, 5th–95th percentile (pcu/h)": f"{num(x.flow_p05, 0)} to {num(x.flow_p95, 0)}",
                     "3W share (%)": pct(x.mean_share_3W), "4W share (%)": pct(x.mean_share_4W), "Bus share (%)": pct(x.mean_share_Bus)})
    add("A9_Traffic_state", "Table A9. Traffic state during the recorded sessions", pd.DataFrame(rows),
        "Rolling 30-s centred flow from YOLO counts, converted with RHD (2005) PCU factors. SP and SP2 are the two Shahjahanpur sessions.")
    ym = A["yolo_metrics"]; rows = []
    for _, x in ym.iterrows():
        rows.append({"Class": x["class"], "AP50": num(x.AP50, 3), "AP50–95": num(x.AP50_95, 3),
                     "Precision": num(x.precision, 3), "Recall": num(x.recall, 3)})
    add("A10_Detector", "Table A10. Detector performance on the validation split (YOLO26n)", pd.DataFrame(rows),
        "Validation split only (the test folder duplicates it). Precision and recall reported for all classes combined.")
wsens = R["07_window_sensitivity"]
fw, tw, dw, nw = wsens["best_family"], wsens["weibull_leader_TR"], wsens["speed_share"], wsens["n_by_window"]
rows = []
for win in nw.Window:
    f_ = fw[fw.Window == win]; t_ = tw[tw.Window == win]; d_ = dw[dw.Window == win].set_index("Site")
    tr_ = lambda s, c, L: (lambda x: ci(x.TR, x.TR_lo95, x.TR_hi95, 2))(t_[(t_.Site == s) & (t_.Class == c) & (t_.Leader == L)].iloc[0])
    rows.append({"Window": win, "n (SP PR / SP BTW / TK PR / TK BTW)": " / ".join(str(int(nw[nw.Window == win][f"{s} {c}"].iloc[0])) for s, c in CASES),
                 "Weibull within 2 AIC of best (cases)": f"{int(f_.weibull_within_2.sum())} of 4",
                 "TR behind BTW, SP BTW": tr_("Shahjahanpur", "BTW", "BTW"), "TR behind BTW, TK BTW": tr_("Tikatuli", "BTW", "BTW"),
                 "TR behind NMT, TK BTW": tr_("Tikatuli", "BTW", "NMT"),
                 "Speed share, SP (%)": pct_ci(d_.loc["Shahjahanpur", "speed_share"], d_.loc["Shahjahanpur", "speed_share_lo95"], d_.loc["Shahjahanpur", "speed_share_hi95"], 0),
                 "Speed share, TK (%)": pct_ci(d_.loc["Tikatuli", "speed_share"], d_.loc["Tikatuli", "speed_share_lo95"], d_.loc["Tikatuli", "speed_share_hi95"], 0)})
add("A11_Window_sensitivity", "Table A11. Sensitivity of the main results to the 0.5–5 s extraction window", pd.DataFrame(rows),
    "Each narrower window is refitted with the likelihood truncated to its own bounds. TR = Weibull AFT time ratio vs a 4W leader (95% Wald CI). "
    "Speed share = share of the PR–BTW log-headway contrast due to speed (stratified bootstrap, 1,000 replicates).")
print(len(TABLES), "tables in total")

### Figure list

In [ ]:
FIGS = [
    ("truncated_fit_density", "Truncated distribution fits by site and target class", "01"),
    ("truncated_fit_cdf_difference", "Empirical minus fitted CDF for each distribution", "01"),
    ("leader_aft_time_ratios", "Time ratios by leader class, Weibull AFT", "02"),
    ("leader_aft_cox_snell", "Cox–Snell residuals of the leader-conditioned Weibull AFT", "02"),
    ("leader_shifted_densities", "Leader-specific headway distributions against the main distribution", "02b"),
    ("leader_shift_vs_main", "Shift of the median and of short headways by leader class", "02b"),
    ("covariate_time_ratios", "Time ratios of off-centred following, occupancy and leader speed", "03"),
    ("decomposition_contrast", "PR–BTW headway contrast split into spacing and speed", "04"),
    ("spacing_speed_scatter", "Spacing against follower speed with constant-headway lines", "04"),
    ("speed_cap_outcome_shares", "Outcome of the speed limits for BTW following events", "05"),
    ("speed_cap_headway_ecdf", "Headways of BTWs still following under the speed limits", "05"),
    ("speed_cap_short_headway_exposure", "Exposure to short headways under the speed limits", "05"),
    ("dedicated_lane_exposure", "Short-headway exposure in mixed traffic and in a dedicated BTW lane", "06"),
    ("dedicated_lane_headway_ecdf", "Headway distribution in mixed traffic and in a dedicated BTW lane", "06"),
    ("window_sensitivity", "Leader time ratios and spacing–speed split across extraction windows", "07"),
]
fig_tab = pd.DataFrame([{"File": f"{f}.png", "Content": c, "Notebook": nb, "Present": "yes" if (FIG / f"{f}.png").exists() else "MISSING"}
                        for f, c, nb in FIGS])
APPX = [("traffic_state_rolling_flow.png", "Rolling 30-s flow during each session"),
        ("event_timing.png", "Timing of the following events within each session"), ("yolo_detector_performance.png", "Detector performance (validation)"),
        ("target_subtypes_ECDF.png", "Headway of BTW and modified BTW targets"), ("leader_speed_by_label.png", "Leader speed by vehicle type"),
        ("headway_by_leader_label.png", "Follower headway by leader vehicle type")]
appx_tab = pd.DataFrame([{"File": f, "Content": c, "Notebook": "1 (appendix)",
                          "Present": "yes" if (FIG / f).exists() else "MISSING"} for f, c in APPX])
fig_tab = pd.concat([fig_tab, appx_tab], ignore_index=True)
print(fig_tab.to_string(index=False))

### Write the workbook

In [ ]:
FONT = "Cambria"; thin = Side(style="thin", color="000000")
wb = Workbook(); ix = wb.active; ix.title = "Index"
ix.append(["Sheet", "Title"])
for sheet, title, df, notes in TABLES:
    ix.append([sheet, title])
ix.append([]); ix.append(["Figures (PNG, 600 dpi)", ""])
ix.append(["File", "Content", "Notebook"])
for _, r in fig_tab.iterrows():
    ix.append([r.File, r.Content, r.Notebook])
for sheet, title, df, notes in TABLES:
    ws = wb.create_sheet(sheet)
    ws.append([title]); ws.append(list(df.columns))
    for row in df.itertuples(index=False):
        ws.append(list(row))
    ws.append([]); ws.append(["Note: " + notes])
    ncol = len(df.columns); last = 2 + len(df)
    for row in ws.iter_rows():
        for c in row:
            c.font = Font(name=FONT, size=11, bold=(c.row in (1, 2)))
            c.alignment = Alignment(vertical="top", wrap_text=(c.row == last + 2))
    for j in range(1, ncol + 1):
        ws.cell(2, j).border = Border(top=thin, bottom=thin); ws.cell(last, j).border = Border(bottom=thin)
        width = max(len(str(ws.cell(i, j).value or "")) for i in range(2, last + 1))
        ws.column_dimensions[get_column_letter(j)].width = min(max(10, width + 2), 48)
    ws.merge_cells(start_row=last + 2, start_column=1, end_row=last + 2, end_column=ncol)
    ws.row_dimensions[last + 2].height = 15 * (1 + len(notes) // 120)
    ws.freeze_panes = "A3"
for row in ix.iter_rows():
    for c in row:
        c.font = Font(name=FONT, size=11, bold=(c.row == 1 or c.value in ("File", "Figures (PNG, 600 dpi)")))
ix.column_dimensions["A"].width = 38; ix.column_dimensions["B"].width = 100; ix.column_dimensions["C"].width = 10
OUT.unlink(missing_ok=True); wb.save(OUT); print("wrote", OUT.name, "with", len(TABLES), "tables")

# Part 4: Figures 1 and 2, and the submission set

Figure 1 joins the study-area map (made in GIS from the DAP database, `data/raw/figure_inputs/study_area_map.png`) with one video frame per recording.
Figure 2 joins the authors' photographs; faces are covered with gray boxes.

In [ ]:
import cv2, shutil
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from pathlib import Path
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INP = ROOT / "data" / "raw" / "figure_inputs"
VID = ROOT.parent / "Traffic Detection (YOLO)" / "Raw Videos"
FIGT = ROOT / "outputs_TRA" / "figures"; FIGT.mkdir(parents=True, exist_ok=True)
SUB = ROOT / "outputs_TRA" / "submission_figures"; SUB.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"font.family": "Cambria", "font.size": 9})
DPI_PHOTO = 600

# one still per recording (times chosen by Tahsin; see LOG, Session 10)
VIEWS = [("SP.mp4", "12:45", "(b) Shahjahanpur, 08:00-09:00"),
         ("SP2 (6-7-26).mp4", "13:35", "(c) Shahjahanpur, 11:00-12:00"),
         ("Tikatuli 1.mp4", "15:57", "(d) Tikatuli, 11:00-13:00")]
def frame(video, t):
    mm, ss = map(int, t.split(":"))
    cap = cv2.VideoCapture(str(VID / video)); cap.set(cv2.CAP_PROP_POS_MSEC, (mm * 60 + ss) * 1000)
    ok, f = cap.read(); cap.release()
    assert ok, f"could not read {video} at {t}"
    return cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
FRAME_FILES = ["site_view_SP.png", "site_view_SP2.png", "site_view_Tikatuli.png"]
if all((VID / v).exists() for v, _, _ in VIEWS):
    frames = [frame(v, t) for v, t, _ in VIEWS]
    for f, name in zip(frames, FRAME_FILES):          # keep the frames so the figure can be rebuilt without the videos
        Image.fromarray(f).save(INP / name)
else:
    print("videos not found; using the saved frames in data/raw/figure_inputs")
    frames = [np.asarray(Image.open(INP / name).convert("RGB")) for name in FRAME_FILES]
print([f.shape for f in frames])

In [ ]:
# Figure 1: map on the left, the three recordings on the right
mp = np.asarray(Image.open(INP / "study_area_map.png").convert("RGB"))
W_IN, GAP, TOP = 7.2, 0.12, 0.28
fh, fw = frames[0].shape[:2]
right_w = 2.75
view_h = right_w * fh / fw
H_IN = 3 * view_h + 3 * TOP + 2 * GAP
map_h = H_IN - TOP; map_w = map_h * mp.shape[1] / mp.shape[0]
fig = plt.figure(figsize=(map_w + GAP + right_w, H_IN))
FW, FH = fig.get_size_inches()
ax = fig.add_axes([0, 0, map_w / FW, map_h / FH]); ax.imshow(mp); ax.axis("off")
fig.text(map_w / 2 / FW, (map_h + TOP / 2) / FH, "(a) Study area and survey segments", ha="center", va="center", fontsize=11)
x0 = (map_w + GAP) / FW
for i, (img, (_, _, title)) in enumerate(zip(frames, VIEWS)):
    y_top = H_IN - i * (view_h + TOP + GAP)
    a = fig.add_axes([x0, (y_top - TOP - view_h) / FH, right_w / FW, view_h / FH]); a.imshow(img); a.axis("off")
    fig.text(x0 + right_w / 2 / FW, (y_top - TOP / 2) / FH, title, ha="center", va="center", fontsize=11)
p = FIGT / "study_area_site_views.png"; fig.savefig(p, dpi=DPI_PHOTO, facecolor="white"); plt.show()
print("saved", p.name, Image.open(p).size)

In [ ]:
# Figure 2: the three follower types; faces covered (box positions in pixels of each photograph)
PHOTOS = [("vehicle_1_PR.jpg", "(a) Pedal rickshaw (PR)", [(245, 53, 112, 112), (726, 78, 112, 114)]),
          ("vehicle_2_BTW.jpg", "(b) Battery-operated\nthree-wheeler (BTW)", [(347, 37, 136, 153), (809, 78, 137, 125)]),
          ("vehicle_3_Mod_BTW_1.jpg", "(c) Modified battery-operated\nthree-wheeler (Mod_BTW)", [(373, 171, 114, 124)])]
if not all((INP / f).exists() for f, _, _ in PHOTOS):
    print("photographs not included; using the finished figure data/raw/figure_inputs/vehicle_types_finished.png")
    shutil.copyfile(INP / "vehicle_types_finished.png", FIGT / "vehicle_types.png")
else:
    imgs = [np.asarray(Image.open(INP / f).convert("RGB")) for f, _, _ in PHOTOS]
    GAP, TOP, W_IN = 0.08, 0.42, 7.2
    asp = [im.shape[1] / im.shape[0] for im in imgs]
    h = (W_IN - GAP * (len(imgs) - 1)) / sum(asp)
    fig = plt.figure(figsize=(W_IN, h + TOP))
    x = 0
    for im, a_, (_, title, boxes) in zip(imgs, asp, PHOTOS):
        w = h * a_
        ax = fig.add_axes([x / W_IN, 0, w / W_IN, h / (h + TOP)]); ax.imshow(im); ax.axis("off")
        for bx, by, bw, bh in boxes:
            ax.add_patch(Rectangle((bx, by), bw, bh, facecolor=(0.5, 0.5, 0.5), edgecolor="none"))
        fig.text((x + w / 2) / W_IN, (h + TOP / 2) / (h + TOP), title, ha="center", va="center", fontsize=9, linespacing=1.1)
        x += w + GAP
    p = FIGT / "vehicle_types.png"; fig.savefig(p, dpi=DPI_PHOTO, facecolor="white"); plt.show()
    print("saved", p.name, Image.open(p).size)

In [ ]:
# Submission set: one file per figure, numbered as in the manuscript and appendix
MAIN = ["study_area_site_views", "vehicle_types", None, "leader_shifted_densities", "leader_shift_vs_main",
        "decomposition_contrast", "spacing_speed_scatter", "speed_cap_outcome_shares",
        "speed_cap_short_headway_exposure", "dedicated_lane_exposure"]
APPX = ["traffic_state_rolling_flow", "event_timing", "target_subtypes_ECDF", "headway_by_leader_label",
        "leader_speed_by_label", "truncated_fit_density", "truncated_fit_cdf_difference", "leader_aft_cox_snell",
        "leader_aft_time_ratios", "covariate_time_ratios", "speed_cap_headway_ecdf", "dedicated_lane_headway_ecdf",
        "window_sensitivity"]
for f in SUB.glob("Figure_*"): f.unlink()
rows = []
for i, n in enumerate(MAIN, 1):
    src = INP / "research_flow.png" if n is None else FIGT / f"{n}.png"
    dst = SUB / f"Figure_{i}.png"; shutil.copyfile(src, dst); rows.append((dst.name, src.name))
for i, n in enumerate(APPX, 1):
    src = FIGT / f"{n}.png"; dst = SUB / f"Figure_A{i}.png"; shutil.copyfile(src, dst); rows.append((dst.name, src.name))
for dst, src in rows:
    im = Image.open(SUB / dst)
    print(f"{dst:16s} {src:40s} {im.size[0]:>6} x {im.size[1]:<6} {round(im.info.get('dpi', (0,))[0])} dpi  {(SUB / dst).stat().st_size / 1e6:5.1f} MB")